# Phase 1 — Python Language Fundamentals

**Outcome:** predict the output of any core Python snippet and explain it in terms of
objects, names, types and frames; use every built-in data type and its API fluently.

| Unit | Topic |
| --- | --- |
| 1.1 | How Python executes code |
| 1.2 | Objects, names and mutability |
| 1.3 | Numeric and boolean data types |
| 1.4 | Text and binary data types |
| 1.5 | Collection data types I: lists, tuples and ranges |
| 1.6 | Collection data types II: dictionaries, sets and the `collections` module |
| 1.7 | Copying: shallow copies, deep copies and structured clones |
| 1.8 | Operators, expressions and control flow |
| 1.9 | Iteration, comprehensions and built-in functions |
| 1.10 | Functions: definition and arguments |
| 1.11 | Scope, namespaces and closures |
| 1.12 | Decorators, `functools` and the call stack |
| 1.13 | Iterators, generators and `itertools` |
| 1.14 | Exceptions and context managers |
| Project | Log analyzer |

**How to use this notebook**

- Select the **Python 3.15 (course)** kernel, then run the *Shared setup* cell first.
- Each unit starts with its own setup cell, so you can jump straight to any unit.
- *Predict the output* questions hide the answer in a collapsible block: decide first,
  then run the proof cell.
- Labs: replace the `raise NotImplementedError` in each stub, then run the check cell.
  Until then the checks print FAIL; that is expected.
- Features newer than Python 3.12 are labelled (*New in 3.13*, *Changed in 3.15*, …).

## Shared setup

This cell runs first in every notebook of the course. It imports common modules and
defines three helpers used everywhere:

| Helper | What it does |
| --- | --- |
| `run_checks(test_cases)` | runs `(label, thunk, expected)` cases, prints PASS/FAIL |
| `show_size(label, obj)` | prints `sys.getsizeof(obj)`, type name and `id()` |
| `find_repo_root()` | finds the folder holding `pyproject.toml` |

`run_checks` catches every exception, so a failing check never stops Run All.

> **Note:** each unit re-imports what it needs, so you can run any unit on its own
> after this cell.

In [1]:
import inspect
import os
import platform
import sys
from collections.abc import Callable
from pathlib import Path

type TestCase = tuple[str, Callable[[], object], object]


def run_checks(test_cases: list[TestCase]) -> None:
    """Run (label, thunk, expected) cases and print PASS/FAIL for each. Never raises."""
    passed_count = 0
    for label, thunk, expected in test_cases:
        try:
            actual = thunk()
        except Exception as error:  # a check cell must never stop Run All
            print(f"FAIL  {label}: raised {type(error).__name__}: {error}")
            continue
        if actual == expected:
            passed_count += 1
            print(f"PASS  {label}")
        else:
            print(f"FAIL  {label}: expected {expected!r}, got {actual!r}")
    print(f"{passed_count}/{len(test_cases)} checks passed")


def show_size(label: str, obj: object) -> None:
    """Print the shallow size in bytes, the type and the id of an object."""
    print(f"{label}: {sys.getsizeof(obj)} bytes, type={type(obj).__name__}, id={id(obj)}")


def find_repo_root(start: Path | None = None) -> Path:
    """Walk up from start (default: the working directory) to the folder holding pyproject.toml."""
    current = (start or Path.cwd()).resolve()
    for folder in (current, *current.parents):
        if (folder / "pyproject.toml").exists():
            return folder
    raise FileNotFoundError("pyproject.toml not found above the working directory")


print(f"Python {platform.python_version()} ({sys.implementation.name}) on {platform.system()}")
print(f"Repository root: {find_repo_root().name}/")

Python 3.15.1 (cpython) on Linux
Repository root: python-course/


## Unit 1.1 — How Python Executes Code

**Objectives**

- Explain what "compiled" and "interpreted" each mean for CPython.
- Follow one line of source through every stage: tokens, AST, symbol table, code
  object, bytecode, evaluation.
- Find and recognise the cached bytecode files Python writes in `__pycache__`.
- Tell statements from expressions and read Python's block syntax.

**JS bridge**

| JavaScript concept | Python equivalent | The difference that bites |
| --- | --- | --- |
| V8 Ignition bytecode | CPython bytecode (`dis.dis`) | You can print CPython's bytecode from inside the program |
| TurboFan optimising JIT | Specializing adaptive interpreter (3.11+) and experimental JIT (3.13+) | CPython's JIT is off by default; most speed-ups come from specialisation |
| Esprima / acorn AST | `ast` module (built in) | The AST is part of the standard library |
| V8 code cache | `__pycache__/*.pyc` | The cache is a file next to your source, one per interpreter version |
| `{ … }` blocks, semicolons | Indentation, newlines | Indentation is syntax, not style |

In [2]:
import ast
import dis
import importlib
import importlib.util
import io
import os
import subprocess
import symtable
import sys
import tokenize
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    """Walk up from start (default: the working directory) to the folder holding pyproject.toml."""
    current = (start or Path.cwd()).resolve()
    for folder in (current, *current.parents):
        if (folder / "pyproject.toml").exists():
            return folder
    raise FileNotFoundError("pyproject.toml not found above the working directory")


repo_root = find_repo_root()
scratch_dir = repo_root / "notebooks" / "_scratch"
scratch_dir.mkdir(parents=True, exist_ok=True)
print(f"{sys.implementation.name=}, {sys.version_info[:2]=}")

sys.implementation.name='cpython', sys.version_info[:2]=(3, 15)


### 1.1.1 Compiled and interpreted

CPython does both. It **compiles** the whole source file to **bytecode** (instructions
for an imaginary machine), then **interprets** that bytecode in a loop. The imaginary
machine is a **stack-based virtual machine**: instructions push values onto a stack,
and operators pop their operands off it and push the result.

The cell disassembles a one-line function. Read it as a stack machine:

| Instruction | Stack after it |
| --- | --- |
| load `price` | `[price]` |
| load `tax` | `[price, tax]` |
| `BINARY_OP (+)` | `[price + tax]` |
| `RETURN_VALUE` | `[]` (value returned) |

> **Note:** exact instruction names change between Python versions (3.15 fuses the two
> loads into one `LOAD_FAST_BORROW_LOAD_FAST_BORROW` instruction). The shape stays the same.

In [3]:
def add_tax(price: int, tax: int) -> int:
    """Return price plus tax."""
    return price + tax


dis.dis(add_tax)

  1           RESUME                   0

  3           LOAD_FAST_BORROW_LOAD_FAST_BORROW 1 (price, tax)
              BINARY_OP                0 (+)
              RETURN_VALUE


**Specializing adaptive interpreter (new in 3.11, PEP 659).** After a function has run
a few times, CPython rewrites ("quickens") its bytecode in place with versions
specialised for the types it actually saw. `BINARY_OP (+)` becomes
`BINARY_OP_ADD_INT` when both operands were always `int`. If a different type turns up,
the instruction falls back to the generic version.

`dis.dis(..., adaptive=True)` shows the specialised bytecode. The cell calls `add_tax`
1000 times with integers first.

In [4]:
for _ in range(1_000):
    add_tax(100, 18)

dis.dis(add_tax, adaptive=True)

  1           RESUME_CHECK             0

  3           LOAD_FAST_BORROW_LOAD_FAST_BORROW 1 (price, tax)
              BINARY_OP_ADD_INT        0 (+)
              RETURN_VALUE


**JIT compiler (experimental, new in 3.13; faster in 3.14 and 3.15, PEP 744).** On top
of specialisation, CPython can compile hot traces of specialised bytecode to machine
code. The JIT is built in to the uv builds of 3.15 but **off by default**; set the
environment variable `PYTHON_JIT=1` to turn it on. `sys._jit` (3.14+) reports the state.

The cell checks this kernel, then starts a child interpreter with `PYTHON_JIT=1`.
Interpreters built without the JIT print a skip message.

In [5]:
if hasattr(sys, "_jit"):
    print(f"{sys._jit.is_available()=}  (built with the JIT)")
    print(f"{sys._jit.is_enabled()=}  (turned on in this kernel)")
    child_result = subprocess.run(
        [sys.executable, "-c", "import sys; print(sys._jit.is_enabled())"],
        env={**os.environ, "PYTHON_JIT": "1"},
        capture_output=True,
        text=True,
        check=True,
    )
    print(f"with PYTHON_JIT=1: is_enabled() -> {child_result.stdout.strip()}")
else:
    print("Skipped: sys._jit needs Python 3.14+")

sys._jit.is_available()=True  (built with the JIT)
sys._jit.is_enabled()=False  (turned on in this kernel)
with PYTHON_JIT=1: is_enabled() -> True


> **JS bridge:** V8 works the same way in outline. **Ignition** compiles JavaScript to
> bytecode and interprets it; **TurboFan** compiles hot functions to optimised machine
> code using type feedback. CPython's *adaptive interpreter* plays the role of
> Ignition's type feedback, and its JIT is a much younger, simpler TurboFan.

### 1.1.2 The pipeline

```mermaid
flowchart LR
    source["source text"] --> tokenizer["tokenizer"]
    tokenizer -- "tokens" --> parser["PEG parser"]
    parser -- "AST" --> compiler["compiler + symbol table"]
    compiler -- "code object (bytecode)" --> eval_loop["eval loop (VM)"]
    eval_loop --> result["side effects / result"]
```

<details><summary>ASCII fallback</summary>

```text
source --tokenize--> tokens --PEG parser--> AST --compiler (symbol table)--> code object --eval loop--> result
```

</details>

| Stage | What it produces | Module to look at it |
| --- | --- | --- |
| Tokenizer | `NAME`, `OP`, `NUMBER`… tokens | `tokenize` |
| PEG parser (3.9+, PEP 617) | Abstract syntax tree | `ast.parse`, `ast.dump` |
| Symbol table | Which names are local, global, free | `symtable` |
| Compiler | A **code object**: bytecode + constants + names | `compile`, `code.co_*` |
| Eval loop | Runs the bytecode on the VM | `exec`, `eval`, `dis` |

**Stage 1: tokenizer.** It splits source text into tokens.

In [6]:
source_line = "total_in_rupees = price * 2 + 18\n"
for token in tokenize.generate_tokens(io.StringIO(source_line).readline):
    print(f"{tokenize.tok_name[token.type]:<10} {token.string!r}")

NAME       'total_in_rupees'
OP         '='
NAME       'price'
OP         '*'
NUMBER     '2'
OP         '+'
NUMBER     '18'
NEWLINE    '\n'
ENDMARKER  ''


**Stage 2: parser.** It builds an **abstract syntax tree** from the tokens.
`ast.dump(ast.parse(...))` prints the tree. Notice that operator precedence is already
resolved: `price * 2` is nested inside the `+`.

In [7]:
print(ast.dump(ast.parse(source_line), indent=2))

Module(
  body=[
    Assign(
      targets=[
        Name(id='total_in_rupees', ctx=Store())],
      value=BinOp(
        left=BinOp(
          left=Name(id='price'),
          op=Mult(),
          right=Constant(value=2)),
        op=Add(),
        right=Constant(value=18)))])


**Stage 3: symbol table.** Before generating bytecode, the compiler decides the scope of
every name. Inside a function, a name that is assigned is **local**; a name that is only
read is **global** (or a closure variable). This is decided at compile time, which is
why Unit 1.11's `UnboundLocalError` exists.

In [8]:
function_source = """
def add_gst(price):
    taxed_price = price * gst_rate
    return taxed_price
"""
module_table = symtable.symtable(function_source, "<demo>", "exec")
function_table = module_table.lookup("add_gst").get_namespace()
print(f"{function_table.get_parameters()=}")
print(f"{function_table.get_locals()=}")
print(f"{function_table.get_globals()=}")

function_table.get_parameters()=('price',)
function_table.get_locals()=('price', 'taxed_price')
function_table.get_globals()=('gst_rate',)


**Stage 4: compiler.** `compile(source, filename, mode)` returns a **code object**. The
mode decides what source is allowed: `"exec"` for statements (a module), `"eval"` for a
single expression. The code object holds the bytecode (`co_code`) and the tables it
refers to.

In [9]:
module_code = compile(source_line, "<demo>", "exec")
expression_code = compile("price * 2 + 18", "<demo>", "eval")
print(f"{type(module_code).__name__=}")
print(f"{module_code.co_names=}")
print(f"{module_code.co_consts=}")
print(f"{len(module_code.co_code)=} bytes of bytecode")
print(f"{expression_code.co_names=}")

type(module_code).__name__='code'
module_code.co_names=('price', 'total_in_rupees')
module_code.co_consts=(2,)
len(module_code.co_code)=40 bytes of bytecode
expression_code.co_names=('price',)


`dis.dis` turns the code object's bytes back into readable instructions. At module level,
names are stored with `STORE_NAME` (a dict write) instead of `STORE_FAST` (a slot in the
frame), which is one reason code inside functions runs faster.

In [10]:
dis.dis(module_code)

  0           RESUME                   0

  1           LOAD_NAME                0 (price)
              LOAD_SMALL_INT           2
              BINARY_OP                5 (*)
              LOAD_SMALL_INT          18
              BINARY_OP                0 (+)
              STORE_NAME               1 (total_in_rupees)
              LOAD_COMMON_CONSTANT     7 (None)
              RETURN_VALUE


**Stage 5: eval loop.** `exec(code, namespace)` runs a code object with a dict you
supply as its global namespace. The assignment lands in that dict, not in the notebook's
globals. `eval` runs an expression and returns its value.

In [11]:
demo_namespace: dict[str, object] = {"price": 500}
exec(module_code, demo_namespace)
print(f"{demo_namespace['total_in_rupees']=}")
print(f"{'total_in_rupees' in globals()=}")
print(f"{eval(expression_code, {'price': 10})=}")

demo_namespace['total_in_rupees']=1018
'total_in_rupees' in globals()=False
eval(expression_code, {'price': 10})=38


> **Warning:** never `exec` or `eval` text that came from a user. It runs with the
> full power of your program. Use `ast.literal_eval` to parse literals safely.

### 1.1.3 Bytecode caching

Compiling is not free, so when a module is **imported**, CPython saves the code object
to `__pycache__/<module>.cpython-315.pyc` next to the source. The next import loads
the `.pyc` directly, unless the source changed. (The script you run directly, and
notebook cells, are never cached.)

The cell writes a small module into `notebooks/_scratch/` and imports it.
`importlib.util.cache_from_source` computes where its cache file goes.

In [12]:
module_path = scratch_dir / "price_rules.py"
module_path.write_text(
    "GST_RATE = 18\n\n\ndef with_gst(price: int) -> float:\n"
    '    """Return price including GST."""\n'
    "    return price * (100 + GST_RATE) / 100\n",
    encoding="utf-8",
)
if str(scratch_dir) not in sys.path:
    sys.path.insert(0, str(scratch_dir))
importlib.invalidate_caches()
price_rules = importlib.import_module("price_rules")  # same as: import price_rules

cache_path = Path(importlib.util.cache_from_source(str(module_path)))
print(f"{price_rules.with_gst(1000)=}")
print(f"{cache_path.relative_to(scratch_dir)=}")
print(f"{cache_path.exists()=}")

price_rules.with_gst(1000)=1180.0
cache_path.relative_to(scratch_dir)=PosixPath('__pycache__/price_rules.cpython-315.pyc')
cache_path.exists()=True


Every `.pyc` starts with a 4-byte **magic number** that changes whenever the bytecode
format changes. A `.pyc` with the wrong magic number is ignored and rebuilt, which is
why each Python version gets its own file name tag (`sys.implementation.cache_tag`).
The next 12 bytes hold flags and the source's modification time and size.

In [13]:
pyc_header = cache_path.read_bytes()[:16]
print(f"{importlib.util.MAGIC_NUMBER=}")
print(f"{pyc_header[:4]=}")
print(f"{pyc_header[:4] == importlib.util.MAGIC_NUMBER=}")
print(f"{sys.implementation.cache_tag=}")

importlib.util.MAGIC_NUMBER=b'R\x0e\r\n'
pyc_header[:4]=b'R\x0e\r\n'
pyc_header[:4] == importlib.util.MAGIC_NUMBER=True
sys.implementation.cache_tag='cpython-315'


> **Note:** `__pycache__/` belongs in `.gitignore` (this repository has it). Set
> `PYTHONDONTWRITEBYTECODE=1` to stop Python writing caches, e.g. in read-only containers.

### 1.1.4 Statements vs expressions

An **expression** produces a value (`price * 2`, `f(x)`, `a if c else b`). A
**statement** does something (`x = 1`, `if`, `for`, `def`, `import`, `return`). Every
expression can be used as a statement, but not the reverse: `eval` accepts only
expressions. The AST labels each kind.

In [14]:
sample_module = ast.parse("discount = 10\nprice * 2\nif discount:\n    pass")
for node in sample_module.body:
    print(f"{type(node).__name__:<6} {ast.unparse(node)!r}")

try:
    eval("discount = 10")
except SyntaxError as error:
    print(f"{type(error).__name__}: {error.msg}")

Assign 'discount = 10'
Expr   'price * 2'
If     'if discount:\n    pass'
SyntaxError: invalid syntax


**Indentation is block syntax.** A colon opens a block, and the indented lines below it
form the body; there are no braces. PEP 8 uses 4 spaces. Wrong indentation is a
*compile-time* error (`IndentationError`, a subclass of `SyntaxError`), so the cell
compiles bad source as a string to catch it.

In [15]:
badly_indented_source = "if True:\nprint('missing indent')\n"
try:
    compile(badly_indented_source, "<demo>", "exec")
except IndentationError as error:
    print(f"{type(error).__name__}: {error.msg} (line {error.lineno})")
print(f"{issubclass(IndentationError, SyntaxError)=}")

IndentationError: expected an indented block after 'if' statement on line 1 (line 2)
issubclass(IndentationError, SyntaxError)=True


**`pass`** is the statement that does nothing. Python needs it where a block is required
but you have nothing to put there yet (JS would use `{}`).

In [16]:
def not_written_yet() -> None:
    """Placeholder whose body is intentionally empty."""
    pass


class EmptyMarker:
    pass


print(f"{not_written_yet()=}, {EmptyMarker.__name__=}")

not_written_yet()=None, EmptyMarker.__name__='EmptyMarker'


**Line continuation.** A newline ends a statement *unless* it is inside `()`, `[]` or
`{}`. Wrap long expressions in brackets instead of ending a line with a backslash:

```python
order_total = 499 + \
    150          # explicit continuation: breaks if a space follows the backslash
```

`ruff format` (and Black) never produce backslash continuations; they add parentheses.

In [17]:
line_item_prices = [
    499,
    150,
    1_200,
]
order_total = (
    sum(line_item_prices)
    + 40  # delivery charge
    - 100  # coupon
)
print(f"{order_total=}, {line_item_prices=}")

order_total=1789, line_item_prices=[499, 150, 1200]


#### Under the hood: one code object per function

A function is a thin wrapper around a code object compiled once, when its `def` ran.
`__code__` exposes it, with the variable names and constants the bytecode uses.

In [18]:
add_tax_code = add_tax.__code__
print(f"{add_tax_code.co_name=}")
print(f"{add_tax_code.co_varnames=}")
print(f"{add_tax_code.co_argcount=}")
print(f"{add_tax_code.co_consts=}")
print(f"{add_tax.__code__ is add_tax_code=} (compiled once, reused on every call)")

add_tax_code.co_name='add_tax'
add_tax_code.co_varnames=('price', 'tax')
add_tax_code.co_argcount=2
add_tax_code.co_consts=('Return price plus tax.',)
add_tax.__code__ is add_tax_code=True (compiled once, reused on every call)


#### Predict the output

**Predict the output.**

```python
namespace = {}
exec("x = 2\ny = x * 21", namespace)
print(namespace["y"], "y" in globals())
```

<details><summary>Answer</summary>

`42 False`. `exec` used `namespace` as the global namespace of the code it ran, so both
`x` and `y` were stored there and not in the notebook's globals.

</details>

Run the next cell to check your prediction.

In [19]:
namespace: dict[str, object] = {}
exec("x = 2\ny = x * 21", namespace)
print(namespace["y"], "y" in globals())

42 False


**Predict the output.**

```python
print(type(ast.parse("1 + 2").body[0]).__name__)
```

<details><summary>Answer</summary>

`Expr`. `ast.parse` defaults to `"exec"` mode, so the source is a module whose body is a
list of **statements**. A bare expression becomes an *expression statement* (`Expr`)
that wraps the `BinOp`.

</details>

Run the next cell to check your prediction.

In [20]:
print(type(ast.parse("1 + 2").body[0]).__name__)
print(type(ast.parse("1 + 2").body[0].value).__name__)

Expr
BinOp


### Recap — Unit 1.1

| Concept | One-line takeaway | JS equivalent |
| --- | --- | --- |
| Compiled *and* interpreted | Source → bytecode once, then a stack VM runs it | V8 Ignition |
| Adaptive interpreter | Hot bytecode is specialised for the types it sees (3.11+) | Inline caches / type feedback |
| JIT | Experimental, off by default, enable with `PYTHON_JIT=1` (3.13+) | TurboFan |
| Pipeline | tokenize → PEG parser → AST → symbol table → code object → eval loop | parse → AST → bytecode → execute |
| `exec` / `eval` | Run code in a namespace dict you control; never on untrusted input | `eval`, `new Function` |
| `__pycache__` | Imported modules cache bytecode in `.pyc`, checked by magic number | V8 code cache |
| Statement vs expression | Only expressions have values; `eval` accepts only expressions | Same split |
| Blocks | Indentation + colon; `pass` for an empty body; wrap long lines in brackets | `{ }` and `;` |

**Further reading**

- [`dis` — Disassembler](https://docs.python.org/3/library/dis.html),
  [`ast`](https://docs.python.org/3/library/ast.html),
  [`symtable`](https://docs.python.org/3/library/symtable.html),
  [`tokenize`](https://docs.python.org/3/library/tokenize.html)
- [Built-in `compile`, `exec`, `eval`](https://docs.python.org/3/library/functions.html#compile),
  [`importlib.util.cache_from_source`](https://docs.python.org/3/library/importlib.html#importlib.util.cache_from_source)
- [Language reference: lexical analysis (indentation, line joining)](https://docs.python.org/3/reference/lexical_analysis.html)
- [PEP 617 — PEG parser](https://peps.python.org/pep-0617/),
  [PEP 659 — Specializing adaptive interpreter](https://peps.python.org/pep-0659/),
  [PEP 744 — JIT compilation](https://peps.python.org/pep-0744/),
  [PEP 3147 — `__pycache__` directories](https://peps.python.org/pep-3147/)

## Unit 1.2 — Objects, Names and Mutability

**Objectives**

- Describe every value as an object with an identity, a type and a value.
- Explain variables as names bound to objects in a namespace dict.
- Predict when a function call changes the caller's data.
- Classify built-in types as mutable or immutable, and use `is` vs `==` correctly.
- Explain why names have no type but objects do.

**JS bridge**

| JavaScript concept | Python equivalent | The difference that bites |
| --- | --- | --- |
| Primitives (`number`, `string`) vs objects | Everything is an object | `int` and `str` are objects too, just immutable ones |
| `let x = obj` copies a reference | `x = obj` binds a name | Same idea, but it applies to *every* value |
| `===` | `==` (value) and `is` (identity) | `is` on numbers or strings depends on caching; never use it for that |
| `Object.is(a, b)` | `a is b` | `is` compares identity, not value |
| `typeof x` | `type(x)` / `isinstance(x, T)` | `isinstance` respects subclasses |
| `delete obj.key` | `del name` | `del` unbinds a name; the object lives on while referenced |

The next cell is this unit's setup.

In [21]:
import inspect
import math
import sys
import types
import warnings


def show_size(label: str, obj: object) -> None:
    """Print the shallow size in bytes, the type and the id of an object."""
    print(f"{label}: {sys.getsizeof(obj)} bytes, type={type(obj).__name__}, id={id(obj)}")


print(f"{sys.version_info[:2]=}")

sys.version_info[:2]=(3, 15)


### 1.2.1 Everything is an object

Every object has three things:

| Property | How to read it | Can it change? |
| --- | --- | --- |
| **Identity** | `id(obj)` (CPython: the memory address) | Never, for the object's lifetime |
| **Type** | `type(obj)` | Never (in practice) |
| **Value** | the object itself, `repr(obj)` | Only for mutable types |

> **Note:** `id()` values differ on every run. The demos compare ids; they never rely on
> the number itself.

In [22]:
price_in_rupees = 499
cart_items = ["pen", "notebook"]
for value in (price_in_rupees, "Asha", cart_items, 3.5, None):
    print(f"type={type(value).__name__:<8} value={value!r:<20} has an id: {id(value) > 0}")

type=int      value=499                  has an id: True
type=str      value='Asha'               has an id: True
type=list     value=['pen', 'notebook']  has an id: True
type=float    value=3.5                  has an id: True
type=NoneType value=None                 has an id: True


Types, functions, modules and classes are objects too: you can put them in a list, pass
them to functions and give them attributes. Even `int` is an object, whose type is
`type`.

In [23]:
def apply_discount(amount: float) -> float:
    """Return the amount with a 10% discount."""
    return amount * 0.9


first_class_values = [int, apply_discount, math, type, len]
for value in first_class_values:
    print(f"{getattr(value, '__name__', value)!s:<16} is a {type(value).__name__}")

print(f"{type(int) is type=}, {type(type) is type=}")
print(f"{inspect.isclass(int)=}, {inspect.isfunction(apply_discount)=}, {inspect.ismodule(math)=}")
apply_discount.author = "Asha"  # functions accept new attributes like any object
print(f"{apply_discount.author=}")

int              is a type
apply_discount   is a function
math             is a module
type             is a type
len              is a builtin_function_or_method
type(int) is type=True, type(type) is type=True
inspect.isclass(int)=True, inspect.isfunction(apply_discount)=True, inspect.ismodule(math)=True
apply_discount.author='Asha'


### 1.2.2 Names and binding

A variable is a **name**: a key in a namespace dict, bound to an object. At module level
(and in a notebook) that dict is `globals()`. Writing to the dict is the same as
assigning a variable.

In [24]:
delivery_fee = 40
module_namespace = globals()
print(f"{type(module_namespace).__name__=}")
print(f"{module_namespace['delivery_fee']=}")

module_namespace["service_fee"] = 15  # same effect as: service_fee = 15
print(f"{service_fee=}")  # noqa: F821  (created through the dict, invisible to linters)

type(module_namespace).__name__='dict'
module_namespace['delivery_fee']=40
service_fee=15


**Assignment never copies.** `b = a` binds a second name to the *same* object. If the
object is mutable, a change through one name is visible through the other.

```mermaid
flowchart LR
    subgraph names ["globals()"]
        morning["morning_orders"]
        alias["evening_orders"]
    end
    list_object["list object<br/>['idli', 'dosa', 'vada']"]
    morning --> list_object
    alias --> list_object
```

<details><summary>ASCII fallback (Python Tutor style)</summary>

```text
Frames            Objects
+----------------+
| morning_orders | ----+
| evening_orders | ----+--> list ['idli', 'dosa', 'vada']
+----------------+
```

</details>

In [25]:
morning_orders = ["idli", "dosa"]
evening_orders = morning_orders  # a second name, not a second list
evening_orders.append("vada")
print(f"{morning_orders=}")
print(f"{evening_orders is morning_orders=}")

morning_orders=['idli', 'dosa', 'vada']
evening_orders is morning_orders=True


**`del` removes a name, not the object.** The object survives as long as something
still refers to it. `sys.getrefcount` counts the references (it reports one extra for
its own argument). CPython frees the object when the count reaches zero.

In [26]:
reference_count_before = sys.getrefcount(morning_orders)
del morning_orders
print(f"{'morning_orders' in globals()=}")
print(f"{evening_orders=}  (the list is still alive)")
print(f"{sys.getrefcount(evening_orders) == reference_count_before - 1=}")
try:
    print(morning_orders)  # noqa: F821
except NameError as error:
    print(f"{type(error).__name__}: {error}")

'morning_orders' in globals()=False
evening_orders=['idli', 'dosa', 'vada']  (the list is still alive)
sys.getrefcount(evening_orders) == reference_count_before - 1=True
NameError: name 'morning_orders' is not defined


### 1.2.3 Call by object reference

Calling a function binds each parameter name to the **same object** the caller passed
(sometimes called *call by sharing*). Two consequences:

| Inside the function | Effect on the caller |
| --- | --- |
| **Mutate** the object (`items.append(...)`) | Visible: it is the same object |
| **Rebind** the parameter (`items = [...]`) | Invisible: only the local name moved |

In [27]:
def add_free_gift(order_items: list[str]) -> None:
    """Mutate the caller's list by appending a gift."""
    order_items.append("free sticker")


def replace_order(order_items: list[str]) -> None:
    """Rebind the local name only; the caller's list is untouched."""
    order_items = ["something else"]
    print(f"  inside replace_order: {order_items=}")


customer_order = ["tea"]
add_free_gift(customer_order)
print(f"after add_free_gift: {customer_order=}")
replace_order(customer_order)
print(f"after replace_order: {customer_order=}")

after add_free_gift: customer_order=['tea', 'free sticker']
  inside replace_order: order_items=['something else']
after replace_order: customer_order=['tea', 'free sticker']


The same rule applies to immutable arguments: `count += 1` inside a function rebinds the
local name to a new `int`, so the caller sees no change. Return the new value instead.

> **JS bridge:** Python has **no primitives**. In JS, numbers and strings are passed by
> value and objects by reference-copy. In Python *everything* is passed the same way;
> `int` and `str` only *look* like they are passed by value because they cannot be
> mutated.

In [28]:
def increment(counter: int) -> int:
    """Return counter + 1; rebinding the parameter does not touch the caller."""
    counter += 1
    return counter


visit_count = 10
new_visit_count = increment(visit_count)
print(f"{visit_count=}, {new_visit_count=}")

visit_count=10, new_visit_count=11


### 1.2.4 Mutable vs immutable types

| Immutable (value fixed at creation) | Mutable (value can change in place) |
| --- | --- |
| `int`, `float`, `complex`, `bool` | `list` |
| `decimal.Decimal`, `fractions.Fraction` | `dict` |
| `str` | `set` |
| `bytes` | `bytearray` |
| `tuple` (its items may be mutable) | `memoryview` over a writable buffer |
| `frozenset` | `collections.deque`, `array.array` |
| `frozendict` (new in 3.15) | instances of most user-defined classes |
| `range`, `NoneType`, functions' code objects | `types.SimpleNamespace` |

"Changing" an immutable object really creates a **new** object and rebinds the name.
Watch the identity across `+=` (comparing ids, never printing them):

In [29]:
stock_count = 1_000
id_before = id(stock_count)
stock_count += 1
print(f"int:  same object after += ? {id(stock_count) == id_before}")

greeting_text = "Namaste"
id_before = id(greeting_text)
greeting_text += ", Asha"
print(f"str:  same object after += ? {id(greeting_text) == id_before}")

basket = ["rice"]
id_before = id(basket)
basket += ["dal"]  # list += extends in place
print(f"list: same object after += ? {id(basket) == id_before}")

int:  same object after += ? False
str:  same object after += ? False
list: same object after += ? True


The rule of thumb: an immutable type has no method that modifies it. `str.upper()`
returns a new string; `list.sort()` changes the list and returns `None`. Trying to
assign into an immutable object raises `TypeError`.

In [30]:
coordinates = (12.97, 77.59)
try:
    coordinates[0] = 13.0
except TypeError as error:
    print(f"{type(error).__name__}: {error}")

city = "bengaluru"
print(f"{city.upper()=}, {city=}  (unchanged)")
print(f"{basket.sort()=}  (sorted in place, returns None)")

TypeError: 'tuple' object does not support item assignment
city.upper()='BENGALURU', city='bengaluru'  (unchanged)
basket.sort()=None  (sorted in place, returns None)


### 1.2.5 Identity vs equality

| Operator | Question | Implemented by |
| --- | --- | --- |
| `a == b` | Do they have equal **values**? | `a.__eq__(b)`, which each type defines |
| `a is b` | Are they the **same object**? | Identity comparison; cannot be overridden |

Use `is` only for singletons: `None`, `True`, `False`, `...`, and sentinel objects you
create yourself. Use `==` for everything else.

In [31]:
class Price:
    """An amount in paise that compares by value."""

    def __init__(self, paise: int) -> None:
        self.paise = paise

    def __eq__(self, other: object) -> bool:
        if not isinstance(other, Price):
            return NotImplemented
        return self.paise == other.paise


first_price = Price(49_900)
second_price = Price(49_900)
print(f"{first_price == second_price=}  (__eq__ compares paise)")
print(f"{first_price is second_price=}  (two distinct objects)")
first_list, second_list = [1, 2], [1, 2]
print(f"{first_list == second_list=}, {first_list is second_list=}")

first_price == second_price=True  (__eq__ compares paise)
first_price is second_price=False  (two distinct objects)
first_list == second_list=True, first_list is second_list=False


**Small-integer cache.** CPython pre-creates small `int` objects and reuses them, so two
names bound to the same small value can be the *same object*. The range changed:

| Python | Cached range |
| --- | --- |
| up to 3.14 | `-5` … `256` |
| **3.15** (*changed in 3.15*) | `-5` … `1024` |

The cell builds numbers at run time with `int("…")`. Literals in one cell would be
merged into a single constant by the compiler and hide the effect. This is an
implementation detail, which is exactly why you must not use `is` for numbers.

In [32]:
for number_text in ["256", "257", "1024", "1025"]:
    first_number, second_number = int(number_text), int(number_text)
    is_same_object = first_number is second_number
    print(f"{number_text:>5}: equal={first_number == second_number}, same object={is_same_object}")
if sys.version_info < (3, 15):
    print("Note: on this Python the cache ends at 256, so 1024 is not shared")

  256: equal=True, same object=True
  257: equal=True, same object=True
 1024: equal=True, same object=True
 1025: equal=True, same object=False


**String interning.** CPython also shares some strings: identifier-like literals and
names are *interned* automatically. Strings built at run time usually are not.
`sys.intern` interns one on request, which can save memory and makes later comparisons
identity checks (used inside dicts of attribute names).

In [33]:
built_status = "".join(["sh", "ipped"])
literal_status = "shipped"
print(f"{built_status == literal_status=}, {built_status is literal_status=}")

interned_status = sys.intern(built_status)
print(f"{interned_status is literal_status=}  (sys.intern returned the shared copy)")

built_status == literal_status=True, built_status is literal_status=False
interned_status is literal_status=True  (sys.intern returned the shared copy)


Because results like these vary by version and context, the compiler warns when you use
`is` with a literal. The cell compiles such code and captures the **`SyntaxWarning`**
(new in 3.8) instead of letting it print.

In [34]:
suspicious_source = "order_count = 5\nprint(order_count is 5)"
with warnings.catch_warnings(record=True) as caught_warnings:
    warnings.simplefilter("always")
    compile(suspicious_source, "<demo>", "exec")
for caught_warning in caught_warnings:
    print(f"{caught_warning.category.__name__}: {caught_warning.message}")

### 1.2.6 Dynamic typing

**Names have no types; objects do.** One name can be rebound to objects of different
types. Type hints (`price: int`) document intent and help tools such as pyright, but
the interpreter does not enforce them (Phase 4).

In [35]:
shipment_reference = 1042
print(f"{type(shipment_reference).__name__=}")
shipment_reference = "SHIP-1042"
print(f"{type(shipment_reference).__name__=}")

hinted_quantity: int = "three"  # legal at run time: hints are not checked
print(f"{hinted_quantity=}, {type(hinted_quantity).__name__=}")

type(shipment_reference).__name__='int'
type(shipment_reference).__name__='str'
hinted_quantity='three', type(hinted_quantity).__name__='str'


**`type()` vs `isinstance()`.** `type(x) is T` matches one exact class.
`isinstance(x, T)` also accepts subclasses, and takes a tuple (or a `X | Y` union) of
types. Prefer `isinstance` for checks.

In [36]:
class DiscountedPrice(Price):
    """A Price subclass."""


sale_price = DiscountedPrice(39_900)
print(f"{type(sale_price) is Price=}")
print(f"{isinstance(sale_price, Price)=}")
print(f"{isinstance(3.5, int | float)=}")
print(f"{isinstance(True, int)=}  (bool is a subclass of int; see 1.3.6)")

type(sale_price) is Price=False
isinstance(sale_price, Price)=True
isinstance(3.5, int | float)=True
isinstance(True, int)=True  (bool is a subclass of int; see 1.3.6)


**Duck typing preview.** Python code usually does not check types at all: it calls the
method it needs and lets anything that has it work ("if it walks like a duck…").
Phase 2 formalises this with protocols.

In [37]:
class PaperReceipt:
    """A receipt that is printed."""

    def render(self) -> str:
        """Return a description of the receipt."""
        return "Printed receipt"


class EmailReceipt:
    """A receipt that is emailed; unrelated to PaperReceipt."""

    def render(self) -> str:
        """Return a description of the receipt."""
        return "Emailed receipt"


def send_receipt(receipt: PaperReceipt | EmailReceipt) -> None:
    """Works with any object that has a render() method."""
    print(receipt.render())


ad_hoc_receipt = types.SimpleNamespace(render=lambda: "Ad-hoc receipt")
for receipt in (PaperReceipt(), EmailReceipt(), ad_hoc_receipt):
    send_receipt(receipt)

Printed receipt
Emailed receipt
Ad-hoc receipt


#### Under the hood: names live in `__dict__`, objects carry headers

A module's globals are literally its `__dict__`, and an instance's attributes are in its
own `__dict__`. Every object also carries a header (reference count and type pointer),
which is why even `0` costs bytes. `show_size` (from the shared setup) prints the
shallow size.

In [38]:
print(f"{globals() is sys.modules['__main__'].__dict__=}")
print(f"{vars(first_price)=}")
show_size("small int 0", 0)
show_size("None", None)
show_size("empty list", [])

globals() is sys.modules['__main__'].__dict__=True
vars(first_price)={'paise': 49900}
small int 0: 28 bytes, type=int, id=22124928
None: 16 bytes, type=NoneType, id=22109848
empty list: 56 bytes, type=list, id=139669134798016


#### Predict the output

**Predict the output.**

```python
shopping_list = ["milk"]
other_list = shopping_list
other_list = other_list + ["bread"]
print(shopping_list)
```

<details><summary>Answer</summary>

`['milk']`. `other_list + [...]` builds a **new** list and rebinds `other_list` to it;
`shopping_list` still points at the original. Compare `basket += [...]` in 1.2.4,
which extends the list in place.

</details>

Run the next cell to check your prediction.

In [39]:
shopping_list = ["milk"]
other_list = shopping_list
other_list = other_list + ["bread"]
print(shopping_list)

['milk']


**Predict the output.**

```python
def reset(scores):
    scores.clear()
    scores = [100]

team_scores = [7, 9]
reset(team_scores)
print(team_scores)
```

<details><summary>Answer</summary>

`[]`. `clear()` mutates the caller's list; the rebinding on the next line only moves
the local name.

</details>

Run the next cell to check your prediction.

In [40]:
def reset(scores: list[int]) -> None:
    """Clear the caller's list, then rebind the local name."""
    scores.clear()
    scores = [100]


team_scores = [7, 9]
reset(team_scores)
print(team_scores)

[]


### Recap — Unit 1.2

| Concept | One-line takeaway | JS equivalent |
| --- | --- | --- |
| Everything is an object | Identity, type and value; classes, functions, modules too | Objects, but JS also has primitives |
| Names and binding | A variable is a key in a namespace dict bound to an object | Variable holds a reference |
| Assignment | Never copies; two names can share one object | Same for objects |
| `del` | Removes a name; the object dies when unreferenced | `delete` removes a property |
| Call by object reference | Mutation is visible to the caller, rebinding is not | Same for objects; primitives copy |
| Mutable vs immutable | `list`/`dict`/`set` change in place; `int`/`str`/`tuple` never do | Objects vs primitives |
| `is` vs `==` | `is` for singletons only; small-int cache is `-5..1024` in 3.15 | `Object.is` vs `===` |
| Dynamic typing | Names untyped, objects typed; prefer `isinstance` and duck typing | `typeof`, `instanceof` |

**Further reading**

- [Data model: objects, values and types](https://docs.python.org/3/reference/datamodel.html#objects-values-and-types)
- [Naming and binding](https://docs.python.org/3/reference/executionmodel.html#naming-and-binding)
- [Programming FAQ: how do I write a function with output parameters?](https://docs.python.org/3/faq/programming.html#how-do-i-write-a-function-with-output-parameters-call-by-reference)
- [`sys.intern`](https://docs.python.org/3/library/sys.html#sys.intern),
  [`isinstance`](https://docs.python.org/3/library/functions.html#isinstance),
  [`id`](https://docs.python.org/3/library/functions.html#id)
- [PEP 8 — comparisons to singletons](https://peps.python.org/pep-0008/#programming-recommendations)

## Unit 1.3 — Numeric and Boolean Data Types

**Objectives**

- Use `int`, `float`, `Decimal`, `Fraction` and `complex`, and choose the right one for
  money, measurements and exact ratios.
- Predict `//`, `%` and `round()` results, including negative numbers and halves.
- Explain why `bool` is an `int`, what `None` is, and which values are falsy.
- Convert between types explicitly, knowing what raises.

**JS bridge**

| JavaScript concept | Python equivalent | The difference that bites |
| --- | --- | --- |
| `number` (one type) | `int` and `float` (two types) | `7 / 2 == 3.5`, but `7 // 2 == 3`; ints never lose precision |
| `BigInt` (`10n`) | `int` | Every `int` is a big int; no suffix, no mixing errors |
| `-7 % 2 === -1` | `-7 % 2 == 1` | `%` takes the sign of the divisor |
| `Math.round(2.5) === 3` | `round(2.5) == 2` | Banker's rounding (round half to even) |
| `1 / 0 === Infinity` | `1 / 0` raises `ZeroDivisionError` | Division by zero is an error |
| `null` and `undefined` | `None` only | Missing dict keys raise `KeyError` |
| `[]` and `{}` are truthy | `[]` and `{}` are falsy | Empty containers are false |
| `1 + "1" === "11"` | `TypeError` | No implicit coercion |

The next cell is this unit's setup.

In [41]:
import cmath
import math
import sys
import timeit
from collections.abc import Callable
from decimal import ROUND_HALF_EVEN, ROUND_HALF_UP, Decimal, getcontext, localcontext
from fractions import Fraction


def show_size(label: str, obj: object) -> None:
    """Print the shallow size in bytes, the type and the id of an object."""
    print(f"{label}: {sys.getsizeof(obj)} bytes, type={type(obj).__name__}, id={id(obj)}")


PAISE = Decimal("0.01")  # quantize() target for rupee amounts
print(f"{sys.version_info[:2]=}, {getcontext().prec=}")

sys.version_info[:2]=(3, 15), getcontext().prec=28


### 1.3.0 Python's built-in data types at a glance

Units 1.3–1.7 teach these types:

| Category | Type | Mutable | Ordered | Hashable | Literal / constructor | Section |
| --- | --- | --- | --- | --- | --- | --- |
| Numeric | `int` | No | — | Yes | `42`, `0xff`, `1_000` | 1.3.1 |
| Numeric | `float` | No | — | Yes | `3.14`, `1e-3` | 1.3.2 |
| Numeric | `complex` | No | — | Yes | `3 + 4j` | 1.3.5 |
| Numeric (stdlib) | `Decimal`, `Fraction` | No | — | Yes | `Decimal("0.1")`, `Fraction(1, 3)` | 1.3.3–1.3.4 |
| Boolean | `bool` | No | — | Yes | `True`, `False` | 1.3.6 |
| Null | `NoneType` | No | — | Yes | `None` | 1.3.7 |
| Text | `str` | No | Yes | Yes | `"hi"`, `f"{name}"` | 1.4 |
| Binary | `bytes` | No | Yes | Yes | `b"hi"` | 1.4.5 |
| Binary | `bytearray` | Yes | Yes | No | `bytearray(b"hi")` | 1.4.5 |
| Binary | `memoryview` | Depends on buffer | Yes | Read-only views only | `memoryview(data)` | 1.4.6 |
| Sequence | `list` | Yes | Yes | No | `[1, 2]` | 1.5 |
| Sequence | `tuple` | No | Yes | If all items are | `(1, 2)`, `(1,)` | 1.5.5 |
| Sequence | `range` | No | Yes | Yes | `range(10)` | 1.5.7 |
| Mapping | `dict` | Yes | Insertion order | No | `{"a": 1}` | 1.6 |
| Mapping | `frozendict` (3.15) | No | Insertion order | If all values are | `frozendict(a=1)` | 1.6.5 |
| Set | `set` | Yes | No | No | `{1, 2}`, `set()` | 1.6.6 |
| Set | `frozenset` | No | No | Yes | `frozenset({1, 2})` | 1.6.6 |

**Hashable** means `hash(obj)` works, so the object can be a dict key or a set member
(1.6.2). The cell checks the *Hashable* column for one sample of each row, including the
"it depends" cases. Unhashable objects raise `TypeError`, except a writable
`memoryview`, which raises `ValueError`. `frozendict` is *new in 3.15*, so it is added
only on 3.15+.

In [42]:
hashability_samples: list[tuple[str, object]] = [
    ("int", 42),
    ("float", 3.14),
    ("complex", 3 + 4j),
    ("Decimal", Decimal("0.1")),
    ("Fraction", Fraction(1, 3)),
    ("bool", True),
    ("NoneType", None),
    ("str", "hi"),
    ("bytes", b"hi"),
    ("bytearray", bytearray(b"hi")),
    ("memoryview (read-only)", memoryview(b"hi")),
    ("memoryview (writable)", memoryview(bytearray(b"hi"))),
    ("list", [1, 2]),
    ("tuple of ints", (1, 2)),
    ("tuple holding a list", (1, [2])),
    ("range", range(10)),
    ("dict", {"a": 1}),
    ("set", {1, 2}),
    ("frozenset", frozenset({1, 2})),
]
if sys.version_info >= (3, 15):
    hashability_samples += [
        ("frozendict of ints", frozendict(a=1)),
        ("frozendict holding a list", frozendict(a=[1])),
    ]
else:
    print("Skipped frozendict rows: new in 3.15")

for label, sample in hashability_samples:
    try:
        hash(sample)
        verdict = "hashable"
    except (TypeError, ValueError) as error:  # writable memoryview raises ValueError
        verdict = f"unhashable ({type(error).__name__})"
    print(f"{label:<26} {verdict}")

int                        hashable


float                      hashable
complex                    hashable
Decimal                    hashable
Fraction                   hashable
bool                       hashable
NoneType                   hashable
str                        hashable
bytes                      hashable
bytearray                  unhashable (TypeError)
memoryview (read-only)     hashable
memoryview (writable)      unhashable (ValueError)
list                       unhashable (TypeError)
tuple of ints              hashable
tuple holding a list       unhashable (TypeError)
range                      hashable
dict                       unhashable (TypeError)
set                        unhashable (TypeError)
frozenset                  hashable
frozendict of ints         hashable
frozendict holding a list  unhashable (TypeError)


### 1.3.1 Integers (`int`)

Python's `int` has **arbitrary precision**: it grows to hold any value, so it never
overflows. There is no separate BigInt type and no `n` suffix. JavaScript's `number`
silently loses integer precision above `2**53`; Python does not.

In [43]:
js_max_safe_integer = 2**53 - 1
print(f"{js_max_safe_integer + 2=}  (exact; JS number would print ...992)")
print(f"{2**100=}")
print(f"{math.factorial(30)=}")
print(f"{float(2**53 + 1) == 2**53=}  (float, like JS number, cannot hold it)")

js_max_safe_integer + 2=9007199254740993  (exact; JS number would print ...992)
2**100=1267650600228229401496703205376
math.factorial(30)=265252859812191058636308480000000
float(2**53 + 1) == 2**53=True  (float, like JS number, cannot hold it)


Internally CPython stores an `int` as an array of **30-bit digits** (`sys.int_info`), so
its memory grows by 4 bytes for every 30 bits of magnitude. The growth table:

In [44]:
print(f"{sys.int_info=}")
print(f"{'value':>14} {'bits':>5} {'bytes':>6}")
for label, number in [
    ("0", 0),
    ("1", 1),
    ("2**30 - 1", 2**30 - 1),
    ("2**30", 2**30),
    ("2**60", 2**60),
    ("2**90", 2**90),
    ("10**100", 10**100),
    ("10**1000", 10**1000),
]:
    print(f"{label:>14} {number.bit_length():>5} {sys.getsizeof(number):>6}")

sys.int_info=sys.int_info(bits_per_digit=30, sizeof_digit=4, default_max_str_digits=4300, str_digits_check_threshold=640)
         value  bits  bytes
             0     0     28
             1     1     28
     2**30 - 1    30     28
         2**30    31     32
         2**60    61     36
         2**90    91     40
       10**100   333     72
      10**1000  3322    468


**Literals.** Decimal, binary (`0b`), octal (`0o`, never a bare leading zero) and
hexadecimal (`0x`). Underscores group digits for readability and are ignored.

In [45]:
population_of_india = 1_428_627_663
file_mode = 0o755
flag_byte = 0b1010_0001
brand_colour = 0xFF9933
print(f"{population_of_india=}, {file_mode=}, {flag_byte=}, {brand_colour=}")
try:
    eval("0755")
except SyntaxError as error:
    print(f"{type(error).__name__}: {error.msg}")

population_of_india=1428627663, file_mode=493, flag_byte=161, brand_colour=16750899
SyntaxError: leading zeros in decimal integer literals are not permitted; use an 0o prefix for octal integers


**Operators.**

| Operator | Meaning | `17 op 5` |
| --- | --- | --- |
| `+ - *` | add, subtract, multiply | `22`, `12`, `85` |
| `/` | true division, **always returns `float`** | `3.4` |
| `//` | floor division: rounds **down** (towards −∞) | `3` |
| `%` | modulo: the result takes the **divisor's** sign | `2` |
| `**` | power | `1419857` |

Floor means *down*, not *towards zero*, so `-7 // 2 == -4`. And `a == (a // b) * b + a % b`
always holds, which forces `-7 % 2 == 1`. JavaScript's `%` is a *remainder* that keeps the
dividend's sign (`-7 % 2 === -1`).

In [46]:
print(f"{17 + 5=}, {17 - 5=}, {17 * 5=}, {17 ** 5=}")
print(f"{17 / 5=}, {10 / 2=}  (always float)")
print(f"{17 // 5=}, {17 % 5=}")
print(f"{-7 // 2=}, {-7 % 2=}  (JS: Math.trunc gives -3, % gives -1)")
print(f"{7 // -2=}, {7 % -2=}")
dividend, divisor = -7, 2
print(f"{(dividend // divisor) * divisor + dividend % divisor == dividend=}")
print(f"{math.fmod(-7, 2)=}  (JS-style remainder, when you need it)")

17 + 5=22, 17 - 5=12, 17 * 5=85, 17 ** 5=1419857
17 / 5=3.4, 10 / 2=5.0  (always float)
17 // 5=3, 17 % 5=2
-7 // 2=-4, -7 % 2=1  (JS: Math.trunc gives -3, % gives -1)
7 // -2=-4, 7 % -2=-1
(dividend // divisor) * divisor + dividend % divisor == dividend=True
math.fmod(-7, 2)=-1.0  (JS-style remainder, when you need it)


**Functions and methods.** Each gets its own demo below.

| Name | What it does |
| --- | --- |
| `abs(x)` | absolute value |
| `divmod(a, b)` | `(a // b, a % b)` in one call |
| `pow(base, exp, mod)` | modular power, fast even for huge exponents |
| `round(x, ndigits)` | rounds to a multiple of `10**-ndigits` (round half to even) |
| `int.bit_length()` | bits needed, ignoring the sign |
| `int.bit_count()` | number of 1 bits (*popcount*) |
| `int.to_bytes` / `int.from_bytes` | convert to and from raw bytes |
| `bin` / `oct` / `hex` | text in base 2 / 8 / 16 |
| `int(text, base)` | parse text in any base 2–36 |
| `int.is_integer()` | always `True`; exists so code can treat `int` and `float` alike |

`abs` works on every numeric type.

In [47]:
account_balance_change = -2_500
print(f"{abs(account_balance_change)=}, {abs(-3.5)=}, {abs(Decimal('-0.10'))=}")

abs(account_balance_change)=2500, abs(-3.5)=3.5, abs(Decimal('-0.10'))=Decimal('0.10')


`divmod` returns quotient and remainder together, e.g. to split minutes into hours.

In [48]:
hours, minutes = divmod(135, 60)
print(f"{hours=}, {minutes=}")
print(f"{divmod(-7, 2)=}")

hours=2, minutes=15
divmod(-7, 2)=(-4, 1)


Three-argument `pow(base, exp, mod)` computes `base ** exp % mod` without building the
huge intermediate number. It is the core of RSA. With `exp=-1` it returns the modular
inverse (3.8+).

In [49]:
print(f"{pow(2, 10)=}")
print(f"{pow(7, 10**18, 13)=}  (instant; 7 ** 10**18 would never finish)")
print(f"{pow(3, -1, 11)=}  (3 * 4 % 11 == 1)")

pow(2, 10)=1024
pow(7, 10**18, 13)=9  (instant; 7 ** 10**18 would never finish)
pow(3, -1, 11)=4  (3 * 4 % 11 == 1)


`round(int, ndigits)` with a **negative** `ndigits` rounds to tens, hundreds, …. Ties
go to the even neighbour, so `1250` becomes `1200`.

In [50]:
print(f"{round(1_234, -2)=}, {round(1_250, -2)=}, {round(1_350, -2)=}")
print(f"{round(7)=}, {type(round(7.6)).__name__=}  (round() with no ndigits returns int)")

round(1_234, -2)=1200, round(1_250, -2)=1200, round(1_350, -2)=1400
round(7)=7, type(round(7.6)).__name__='int'  (round() with no ndigits returns int)


`bit_length()` is the number of bits needed for the magnitude; `bit_count()` counts the
ones (*new in 3.10*).

In [51]:
sample_number = 0b1011_0000
print(f"{sample_number=}, {sample_number.bit_length()=}, {(-sample_number).bit_length()=}")
print(f"{sample_number.bit_count()=}, {(255).bit_count()=}")

sample_number=176, sample_number.bit_length()=8, (-sample_number).bit_length()=8
sample_number.bit_count()=3, (255).bit_count()=8


`to_bytes(length, byteorder, signed=...)` and `int.from_bytes` convert between an `int`
and its raw bytes, e.g. for binary file formats and network protocols. Both default to
big-endian (3.11+).

In [52]:
port_number = 8080
port_bytes = port_number.to_bytes(2, "big")
print(f"{port_bytes=}, {port_bytes.hex()=}")
print(f"{int.from_bytes(port_bytes, 'big')=}")
print(f"{int.from_bytes(port_bytes, 'little')=}  (byte order matters)")
print(f"{(-1).to_bytes(2, 'big', signed=True)=}")
try:
    (-1).to_bytes(2, "big")
except OverflowError as error:
    print(f"{type(error).__name__}: {error}")

port_bytes=b'\x1f\x90', port_bytes.hex()='1f90'
int.from_bytes(port_bytes, 'big')=8080
int.from_bytes(port_bytes, 'little')=36895  (byte order matters)
(-1).to_bytes(2, 'big', signed=True)=b'\xff\xff'
OverflowError: can't convert negative int to unsigned


`bin`, `oct` and `hex` return prefixed strings. `format()` with `b`, `o`, `x` does the
same without the prefix, and can pad.

In [53]:
print(f"{bin(10)=}, {oct(493)=}, {hex(16_750_899)=}")
print(f"{format(10, '08b')=}, {format(255, 'X')=}")

bin(10)='0b1010', oct(493)='0o755', hex(16_750_899)='0xff9933'
format(10, '08b')='00001010', format(255, 'X')='FF'


`int(text, base)` parses text in bases 2 to 36. Base `0` reads the prefix from the text.
Surrounding whitespace and underscores between digits are allowed.

In [54]:
print(f"{int('ff', 16)=}, {int('1010', 2)=}, {int('z', 36)=}")
print(f"{int('0x1F', 0)=}, {int('0b101', 0)=}")
print(f"{int('  1_000  ')=}")

int('ff', 16)=255, int('1010', 2)=10, int('z', 36)=35
int('0x1F', 0)=31, int('0b101', 0)=5
int('  1_000  ')=1000


`int.is_integer()` (*new in 3.12*) always returns `True`. It exists so code that receives
"a number" can ask the question without checking the type first.

In [55]:
for quantity in (3, 3.0, 3.5):
    print(f"{quantity!r:>4}: {quantity.is_integer()=}")

   3: quantity.is_integer()=True
 3.0: quantity.is_integer()=True
 3.5: quantity.is_integer()=False


**Bitwise operators** work on the binary digits. Their classic use is a set of on/off
flags packed into one integer, such as Unix file permissions. (Phase 2 shows
`enum.Flag`, the type-safe version.)

| Operator | Meaning | Use with flags |
| --- | --- | --- |
| `a \| b` | OR | grant a permission |
| `a & b` | AND | test a permission |
| `a & ~b` | AND NOT | revoke a permission |
| `a ^ b` | XOR | toggle a permission |
| `a << n`, `a >> n` | shift left / right | build masks, multiply / divide by `2**n` |

In [56]:
READ, WRITE, EXECUTE = 0b100, 0b010, 0b001

permissions = READ | WRITE
print(f"{bin(permissions)=}, can write: {bool(permissions & WRITE)}")
permissions &= ~WRITE
print(f"{bin(permissions)=}, can write: {bool(permissions & WRITE)}")
permissions ^= EXECUTE
print(f"{bin(permissions)=}, can execute: {bool(permissions & EXECUTE)}")
print(f"{~5=}  (~x == -x - 1: Python ints behave as infinite two's complement)")
print(f"{1 << 10=}, {1024 >> 3=}")
owner_group_other = (0o7 << 6) | (0o5 << 3) | 0o5
print(f"{oct(owner_group_other)=}")

bin(permissions)='0b110', can write: True
bin(permissions)='0b100', can write: False
bin(permissions)='0b101', can execute: True
~5=-6  (~x == -x - 1: Python ints behave as infinite two's complement)
1 << 10=1024, 1024 >> 3=128
oct(owner_group_other)='0o755'


**The 4300-digit limit (3.11+, also back-ported to security releases).** Converting a
huge `int` to or from **decimal text** takes time quadratic in the number of digits. A
web service that ran `int(request_body)` could be stalled by a megabyte of digits
(CVE-2020-10735). Python therefore refuses decimal conversions above 4300 digits by
default. Binary bases (`hex`, `bin`) are linear and are not limited. Raise the limit
only for trusted data, and put it back afterwards.

In [57]:
huge_number = 10**5000
try:
    str(huge_number)
except ValueError as error:
    print(f"{type(error).__name__}: {str(error)[:70]}...")
print(f"{len(hex(huge_number))=}  (hex is not limited)")

default_limit = sys.get_int_max_str_digits()
sys.set_int_max_str_digits(10_000)
try:
    print(f"{len(str(huge_number))=}")
finally:
    sys.set_int_max_str_digits(default_limit)
print(f"{sys.get_int_max_str_digits()=}")

ValueError: Exceeds the limit (4300 digits) for integer string conversion; use sys...
len(hex(huge_number))=4155  (hex is not limited)
len(str(huge_number))=5001
sys.get_int_max_str_digits()=4300


### 1.3.2 Floating point (`float`)

A Python `float` is an **IEEE 754 double** (64 bits), exactly the same as a JavaScript
`number`. Most decimal fractions such as `0.1` have no exact binary form, so they are
stored as the nearest double, and the errors show up in arithmetic.
`Decimal(some_float)` reveals the value that is really stored. Compare floats with
`math.isclose`, never with `==`.

In [58]:
print(f"{0.1 + 0.2=}")
print(f"{0.1 + 0.2 == 0.3=}")
print(f"{Decimal(0.1)=}")
print(f"{math.isclose(0.1 + 0.2, 0.3)=}")
print(f"{math.isclose(1e-12, 0.0)=}  (relative tolerance fails near 0)")
print(f"{math.isclose(1e-12, 0.0, abs_tol=1e-9)=}")

0.1 + 0.2=0.30000000000000004
0.1 + 0.2 == 0.3=False
Decimal(0.1)=Decimal('0.1000000000000000055511151231257827021181583404541015625')
math.isclose(0.1 + 0.2, 0.3)=True
math.isclose(1e-12, 0.0)=False  (relative tolerance fails near 0)
math.isclose(1e-12, 0.0, abs_tol=1e-9)=True


**`round()` surprises.** Two effects combine:

1. **Banker's rounding:** an exact tie goes to the **even** neighbour, so `round(2.5) == 2`.
   This avoids an upward bias when many values are rounded. JS `Math.round(2.5)` is `3`.
2. **Representation error:** `2.675` is really stored as `2.67499999…`, so it is not a
   tie at all and rounds *down*.

In [59]:
print(f"{round(0.5)=}, {round(1.5)=}, {round(2.5)=}, {round(-2.5)=}")
print(f"{round(2.675, 2)=}")
print(f"{Decimal(2.675)=}")

round(0.5)=0, round(1.5)=2, round(2.5)=2, round(-2.5)=-2


round(2.675, 2)=2.67
Decimal(2.675)=Decimal('2.67499999999999982236431605997495353221893310546875')


**Special values.** `inf`, `-inf` and `nan` ("not a number") come from `float("inf")` /
`math.inf` / `math.nan` or from overflowing operations. `nan` is not equal to anything,
not even itself, so test it with `math.isnan`. Unlike JS, `1 / 0` raises instead of
returning `Infinity`.

In [60]:
positive_infinity = float("inf")
not_a_number = float("nan")
print(f"{positive_infinity > 10**308=}, {-math.inf < -1e308=}")
print(f"{positive_infinity - positive_infinity=}")
print(f"{not_a_number == not_a_number=}, {not_a_number != not_a_number=}")
print(f"{math.isnan(not_a_number)=}, {math.isinf(-math.inf)=}, {math.isfinite(1e308)=}")
print(f"{1e308 * 10=}")
try:
    ratio = 1 / 0
except ZeroDivisionError as error:
    print(f"{type(error).__name__}: {error}")

positive_infinity > 10**308=True, -math.inf < -1e308=True
positive_infinity - positive_infinity=nan
not_a_number == not_a_number=False, not_a_number != not_a_number=True
math.isnan(not_a_number)=True, math.isinf(-math.inf)=True, math.isfinite(1e308)=True
1e308 * 10=inf
ZeroDivisionError: division by zero


**`float` methods.** `as_integer_ratio()` returns the exact fraction the double stores.
For `0.1` that is not `1/10`.

In [61]:
print(f"{(0.75).as_integer_ratio()=}")
print(f"{(0.1).as_integer_ratio()=}")

(0.75).as_integer_ratio()=(3, 4)
(0.1).as_integer_ratio()=(3602879701896397, 36028797018963968)


`is_integer()` tells whether the float has no fractional part.

In [62]:
print(f"{(4.0).is_integer()=}, {(4.5).is_integer()=}, {(1e20).is_integer()=}")

(4.0).is_integer()=True, (4.5).is_integer()=False, (1e20).is_integer()=True


`hex()` and `float.fromhex()` write and read the exact bits in hexadecimal, with no
decimal rounding. Use them to store or send a float losslessly.

In [63]:
exact_text = (0.1).hex()
print(f"{exact_text=}")
print(f"{float.fromhex(exact_text) == 0.1=}")

exact_text='0x1.999999999999ap-4'
float.fromhex(exact_text) == 0.1=True


`sys.float_info` describes the platform's doubles. `epsilon` is the gap between `1.0`
and the next float, and `dig` is how many decimal digits are always kept.

In [64]:
print(f"{sys.float_info.max=}")
print(f"{sys.float_info.min=}")
print(f"{sys.float_info.epsilon=}")
print(f"{sys.float_info.dig=}, {sys.float_info.mant_dig=}")
print(f"{1.0 + sys.float_info.epsilon > 1.0=}, {1.0 + sys.float_info.epsilon / 2 > 1.0=}")

sys.float_info.max=1.7976931348623157e+308
sys.float_info.min=2.2250738585072014e-308
sys.float_info.epsilon=2.220446049250313e-16
sys.float_info.dig=15, sys.float_info.mant_dig=53
1.0 + sys.float_info.epsilon > 1.0=True, 1.0 + sys.float_info.epsilon / 2 > 1.0=False


**Useful `math` functions.** `floor` rounds down, `ceil` rounds up, `trunc` drops the
fraction (towards zero, like `int()`). They differ only for negative numbers, and all
three return `int`.

In [65]:
for value in (3.7, -3.7):
    rounded_values = (math.floor(value), math.ceil(value), math.trunc(value), int(value))
    print(f"{value:>5}: floor, ceil, trunc, int = {rounded_values}")

  3.7: floor, ceil, trunc, int = (3, 4, 3, 3)
 -3.7: floor, ceil, trunc, int = (-4, -3, -3, -3)


`math.sqrt` returns a float and rejects negatives (use `cmath`, 1.3.5). `math.isqrt`
gives the exact integer square root of any `int`.

In [66]:
print(f"{math.sqrt(2)=}, {math.isqrt(10**20 + 1)=}")
try:
    math.sqrt(-1)
except ValueError as error:
    print(f"{type(error).__name__}: {error}")

math.sqrt(2)=1.4142135623730951, math.isqrt(10**20 + 1)=10000000000
ValueError: expected a nonnegative input, got -1.0


`math.fsum` adds floats while tracking every lost low-order bit, so the result is
correctly rounded. A plain `+=` loop (what JS `reduce` does) accumulates error.
*Changed in 3.12:* `sum()` of floats uses compensated (Neumaier) summation, which fixes
the common cases, but `fsum` is still the exact one when values of wildly different
sizes cancel.

In [67]:
ten_dimes = [0.1] * 10
loop_total = 0.0
for dime in ten_dimes:
    loop_total += dime
print(f"{loop_total=}  (naive left-to-right)")
print(f"{sum(ten_dimes)=}, {math.fsum(ten_dimes)=}")

mixed_magnitudes = [1e100, 1.0, -1e100, 1e-100, 1e50, -1.0, -1e50]
print(f"{sum(mixed_magnitudes)=}, {math.fsum(mixed_magnitudes)=}")

loop_total=0.9999999999999999  (naive left-to-right)
sum(ten_dimes)=1.0, math.fsum(ten_dimes)=1.0
sum(mixed_magnitudes)=0.0, math.fsum(mixed_magnitudes)=1e-100


`math.prod` multiplies an iterable, the way `sum` adds one (3.8+).

In [68]:
print(f"{math.prod([2, 3, 7])=}")
print(f"{math.prod([1.05] * 3)=}  (three years of 5% growth)")

math.prod([2, 3, 7])=42
math.prod([1.05] * 3)=1.1576250000000001  (three years of 5% growth)


`math.pi`, `math.e` (and `math.tau`, `math.inf`, `math.nan`) are float constants.

In [69]:
circle_radius_in_metres = 2.5
print(f"{math.pi=}, {math.e=}, {math.tau == 2 * math.pi=}")
print(f"{math.pi * circle_radius_in_metres**2=}")

math.pi=3.141592653589793, math.e=2.718281828459045, math.tau == 2 * math.pi=True
math.pi * circle_radius_in_metres**2=19.634954084936208


### 1.3.3 Exact decimals (`decimal.Decimal`)

`Decimal` stores numbers in base 10, so `0.1` is exactly one tenth. That is what money
needs. **Always construct it from a string (or an `int`), never from a float**: a float
argument copies the float's representation error.

In [70]:
print(f"{Decimal('0.1')=}")
print(f"{Decimal(0.1)=}")
print(f"{Decimal('0.1') + Decimal('0.2') == Decimal('0.3')=}")
print(f"{Decimal(25)=}, {Decimal('1.50')=}  (trailing zeros are kept)")

Decimal('0.1')=Decimal('0.1')
Decimal(0.1)=Decimal('0.1000000000000000055511151231257827021181583404541015625')
Decimal('0.1') + Decimal('0.2') == Decimal('0.3')=True
Decimal(25)=Decimal('25'), Decimal('1.50')=Decimal('1.50')  (trailing zeros are kept)


**Arithmetic.** `Decimal` supports the usual operators and mixes with `int`, but not
with `float` (that would bring the inexactness back). Results are rounded to the
**context precision**: 28 significant digits by default. `localcontext()` changes the
precision (or the rounding) only inside a `with` block.

In [71]:
unit_price = Decimal("19.99")
print(f"{unit_price * 3=}, {unit_price - 5=}, {unit_price / 4=}")
print(f"{Decimal(1) / Decimal(7)=}")
with localcontext() as local_context:
    local_context.prec = 6
    print(f"inside localcontext: {Decimal(1) / Decimal(7)=}")
print(f"after the block: {getcontext().prec=}")
try:
    unit_price + 0.5
except TypeError as error:
    print(f"{type(error).__name__}: {error}")

unit_price * 3=Decimal('59.97'), unit_price - 5=Decimal('14.99'), unit_price / 4=Decimal('4.9975')
Decimal(1) / Decimal(7)=Decimal('0.1428571428571428571428571429')
inside localcontext: Decimal(1) / Decimal(7)=Decimal('0.142857')
after the block: getcontext().prec=28
TypeError: unsupported operand type(s) for +: 'decimal.Decimal' and 'float'


**`quantize`** rounds to a fixed exponent, e.g. `Decimal("0.01")` for paise. The
rounding mode is explicit:

| Mode | Ties go… | Typical use |
| --- | --- | --- |
| `ROUND_HALF_UP` | away from zero (`2.675 → 2.68`) | invoices and tax, as taught in school |
| `ROUND_HALF_EVEN` | to the even digit (`2.665 → 2.66`) | statistics, the `Decimal` default |

In [72]:
print(f"{'value':>7} {'HALF_UP':>8} {'HALF_EVEN':>10}")
for amount_text in ["2.675", "2.665", "0.125", "-1.005"]:
    amount = Decimal(amount_text)
    half_up = amount.quantize(PAISE, rounding=ROUND_HALF_UP)
    half_even = amount.quantize(PAISE, rounding=ROUND_HALF_EVEN)
    print(f"{amount_text:>7} {half_up!s:>8} {half_even!s:>10}")

  value  HALF_UP  HALF_EVEN
  2.675     2.68       2.68
  2.665     2.67       2.66
  0.125     0.13       0.12
 -1.005    -1.01      -1.00


**Money: an 18% GST demo.** One thousand orders of ₹99.99 each, with GST computed and
rounded per order. Float accumulates representation error across the additions;
`Decimal` is exact to the paisa.

In [73]:
order_count = 1_000

float_grand_total = 0.0
for _ in range(order_count):
    float_grand_total += 99.99 + round(99.99 * 0.18, 2)

decimal_grand_total = Decimal("0.00")
for _ in range(order_count):
    gst = (Decimal("99.99") * Decimal("0.18")).quantize(PAISE, rounding=ROUND_HALF_UP)
    decimal_grand_total += Decimal("99.99") + gst

print(f"{float_grand_total=}")
print(f"{decimal_grand_total=}")

float_grand_total=117990.00000000196
decimal_grand_total=Decimal('117990.00')


**Performance trade-off.** `float` arithmetic runs in hardware; `Decimal` is software
(C-accelerated `_decimal`, but still slower). The usual pattern is `Decimal` (or
integer paise) for money and `float` for measurements and science. Timings vary by
machine; the ratio is what matters.

In [74]:
float_operands = {"a": 99.99, "b": 0.18, "c": 1.5}
decimal_operands = {"a": Decimal("99.99"), "b": Decimal("0.18"), "c": Decimal("1.5")}
float_seconds = timeit.timeit("a * b + c", globals=float_operands, number=200_000)
decimal_seconds = timeit.timeit("a * b + c", globals=decimal_operands, number=200_000)
slowdown = decimal_seconds / float_seconds
print(f"float:   {float_seconds * 1000:.1f} ms")
print(f"Decimal: {decimal_seconds * 1000:.1f} ms  (about {slowdown:.0f}x slower)")

float:   9.9 ms


Decimal: 42.7 ms  (about 4x slower)


### 1.3.4 Rational numbers (`fractions.Fraction`)

A `Fraction` is an exact numerator/denominator pair, always kept in lowest terms. One
third really is one third, so `Fraction(1, 3) * 3 == 1`.

In [75]:
one_third = Fraction(1, 3)
print(f"{one_third * 3 == 1=}, {one_third * 3=}")
print(f"{Fraction(6, 8)=}  (reduced automatically)")
print(f"{one_third + Fraction(1, 6)=}")
print(f"{1 / 3 * 3 == 1=}  (float happens to round back here, but not in general)")

one_third * 3 == 1=True, one_third * 3=Fraction(1, 1)
Fraction(6, 8)=Fraction(3, 4)  (reduced automatically)
one_third + Fraction(1, 6)=Fraction(1, 2)
1 / 3 * 3 == 1=True  (float happens to round back here, but not in general)


Build a `Fraction` from a **string** for the exact value you typed. Building it from a
float shows the float's exact binary value. `limit_denominator(max_denominator)` finds
the closest fraction with a small denominator, which recovers the intended value.

In [76]:
print(f"{Fraction('3/7')=}, {Fraction('0.125')=}, {Fraction('1.5e-2')=}")
print(f"{Fraction(0.1)=}")
print(f"{Fraction(0.1).limit_denominator(1000)=}")
print(f"{Fraction(math.pi).limit_denominator(1000)=}")

Fraction('3/7')=Fraction(3, 7), Fraction('0.125')=Fraction(1, 8), Fraction('1.5e-2')=Fraction(3, 200)
Fraction(0.1)=Fraction(3602879701896397, 36028797018963968)
Fraction(0.1).limit_denominator(1000)=Fraction(1, 10)
Fraction(math.pi).limit_denominator(1000)=Fraction(355, 113)


### 1.3.5 Complex numbers (`complex`)

Python has complex numbers built in: write the imaginary part with a `j` suffix. They
are used in signal processing and electrical engineering; you will rarely need them for
web work.

In [77]:
impedance = 3 + 4j
print(f"{impedance=}, {type(impedance).__name__=}")
print(f"{impedance.real=}, {impedance.imag=}")
print(f"{abs(impedance)=}  (magnitude)")
print(f"{impedance.conjugate()=}, {impedance * impedance.conjugate()=}")
print(f"{complex(3, 4) == impedance=}")

impedance=(3+4j), type(impedance).__name__='complex'
impedance.real=3.0, impedance.imag=4.0
abs(impedance)=5.0  (magnitude)
impedance.conjugate()=(3-4j), impedance * impedance.conjugate()=(25+0j)
complex(3, 4) == impedance=True


`cmath` is the complex-number version of `math`: its functions accept negative and
complex inputs.

In [78]:
print(f"{cmath.sqrt(-1)=}")
print(f"{cmath.phase(1j)=}  (pi / 2)")
print(f"{cmath.polar(impedance)=}")
print(f"{cmath.rect(5, 0.9272952180016122)=}")

cmath.sqrt(-1)=1j
cmath.phase(1j)=1.5707963267948966  (pi / 2)
cmath.polar(impedance)=(5.0, 0.9272952180016122)
cmath.rect(5, 0.9272952180016122)=(3.0000000000000004+3.9999999999999996j)


### 1.3.6 Booleans (`bool`)

`bool` is a **subclass of `int`**: `True` is `1` and `False` is `0` in arithmetic. A
useful consequence: `sum(flags)` counts the `True` values.

In [79]:
print(f"{issubclass(bool, int)=}")
print(f"{True + True=}, {True * 10=}, {True == 1=}")
order_is_paid = [True, False, True, True]
print(f"{sum(order_is_paid)=} paid orders")

issubclass(bool, int)=True
True + True=2, True * 10=10, True == 1=True
sum(order_is_paid)=3 paid orders


**The validation trap.** Because `True` *is* an `int`, `isinstance(True, int)` is
`True`, and a check meant to accept quantities also accepts booleans. Reject `bool`
explicitly.

In [80]:
def is_valid_quantity_naive(quantity: object) -> bool:
    """Accept positive ints (buggy: also accepts True)."""
    return isinstance(quantity, int) and quantity > 0


def is_valid_quantity(quantity: object) -> bool:
    """Accept positive ints, rejecting bool."""
    return isinstance(quantity, int) and not isinstance(quantity, bool) and quantity > 0


for candidate in (3, True, 0, 2.0):
    naive_verdict = is_valid_quantity_naive(candidate)
    print(f"{candidate!r:>5}: naive={naive_verdict}, fixed={is_valid_quantity(candidate)}")

    3: naive=True, fixed=True
 True: naive=True, fixed=False
    0: naive=False, fixed=False
  2.0: naive=False, fixed=False


`bool(x)` converts any value using the truthiness rules (1.3.8). Watch out for strings:
any non-empty string is `True`, including `"False"` and `"0"`.

In [81]:
print(f"{bool(0)=}, {bool(42)=}, {bool('')=}, {bool('False')=}, {bool('0')=}")
form_value = "false"
print(f"{form_value.strip().lower() in {'true', '1', 'yes'}=}  (parse text explicitly)")

bool(0)=False, bool(42)=True, bool('')=False, bool('False')=True, bool('0')=True
form_value.strip().lower() in {'true', '1', 'yes'}=False  (parse text explicitly)


### 1.3.7 The null value (`None`)

`None` is the only instance of `NoneType`, so every `None` is the same object; check it
with `is None`. A function without a `return` (or with a bare `return`) returns `None`.

In [82]:
def log_message(message: str) -> None:
    """Print a message; no return statement."""
    print(f"log: {message}")


result = log_message("order created")
print(f"{result is None=}")
print(f"{type(None).__name__=}, {type(None)() is None=}")

log: order created
result is None=True
type(None).__name__='NoneType', type(None)() is None=True


> **JS bridge:** Python has **no `undefined`**. An unassigned name raises `NameError`,
> a missing attribute raises `AttributeError`, and a missing dict key raises `KeyError`.
> `dict.get(key)` returns `None` (or a default) when you want JS-like behaviour.

In [83]:
customer = {"name": "Asha", "city": "Pune"}
try:
    customer["phone"]
except KeyError as error:
    print(f"{type(error).__name__}: {error}")
print(f"{customer.get('phone')=}, {customer.get('phone', 'not given')=}")

KeyError: 'phone'
customer.get('phone')=None, customer.get('phone', 'not given')='not given'


### 1.3.8 Truthiness

Every object is true or false in an `if`, `while`, `and`, `or` or `not`. These values are
**falsy**; everything else is truthy:

| Falsy value | Type |
| --- | --- |
| `None` | `NoneType` |
| `False` | `bool` |
| `0`, `0.0`, `0j` | `int`, `float`, `complex` |
| `Decimal(0)`, `Fraction(0)` | `Decimal`, `Fraction` |
| `""`, `b""` | `str`, `bytes` |
| `[]`, `()`, `{}` | `list`, `tuple`, `dict` |
| `set()`, `range(0)` | `set`, `range` |
| an object whose `__bool__` returns `False`, or whose `__len__` returns `0` | user classes |

The cell evaluates every row.

In [84]:
falsy_candidates = [None, False, 0, 0.0, 0j, Decimal(0), Fraction(0)]
falsy_candidates += ["", b"", [], (), {}, set(), range(0)]
for candidate in falsy_candidates:
    print(f"{candidate!r:<14} -> {bool(candidate)}")

None           -> False
False          -> False
0              -> False
0.0            -> False
0j             -> False
Decimal('0')   -> False
Fraction(0, 1) -> False
''             -> False
b''            -> False
[]             -> False
()             -> False
{}             -> False
set()          -> False
range(0, 0)    -> False


User classes decide with `__bool__`, or, if it is missing, with `__len__` (zero length
is false). With neither, every instance is truthy.

In [85]:
class Wallet:
    """A wallet that is truthy only when it holds money."""

    def __init__(self, balance_in_paise: int) -> None:
        self.balance_in_paise = balance_in_paise

    def __bool__(self) -> bool:
        return self.balance_in_paise > 0


class ShoppingBag:
    """A bag whose truthiness comes from its length."""

    def __init__(self, items: list[str]) -> None:
        self.items = items

    def __len__(self) -> int:
        return len(self.items)


class PlainObject:
    """No __bool__ and no __len__."""


print(f"{bool(Wallet(0))=}, {bool(Wallet(500))=}")
print(f"{bool(ShoppingBag([]))=}, {bool(ShoppingBag(['soap']))=}")
print(f"{bool(PlainObject())=}")

bool(Wallet(0))=False, bool(Wallet(500))=True
bool(ShoppingBag([]))=False, bool(ShoppingBag(['soap']))=True
bool(PlainObject())=True


**Differences from JavaScript**

| Value | JavaScript | Python |
| --- | --- | --- |
| `[]`, `{}` | truthy | **falsy** |
| `"0"`, `" "` | truthy | truthy |
| `NaN` | falsy | **truthy** (`nan` is a non-zero float) |
| `0n` / `0` | falsy | falsy |
| `document.all` | falsy (!) | — |

**When to prefer `is not None`.** `if value:` treats `0`, `""` and `[]` the same as a
missing value. When `0` is a legitimate input, test for `None` explicitly.

In [86]:
def price_after_discount_buggy(price: int, discount: int | None = None) -> int:
    """Apply a discount; treats a discount of 0 as 'not given' (bug)."""
    if not discount:
        discount = 10  # default discount
    return price - discount


def price_after_discount(price: int, discount: int | None = None) -> int:
    """Apply a discount; only None means 'not given'."""
    if discount is None:
        discount = 10
    return price - discount


print(f"{bool(float('nan'))=}")
print(f"{price_after_discount_buggy(100, 0)=}  (wanted 100)")
print(f"{price_after_discount(100, 0)=}")

bool(float('nan'))=True
price_after_discount_buggy(100, 0)=90  (wanted 100)
price_after_discount(100, 0)=100


### 1.3.9 Type conversion

Conversions are always **explicit**: call the type. They raise `ValueError` when the
text is not a valid literal for that type. Note that `int("3.5")` fails: convert to
`float` first if decimals are allowed. `int(3.99)` truncates.

In [87]:
print(f"{int('42')=}, {int(' -7 ')=}, {int(3.99)=}, {int(True)=}")
print(f"{float('3.5')=}, {float('1e3')=}, {float('nan')=}, {float(7)=}")
print(f"{str(42)=}, {str(3.5)=}, {str(None)=}")
print(f"{bool(1)=}, {bool('')=}")
print(f"{complex('3+4j')=}, {complex(2)=}")
print(f"{int(float('3.5'))=}")

int('42')=42, int(' -7 ')=-7, int(3.99)=3, int(True)=1
float('3.5')=3.5, float('1e3')=1000.0, float('nan')=nan, float(7)=7.0
str(42)='42', str(3.5)='3.5', str(None)='None'
bool(1)=True, bool('')=False
complex('3+4j')=(3+4j), complex(2)=(2+0j)
int(float('3.5'))=3


Each failing conversion raises `ValueError` with a message that quotes the bad text.

In [88]:
for converter, bad_text in [(int, "3.5"), (int, "12abc"), (float, "abc"), (complex, "3 + 4j")]:
    try:
        converter(bad_text)
    except ValueError as error:
        print(f"{converter.__name__}({bad_text!r}) -> {type(error).__name__}: {error}")

int('3.5') -> ValueError: invalid literal for int() with base 10: '3.5'
int('12abc') -> ValueError: invalid literal for int() with base 10: '12abc'
float('abc') -> ValueError: could not convert string to float: 'abc'
complex('3 + 4j') -> ValueError: complex() arg is a malformed string


**No implicit coercion.** JavaScript converts silently (`1 + "1" === "11"`,
`1 == "1"` is `true`). Python raises `TypeError` for mixed `+` and returns `False` for
`==` between unrelated types. Numeric types do mix with each other (`1 + 2.5`, `1 == 1.0`).

In [89]:
try:
    mixed_total = 1 + "1"
except TypeError as error:
    print(f"{type(error).__name__}: {error}")
print(f"{1 == '1'=}")
print(f"{1 + int('1')=}, {str(1) + '1'=}")
print(f"{1 + 2.5=}, {1 == 1.0 == Decimal(1) == Fraction(1)=}")

TypeError: unsupported operand type(s) for +: 'int' and 'str'
1 == '1'=False
1 + int('1')=2, str(1) + '1'='11'
1 + 2.5=3.5, 1 == 1.0 == Decimal(1) == Fraction(1)=True


#### Under the hood: object sizes and the int digit formula

Every number is a full object with a header, so even `True` costs bytes. For `int`,
the size is a fixed header plus 4 bytes per 30-bit digit; the cell checks the formula
against `sys.getsizeof` for growing values. `show_size` (shared setup) prints a few
other numeric types for comparison.

In [90]:
int_header_bytes = sys.getsizeof(1) - sys.int_info.sizeof_digit
for exponent in (10, 100, 1_000):
    number = 2**exponent
    digit_count = -(-number.bit_length() // sys.int_info.bits_per_digit)  # ceiling division
    predicted_bytes = int_header_bytes + digit_count * sys.int_info.sizeof_digit
    actual_bytes = sys.getsizeof(number)
    print(f"2**{exponent:<5} digits={digit_count:<3} {predicted_bytes=:<4} {actual_bytes=}")

show_size("float", 1.5)
show_size("complex", 1 + 2j)
show_size("Decimal", Decimal("1.5"))
show_size("Fraction", Fraction(1, 2))
show_size("True", True)

2**10    digits=1   predicted_bytes=28   actual_bytes=28
2**100   digits=4   predicted_bytes=40   actual_bytes=40
2**1000  digits=34  predicted_bytes=160  actual_bytes=160
float: 24 bytes, type=float, id=139669155623344
complex: 32 bytes, type=complex, id=139669155147088
Decimal: 120 bytes, type=Decimal, id=139669135360448
Fraction: 48 bytes, type=Fraction, id=139669135190928
True: 28 bytes, type=bool, id=22110296


### Lab 1.3

Three exercises: an exact bill splitter, an invoice calculator (with a float version to
show drift), and predict-the-output questions. For each coding exercise, replace the
`raise NotImplementedError` in the stub, then run the check cell. Until then the checks
print FAIL with `NotImplementedError`; that is expected.

#### Exercise 1 — `split_bill`

Write `split_bill(total_amount: str, people_count: int) -> list[Decimal]`:

- Split the total (a rupee amount as text, e.g. `"100"` or `"1000.01"`) into
  `people_count` parts **exact to the paisa**: every part has two decimal places and
  the parts sum to exactly the total.
- Parts differ by at most one paisa; the leftover paise go to the **first** people.
  `split_bill("100", 3) == [Decimal("33.34"), Decimal("33.33"), Decimal("33.33")]`.
- Raise `ValueError` when `people_count` is less than 1.

*Hint:* work in whole paise with `divmod`.

In [91]:
def split_bill(total_amount: str, people_count: int) -> list[Decimal]:
    """Split a rupee amount into people_count parts that sum exactly to the total."""
    raise NotImplementedError("Lab 1.3: implement split_bill()")

The check cell defines `raised_error_name(thunk)`, which returns the name of the
exception a call raises (or `"no error"`), so error cases fit the
`(label, thunk, expected)` format of `run_checks`.

In [92]:
def raised_error_name(thunk: Callable[[], object]) -> str:
    """Call thunk and return the name of the exception it raised, or "no error"."""
    try:
        thunk()
    except Exception as error:
        return type(error).__name__
    return "no error"


run_checks(
    [
        (
            "100 between 3",
            lambda: split_bill("100", 3),
            [Decimal("33.34"), Decimal("33.33"), Decimal("33.33")],
        ),
        ("parts sum to the total", lambda: sum(split_bill("1000.01", 7)), Decimal("1000.01")),
        ("one part per person", lambda: len(split_bill("10", 4)), 4),
        (
            "parts differ by at most one paisa",
            lambda: max(split_bill("0.05", 3)) - min(split_bill("0.05", 3)) <= PAISE,
            True,
        ),
        (
            "every part has two decimal places",
            lambda: {part.as_tuple().exponent for part in split_bill("90", 4)},
            {-2},
        ),
        ("returns Decimal", lambda: type(split_bill("1", 1)[0]).__name__, "Decimal"),
        (
            "zero people raises",
            lambda: raised_error_name(lambda: split_bill("10", 0)),
            "ValueError",
        ),
    ]
)

FAIL  100 between 3: raised NotImplementedError: Lab 1.3: implement split_bill()
FAIL  parts sum to the total: raised NotImplementedError: Lab 1.3: implement split_bill()
FAIL  one part per person: raised NotImplementedError: Lab 1.3: implement split_bill()
FAIL  parts differ by at most one paisa: raised NotImplementedError: Lab 1.3: implement split_bill()
FAIL  every part has two decimal places: raised NotImplementedError: Lab 1.3: implement split_bill()
FAIL  returns Decimal: raised NotImplementedError: Lab 1.3: implement split_bill()
FAIL  zero people raises: expected 'ValueError', got 'NotImplementedError'
0/7 checks passed


#### Exercise 2 — invoice calculator

An invoice is a list of `(unit_price, quantity)` line items. For **each line**:
net amount = unit price × quantity, GST = net amount × rate, **rounded to paise per
line**. The invoice total is the sum of every line's net amount plus its GST.

1. `invoice_total_decimal(line_items: list[tuple[str, int]], gst_rate_percent: str = "18") -> Decimal`:
   prices and rate are strings, GST is rounded with `quantize(PAISE, rounding=ROUND_HALF_UP)`,
   and the total is returned with two decimal places.
2. `invoice_total_float(line_items: list[tuple[float, int]], gst_rate_percent: float = 18.0) -> float`:
   the same calculation with floats and `round(gst, 2)`. Do **not** round the final
   total, so the drift stays visible.

In [93]:
def invoice_total_decimal(
    line_items: list[tuple[str, int]], gst_rate_percent: str = "18"
) -> Decimal:
    """Return net + per-line GST (ROUND_HALF_UP to paise) for all line items."""
    raise NotImplementedError("Lab 1.3: implement invoice_total_decimal()")


def invoice_total_float(
    line_items: list[tuple[float, int]], gst_rate_percent: float = 18.0
) -> float:
    """Return the same total computed with float and round()."""
    raise NotImplementedError("Lab 1.3: implement invoice_total_float()")

The checks include a line of ₹2.75: its GST is exactly ₹0.495, which `ROUND_HALF_UP`
makes ₹0.50. The float `2.75 * 18 / 100` is stored just *below* 0.495, so `round()`
gives ₹0.49.

In [94]:
sample_line_items = [
    ("19.99", 3),
    ("0.10", 7),
    ("2.75", 1),
    ("12.25", 2),
    ("0.25", 1),
    ("5.25", 1),
]
float_line_items = [(float(price), quantity) for price, quantity in sample_line_items]

run_checks(
    [
        ("empty invoice", lambda: invoice_total_decimal([]), Decimal("0.00")),
        ("one line, 18% GST", lambda: invoice_total_decimal([("100", 2)]), Decimal("236.00")),
        ("GST rounds half up", lambda: invoice_total_decimal([("1.25", 1)]), Decimal("1.48")),
        ("custom rate 5%", lambda: invoice_total_decimal([("10.10", 1)], "5"), Decimal("10.61")),
        ("sample invoice", lambda: invoice_total_decimal(sample_line_items), Decimal("110.25")),
        ("float: one line, 18% GST", lambda: invoice_total_float([(100.0, 2)]), 236.0),
        ("float: round() loses a paisa on 2.75", lambda: invoice_total_float([(2.75, 1)]), 3.24),
        (
            "float: sample invoice drifts from the exact total",
            lambda: (
                invoice_total_float(float_line_items)
                != float(invoice_total_decimal(sample_line_items))
            ),
            True,
        ),
    ]
)

FAIL  empty invoice: raised NotImplementedError: Lab 1.3: implement invoice_total_decimal()
FAIL  one line, 18% GST: raised NotImplementedError: Lab 1.3: implement invoice_total_decimal()
FAIL  GST rounds half up: raised NotImplementedError: Lab 1.3: implement invoice_total_decimal()
FAIL  custom rate 5%: raised NotImplementedError: Lab 1.3: implement invoice_total_decimal()
FAIL  sample invoice: raised NotImplementedError: Lab 1.3: implement invoice_total_decimal()
FAIL  float: one line, 18% GST: raised NotImplementedError: Lab 1.3: implement invoice_total_float()
FAIL  float: round() loses a paisa on 2.75: raised NotImplementedError: Lab 1.3: implement invoice_total_float()
FAIL  float: sample invoice drifts from the exact total: raised NotImplementedError: Lab 1.3: implement invoice_total_float()
0/8 checks passed


Once both functions work, this cell shows the drift side by side. Until then it prints
a reminder.

In [95]:
try:
    exact_invoice_total = invoice_total_decimal(sample_line_items)
    float_invoice_total = invoice_total_float(float_line_items)
except NotImplementedError as error:
    print(f"Not yet: {error}")
else:
    print(f"{exact_invoice_total=}")
    print(f"{float_invoice_total=}")
    print(f"difference: {Decimal(float_invoice_total) - exact_invoice_total:.17f}")

Not yet: Lab 1.3: implement invoice_total_decimal()


#### Exercise 3 — predict the output

**Predict the output.**

```python
print(-7 // 2, 7 // -2, -7.0 // 2)
```

<details><summary>Answer</summary>

`-4 -4 -4.0`. `//` floors towards negative infinity: −3.5 becomes −4 in both sign
combinations. With a float operand the result is a float.

</details>

Run the next cell to check your prediction.

In [96]:
print(-7 // 2, 7 // -2, -7.0 // 2)

-4 -4 -4.0


**Predict the output.**

```python
print(-7 % 2, 7 % -2, -7 % -2)
```

<details><summary>Answer</summary>

`1 -1 -1`. The result of `%` takes the sign of the **divisor**, so that
`(a // b) * b + a % b == a`. For `-7 % 2`: `(-4) * 2 + 1 == -7`.

</details>

Run the next cell to check your prediction.

In [97]:
print(-7 % 2, 7 % -2, -7 % -2)

1 -1 -1


**Predict the output.**

```python
print(round(0.5), round(1.5), round(2.5), round(2.675, 2))
```

<details><summary>Answer</summary>

`0 2 2 2.67`. Exact ties go to the even neighbour (banker's rounding), and `2.675` is
stored as `2.67499999…`, which is below the tie.

</details>

Run the next cell to check your prediction.

In [98]:
print(round(0.5), round(1.5), round(2.5), round(2.675, 2))

0 2 2 2.67


**Predict the output.**

```python
print(bool([0]), bool("0"), bool(0.0), bool(None), bool(" "), bool({}))
```

<details><summary>Answer</summary>

`True True False False True False`. `[0]` is a non-empty list, and `"0"` and `" "` are
non-empty strings, so all three are truthy. `0.0`, `None` and the empty dict `{}` are
falsy.

</details>

Run the next cell to check your prediction.

In [99]:
print(bool([0]), bool("0"), bool(0.0), bool(None), bool(" "), bool({}))

True True False False True False


### Recap — Unit 1.3

| Concept | One-line takeaway | JS equivalent |
| --- | --- | --- |
| `int` | Arbitrary precision, 30-bit digits, never overflows | `BigInt` |
| `/`, `//`, `%` | `/` is always float; `//` floors; `%` takes the divisor's sign | `/`, `Math.floor`, remainder `%` |
| 4300-digit limit | Decimal-text conversion of huge ints is capped against DoS | — |
| `float` | IEEE 754 double; compare with `math.isclose`; `1/0` raises | `number` |
| `round()` | Round half to even, plus representation error | `Math.round` rounds half up |
| `Decimal` | Exact base-10; build from strings; `quantize` with explicit rounding for money | decimal.js, big.js |
| `Fraction` | Exact rationals; `limit_denominator` recovers simple ratios | — |
| `complex` | `3 + 4j`, `.real`, `.imag`, `abs`, `cmath` | — |
| `bool` | Subclass of `int`; reject it explicitly when validating ints | `boolean` |
| `None` | The only null; check with `is None`; no `undefined` | `null` / `undefined` |
| Truthiness | Empty containers and zeros are falsy; use `is not None` when 0 is valid | `[]` and `{}` are truthy |
| Conversion | Always explicit; bad text raises `ValueError`; mixed `+` raises `TypeError` | Implicit coercion |

**Further reading**

- [Numeric types — `int`, `float`, `complex`](https://docs.python.org/3/library/stdtypes.html#numeric-types-int-float-complex),
  [integer string conversion length limitation](https://docs.python.org/3/library/stdtypes.html#int-max-str-digits)
- [Floating-point arithmetic: issues and limitations](https://docs.python.org/3/tutorial/floatingpoint.html),
  [`math`](https://docs.python.org/3/library/math.html),
  [`cmath`](https://docs.python.org/3/library/cmath.html)
- [`decimal`](https://docs.python.org/3/library/decimal.html),
  [`fractions`](https://docs.python.org/3/library/fractions.html)
- [Truth value testing](https://docs.python.org/3/library/stdtypes.html#truth-value-testing),
  [`None`](https://docs.python.org/3/library/constants.html#None)
- [PEP 515 — Underscores in numeric literals](https://peps.python.org/pep-0515/),
  [PEP 285 — Adding a bool type](https://peps.python.org/pep-0285/),
  [PEP 327 — Decimal data type](https://peps.python.org/pep-0327/),
  [PEP 237 — Unifying long integers and integers](https://peps.python.org/pep-0237/)

## Unit 1.4 — Text and Binary Data Types

**Objectives**

- Index, slice, search, clean, split and format text with the complete `str` API.
- Explain what `len` counts, how CPython stores a string, and why `"10" < "9"`.
- Compare Unicode text correctly with normalization and `casefold`.
- Move between `str` and `bytes` on purpose, choosing an `errors` policy.
- Render untrusted values safely with t-strings (*new in 3.14*).

**JS bridge**

| JavaScript concept | Python equivalent | The difference that bites |
| --- | --- | --- |
| `"😀".length === 2` | `len("😀") == 1` | Python counts code points; JS counts UTF-16 code units |
| `` `Hi ${name}` `` | `f"Hi {name}"` | Same idea; f-strings add a format spec (`{price:,.2f}`) and `=` debugging |
| Tagged templates (`` html`…` ``) | t-strings `t"…"` (3.14) | They give you a `Template` object, not a string; your renderer decides how to escape |
| `"a-b-a".replace("a", "x")` replaces the first match | `str.replace` replaces **every** match | Pass `count=1` for the JS behaviour |
| `"10" < "9"` is `true` | `"10" < "9"` is `True` | Same: strings compare character by character, never numerically |
| `str.trim()` | `str.strip()` | `strip("xy")` removes a *set* of characters, not a prefix |
| `str[99]` is `undefined` | `IndexError` | Slices (`text[99:]`) never raise |
| `Buffer` / `Uint8Array` | `bytes` / `bytearray` | `bytes` is immutable; `str + bytes` raises `TypeError` |

The next cell is this unit's setup.

In [100]:
import base64
import csv
import hashlib
import html
import locale
import sqlite3
import string
import sys
import unicodedata
from collections.abc import Callable
from datetime import date
from decimal import Decimal
from string.templatelib import Interpolation, Template


def show_size(label: str, obj: object) -> None:
    """Print the shallow size in bytes, the type and the id of an object."""
    print(f"{label}: {sys.getsizeof(obj)} bytes, type={type(obj).__name__}, id={id(obj)}")


def raised_error_name(thunk: Callable[[], object]) -> str:
    """Call thunk and return the name of the exception it raised, or "no error"."""
    try:
        thunk()
    except Exception as error:
        return type(error).__name__
    return "no error"


print(f"{sys.version_info[:2]=}, {sys.getdefaultencoding()=}, {unicodedata.unidata_version=}")

sys.version_info[:2]=(3, 15), sys.getdefaultencoding()='utf-8', unicodedata.unidata_version='17.0.0'


### 1.4.1 Strings (`str`) fundamentals

A `str` is an **immutable sequence of Unicode code points**. Single and double quotes
mean the same thing; pick the one that avoids escaping. Triple quotes span lines. A raw
string `r"..."` keeps backslashes as typed (useful for regexes and Windows paths).
Common escapes: `\n` newline, `\t` tab, `\u20b9` a code point by its hex number (₹).

In [101]:
shop_name = 'Annapurna "Tiffin" Centre'
opening_note = "It's open"
menu_text = """Idli
Dosa
Vada"""
raw_windows_path = r"C:\new\table"
escaped_windows_path = "C:\new\table"
rupee_sign = "\u20b9"
print(shop_name, "|", opening_note)
print(f"{menu_text.splitlines()=}")
print(f"{raw_windows_path=}, {len(raw_windows_path)=}")
print(f"{escaped_windows_path=}, {len(escaped_windows_path)=}  (\\n and \\t were escapes)")
print(f"{rupee_sign=}", "Total:\t" + rupee_sign + "120")

Annapurna "Tiffin" Centre | It's open
menu_text.splitlines()=['Idli', 'Dosa', 'Vada']
raw_windows_path='C:\\new\\table', len(raw_windows_path)=12
escaped_windows_path='C:\new\table', len(escaped_windows_path)=10  (\n and \t were escapes)
rupee_sign='₹' Total:	₹120


Immutable means no method or assignment changes a string in place. Every "change"
builds a new string.

In [102]:
dish_name = "dosa"
try:
    dish_name[0] = "D"
except TypeError as error:
    print(f"{type(error).__name__}: {error}")
capitalized_dish_name = "D" + dish_name[1:]
print(f"{dish_name=}, {capitalized_dish_name=}  (a new string; the original is unchanged)")

TypeError: 'str' object does not support item assignment
dish_name='dosa', capitalized_dish_name='Dosa'  (a new string; the original is unchanged)


**Indexing and slicing.** `text[i]` returns a one-character string; negative indexes
count from the end. `text[start:stop:step]` copies a range (stop excluded), and
`[::-1]` reverses. An index out of range raises `IndexError`, but a slice out of range
just returns fewer characters.

In [103]:
order_code = "BLR-2026-0042"
print(f"{order_code[0]=}, {order_code[-1]=}, {order_code[-4:]=}")
print(f"{order_code[:3]=}, {order_code[4:8]=}, {order_code[::2]=}")
print(f"{order_code[::-1]=}")
print(f"{order_code[100:]=}, {order_code[-100:3]=}  (slices clamp, never raise)")
try:
    print(order_code[100])
except IndexError as error:
    print(f"{type(error).__name__}: {error}")

order_code[0]='B', order_code[-1]='2', order_code[-4:]='0042'
order_code[:3]='BLR', order_code[4:8]='2026', order_code[::2]='BR22-02'
order_code[::-1]='2400-6202-RLB'
order_code[100:]='', order_code[-100:3]='BLR'  (slices clamp, never raise)
IndexError: string index out of range


**What `len` counts.** Python's `len` counts **code points**. JavaScript's `.length`
counts UTF-16 code units, so any character outside the Basic Multilingual Plane (most
emoji) counts as 2 in JS. Neither counts what a person sees: 👍🏽 is two code points (a
thumb plus a skin-tone modifier) and 🇮🇳 is two regional-indicator letters.

In [104]:
for sample_text in ["dosa", "₹120", "😀", "👍🏽", "🇮🇳"]:
    code_point_count = len(sample_text)
    utf16_unit_count = len(sample_text.encode("utf-16-le")) // 2
    utf8_byte_count = len(sample_text.encode("utf-8"))
    print(
        f"{sample_text!r:8} len={code_point_count}  JS .length={utf16_unit_count}"
        f"  UTF-8 bytes={utf8_byte_count}"
    )

'dosa'   len=4  JS .length=4  UTF-8 bytes=4
'₹120'   len=4  JS .length=4  UTF-8 bytes=6
'😀'      len=1  JS .length=2  UTF-8 bytes=4
'👍🏽'     len=2  JS .length=4  UTF-8 bytes=8
'🇮🇳'     len=2  JS .length=4  UTF-8 bytes=8


#### Under the hood: PEP 393 compact strings

CPython stores each string with the narrowest fixed width that fits its **largest**
code point: 1 byte per character (Latin-1), 2 bytes (the rest of the BMP, e.g. ₹) or
4 bytes (emoji). Indexing stays O(1). One emoji makes the whole string 4 bytes per
character. Each row adds one more character and measures the growth. Exact sizes vary
by Python version.

In [105]:
for width_label, sample_text in [
    ("ASCII", "a" * 10),
    ("Latin-1 (é)", "é" * 10),
    ("BMP (₹)", "₹" * 10),
    ("astral (😀)", "😀" * 10),
    ("ASCII + one 😀", "a" * 9 + "😀"),
]:
    bytes_per_character = sys.getsizeof(sample_text + sample_text[-1]) - sys.getsizeof(sample_text)
    print(
        f"{width_label:15} {sys.getsizeof(sample_text):>4} bytes total, "
        f"{bytes_per_character} byte(s) per character"
    )

ASCII             51 bytes total, 1 byte(s) per character
Latin-1 (é)       67 bytes total, 1 byte(s) per character
BMP (₹)           78 bytes total, 2 byte(s) per character
astral (😀)       100 bytes total, 4 byte(s) per character
ASCII + one 😀    100 bytes total, 4 byte(s) per character


**Operators.** `+` concatenates, `*` repeats, `in` tests for a substring. Comparison is
**lexicographic by code point**: `"10" < "9"` because `"1"` (U+0031) comes before `"9"`
(U+0039). Uppercase letters sort before lowercase ones. Sort numbers stored as text with
`key=int`.

In [106]:
print(f"{'tea' + 'pot'=}, {'-' * 10=}, {'dos' in 'dosa'=}")
print(f"{'10' < '9'=}  (compares '1' with '9' first)")
print(f"{sorted(['10', '9', '100', '2'])=}")
print(f"{sorted(['10', '9', '100', '2'], key=int)=}")
print(f"{'apple' < 'Banana'=}, {ord('a')=}, {ord('B')=}  (uppercase sorts first)")

'tea' + 'pot'='teapot', '-' * 10='----------', 'dos' in 'dosa'=True
'10' < '9'=True  (compares '1' with '9' first)
sorted(['10', '9', '100', '2'])=['10', '100', '2', '9']
sorted(['10', '9', '100', '2'], key=int)=['2', '9', '10', '100']
'apple' < 'Banana'=False, ord('a')=97, ord('B')=66  (uppercase sorts first)


### 1.4.2 String methods (complete API by group)

| Group | Methods |
| --- | --- |
| Case | `lower`, `upper`, `title`, `capitalize`, `casefold`, `swapcase` |
| Searching | `find`, `rfind`, `index`, `rindex`, `count`, `startswith`, `endswith` (accept tuples), `in` |
| Cleaning | `strip`, `lstrip`, `rstrip` (character sets), `removeprefix`, `removesuffix` |
| Splitting and joining | `split()` vs `split(sep)`, `maxsplit`, `rsplit`, `splitlines`, `partition`, `rpartition`, `"sep".join(iterable)` |
| Replacing | `replace(old, new, count)`, `translate` with `str.maketrans`, `expandtabs` |
| Testing | `isdigit` vs `isdecimal` vs `isnumeric`, `isalpha`, `isalnum`, `isspace`, `isidentifier`, `islower`, `isupper`, `istitle`, `isascii`, `isprintable` |
| Layout | `center`, `ljust`, `rjust`, `zfill` |
| Encoding | `encode` |

Every method returns a **new** object; none of them changes the original string. Each
group below gets its own demo.

#### Case

`casefold` is a stronger `lower` meant for comparisons (1.4.4). `title` capitalizes
after *every* non-letter, including apostrophes.

In [107]:
city_name = "new DELHI"
print(f"{city_name.lower()=}")
print(f"{city_name.upper()=}")
print(f"{city_name.title()=}")
print(f"{city_name.capitalize()=}  (first character only, the rest lowered)")
print(f"{city_name.swapcase()=}")
print(f"{'Straße'.lower()=}, {'Straße'.casefold()=}")
print(f"{"o'neil's café".title()=}  (the apostrophe starts a new word)")

city_name.lower()='new delhi'
city_name.upper()='NEW DELHI'
city_name.title()='New Delhi'
city_name.capitalize()='New delhi'  (first character only, the rest lowered)
city_name.swapcase()='NEW delhi'
'Straße'.lower()='straße', 'Straße'.casefold()='strasse'
"o'neil's café".title()="O'Neil'S Café"  (the apostrophe starts a new word)


#### Searching

`find`/`rfind` return `-1` when nothing matches; `index`/`rindex` raise `ValueError`
instead. `startswith` and `endswith` accept a **tuple** of options. Use `in` when you
only need yes or no.

In [108]:
log_line = "GET /api/orders/42 200 GET"
print(f"{log_line.find('GET')=}, {log_line.rfind('GET')=}, {log_line.find('POST')=}")
print(f"{log_line.index('/api')=}, {log_line.rindex('GET')=}")
try:
    log_line.index("POST")
except ValueError as error:
    print(f"index: {type(error).__name__}: {error}")
print(f"{log_line.count('GET')=}, {log_line.count('/')=}")
file_name = "invoice_2026.PDF"
print(f"{file_name.startswith('invoice')=}")
print(f"{file_name.lower().endswith(('.pdf', '.png', '.jpg'))=}")
print(f"{'orders' in log_line=}")

log_line.find('GET')=0, log_line.rfind('GET')=23, log_line.find('POST')=-1
log_line.index('/api')=4, log_line.rindex('GET')=23
index: ValueError: substring not found
log_line.count('GET')=2, log_line.count('/')=3
file_name.startswith('invoice')=True
file_name.lower().endswith(('.pdf', '.png', '.jpg'))=True
'orders' in log_line=True


#### Cleaning

With no argument, `strip`, `lstrip` and `rstrip` remove whitespace. With an argument
they remove any characters **from that set**, not that exact prefix or suffix, which
surprises everyone once. Use `removeprefix` / `removesuffix` (3.9+) to drop an exact
piece of text.

In [109]:
raw_email = "  \t ramesh@example.com \n"
print(f"{raw_email.strip()=}")
print(f"{raw_email.lstrip()=}")
print(f"{raw_email.rstrip()=}")
print(f"{'xxhixyx'.strip('xy')=}  (any x or y, from both ends)")
site_name = "commerce.com"
print(f"{site_name.strip('.com')=}  (oops: strips the characters . c o m)")
print(f"{site_name.removesuffix('.com')=}")
print(f"{'www.example.in'.removeprefix('www.')=}")
print(f"{'example.in'.removeprefix('www.')=}  (no match: unchanged)")

raw_email.strip()='ramesh@example.com'
raw_email.lstrip()='ramesh@example.com \n'
raw_email.rstrip()='  \t ramesh@example.com'
'xxhixyx'.strip('xy')='hi'  (any x or y, from both ends)
site_name.strip('.com')='erce'  (oops: strips the characters . c o m)
site_name.removesuffix('.com')='commerce'
'www.example.in'.removeprefix('www.')='example.in'
'example.in'.removeprefix('www.')='example.in'  (no match: unchanged)


#### Splitting and joining

`split()` with no argument splits on any run of whitespace and drops empty edges;
`split(sep)` splits on exactly `sep` and keeps empty fields. `maxsplit` limits the
number of cuts (`rsplit` cuts from the right). `partition` always returns a 3-tuple,
so it is safe to unpack. `"sep".join(iterable)` is the inverse of `split`: the
separator is the object, and every item must already be a `str`.

In [110]:
messy_line = "  idli   dosa\tvada  "
print(f"{messy_line.split()=}")
print(f"{'a,,b'.split(',')=}  (empty field kept)")
print(f"{''.split()=}, {''.split(',')=}")
log_entry = "2026-10-30 12:00:01 ERROR payment failed: card declined"
print(f"{log_entry.split(' ', maxsplit=2)=}")
print(f"{'/srv/app/logs/today.log'.rsplit('/', maxsplit=1)=}")
print(f"{'line one\nline two\r\nline three'.splitlines()=}")
print(f"{'timeout=30'.partition('=')=}, {'no equals'.partition('=')=}")
print(f"{'report.final.pdf'.rpartition('.')=}")
print(f"{', '.join(['idli', 'dosa', 'vada'])=}")
print(f"join with ints: {raised_error_name(lambda: ', '.join([1, 2]))}")

messy_line.split()=['idli', 'dosa', 'vada']
'a,,b'.split(',')=['a', '', 'b']  (empty field kept)
''.split()=[], ''.split(',')=['']
log_entry.split(' ', maxsplit=2)=['2026-10-30', '12:00:01', 'ERROR payment failed: card declined']
'/srv/app/logs/today.log'.rsplit('/', maxsplit=1)=['/srv/app/logs', 'today.log']
'line one\nline two\r\nline three'.splitlines()=['line one', 'line two', 'line three']
'timeout=30'.partition('=')=('timeout', '=', '30'), 'no equals'.partition('=')=('no equals', '', '')
'report.final.pdf'.rpartition('.')=('report.final', '.', 'pdf')
', '.join(['idli', 'dosa', 'vada'])='idli, dosa, vada'
join with ints: TypeError


#### Replacing

`replace(old, new, count)` replaces every match unless you pass `count`.
`str.maketrans` builds a table for `translate`: map characters one to one, or map a
character to `None` to delete it. `expandtabs` turns tabs into spaces, aligned to tab
stops.

In [111]:
phone_number = "98-45-67-89-01"
print(f"{phone_number.replace('-', '')=}")
print(f"{phone_number.replace('-', ' ', 2)=}  (count limits the replacements)")
vowel_masker = str.maketrans("aeiou", "*****")
print(f"{'bangalore'.translate(vowel_masker)=}")
amount_cleaner = str.maketrans({"₹": None, ",": None, " ": None})
print(f"{'₹ 1,23,456'.translate(amount_cleaner)=}")
print(f"{str.maketrans('ab', 'xy', '!')=}  (a dict of code points)")
tabbed_row = "id\tname\tcity"
print(f"{tabbed_row.expandtabs(8)=}")
print(f"{tabbed_row.expandtabs(4)=}")

phone_number.replace('-', '')='9845678901'
phone_number.replace('-', ' ', 2)='98 45 67-89-01'  (count limits the replacements)
'bangalore'.translate(vowel_masker)='b*ng*l*r*'
'₹ 1,23,456'.translate(amount_cleaner)='123456'
str.maketrans('ab', 'xy', '!')={97: 120, 98: 121, 33: None}  (a dict of code points)
tabbed_row.expandtabs(8)='id      name    city'
tabbed_row.expandtabs(4)='id  name    city'


#### Testing

The three numeric tests differ, and only `isdecimal` matches what `int()` accepts:

| Method | True for | Example that is True |
| --- | --- | --- |
| `isdecimal` | digits in any script (`int()` accepts these) | `"42"`, `"٣"` (Arabic-Indic three) |
| `isdigit` | decimals plus superscripts and similar | `"²"` |
| `isnumeric` | digits plus fractions and numeral characters | `"½"`, `"四"` |

None of them accepts a sign or a decimal point.

In [112]:
print(f"{'sample':>8} {'isdecimal':>10} {'isdigit':>8} {'isnumeric':>10}")
for numeric_text in ["42", "٣", "²", "½", "四", "4.2", "-1"]:
    print(
        f"{numeric_text!r:>8} {numeric_text.isdecimal()!s:>10} "
        f"{numeric_text.isdigit()!s:>8} {numeric_text.isnumeric()!s:>10}"
    )
print(f"{int('٣')=}  (isdecimal text converts)")
print(f"int('²'): {raised_error_name(lambda: int('²'))}")

  sample  isdecimal  isdigit  isnumeric
    '42'       True     True       True
     '٣'       True     True       True
     '²'      False     True       True
     '½'      False    False       True
     '四'      False    False       True
   '4.2'      False    False      False
    '-1'      False    False      False
int('٣')=3  (isdecimal text converts)
int('²'): ValueError


The other tests. Every `is…` method returns `False` for the empty string, except
`isascii` and `isprintable`. `isidentifier` does not reject keywords; use
`keyword.iskeyword` for that.

In [113]:
print(f"{'dosa'.isalpha()=}, {'dosa2'.isalpha()=}, {'dosa2'.isalnum()=}")
print(f"{' \t\n'.isspace()=}, {''.isspace()=}")
print(f"{'order_id'.isidentifier()=}, {'2nd_order'.isidentifier()=}")
print(f"{'class'.isidentifier()=}  (a keyword, but still a valid identifier shape)")
print(f"{'dosa'.islower()=}, {'DOSA 2'.isupper()=}, {'Masala Dosa'.istitle()=}")
print(f"{'dosa'.isascii()=}, {'café'.isascii()=}, {''.isascii()=}")
print(f"{'plain text'.isprintable()=}, {'tab\there'.isprintable()=}")

'dosa'.isalpha()=True, 'dosa2'.isalpha()=False, 'dosa2'.isalnum()=True
' \t\n'.isspace()=True, ''.isspace()=False
'order_id'.isidentifier()=True, '2nd_order'.isidentifier()=False
'class'.isidentifier()=True  (a keyword, but still a valid identifier shape)
'dosa'.islower()=True, 'DOSA 2'.isupper()=True, 'Masala Dosa'.istitle()=True
'dosa'.isascii()=True, 'café'.isascii()=False, ''.isascii()=True
'plain text'.isprintable()=True, 'tab\there'.isprintable()=False


#### Layout

`ljust`, `rjust` and `center` pad to a width with an optional fill character; they
never truncate. `zfill` pads with zeros after the sign. The format-spec mini-language
in 1.4.3 does the same and more.

In [114]:
for menu_item, price_in_rupees in [("Idli", 40), ("Masala Dosa", 120), ("Coffee", 35)]:
    print(menu_item.ljust(16, ".") + str(price_in_rupees).rjust(6))
print(" Total ".center(22, "="))
print(f"{'42'.zfill(6)=}, {'-42'.zfill(6)=}  (zfill keeps the sign first)")
print(f"{'Masala Dosa'.ljust(5)=}  (never truncates)")

Idli............    40
Masala Dosa.....   120
Coffee..........    35
======= Total ========
'42'.zfill(6)='000042', '-42'.zfill(6)='-00042'  (zfill keeps the sign first)
'Masala Dosa'.ljust(5)='Masala Dosa'  (never truncates)


#### Encoding

`encode` turns text into `bytes` (UTF-8 unless you name another codec). A character
the codec cannot represent raises `UnicodeEncodeError` unless you choose an `errors`
policy; 1.4.5 covers the policies.

In [115]:
greeting_text = "Namaste ₹"
encoded_greeting = greeting_text.encode()
print(f"{encoded_greeting=}, {len(greeting_text)=}, {len(encoded_greeting)=}")
print(f"{greeting_text.encode('utf-16-le')=}")
try:
    greeting_text.encode("ascii")
except UnicodeEncodeError as error:
    print(f"{type(error).__name__}: {error}")
print(f"{greeting_text.encode('ascii', errors='replace')=}")

encoded_greeting=b'Namaste \xe2\x82\xb9', len(greeting_text)=9, len(encoded_greeting)=11
greeting_text.encode('utf-16-le')=b'N\x00a\x00m\x00a\x00s\x00t\x00e\x00 \x00\xb9 '
UnicodeEncodeError: 'ascii' codec can't encode character '\u20b9' in position 8: ordinal not in range(128)
greeting_text.encode('ascii', errors='replace')=b'Namaste ?'


### 1.4.3 String formatting

**f-strings** (PEP 498) evaluate any expression inside `{}`. Extras:

| Syntax | Meaning |
| --- | --- |
| `{expr=}` | debug: prints `expr=` and then the `repr` of its value (spaces around `=` are kept) |
| `{value!r}`, `{value!s}`, `{value!a}` | apply `repr()`, `str()` or `ascii()` before formatting |
| `{value:>{width}}` | nested field: the format spec itself contains a replacement field |
| multi-line `{…}` | since 3.12 (PEP 701) an expression may span lines, contain comments and reuse the outer quote character |

In [116]:
item_name = "Masala Dosa"
unit_price_in_rupees = 120
quantity = 3
print(f"{item_name} x {quantity} = ₹{unit_price_in_rupees * quantity}")
print(f"{item_name.upper()=}, {quantity * unit_price_in_rupees = }")
print(f"{item_name!r}, {item_name!s}, {'café ₹'!a}")
column_width = 15
print(f"[{item_name:>{column_width}}]  (width taken from a variable)")
print(f"[{item_name:^{column_width + 5}}]")

Masala Dosa x 3 = ₹360
item_name.upper()='MASALA DOSA', quantity * unit_price_in_rupees = 360
'Masala Dosa', Masala Dosa, 'caf\xe9 \u20b9'
[    Masala Dosa]  (width taken from a variable)
[    Masala Dosa     ]


A multi-line f-string (3.12+): the replacement field spans several lines and contains
a comment, with no backslashes.

In [117]:
order_lines = [("Idli", 2), ("Vada", 1), ("Coffee", 2)]
order_summary = f"Items: {
    ', '.join(
        f'{dish} x{count}'  # a comment inside the replacement field
        for dish, count in order_lines
    )
}"
print(order_summary)

Items: Idli x2, Vada x1, Coffee x2


**Format-spec mini-language** (everything after the `:`):

```text
[[fill]align][sign]["z"]["#"]["0"][width][grouping][.precision][type]
```

| Part | Options | Example → result |
| --- | --- | --- |
| fill + align | `<` left, `>` right, `^` centre, any fill character | `f"{'hi':*^6}"` → `**hi**` |
| sign | `+` always, `-` negatives only (default), space for a leading space | `f"{42:+d}"` → `+42` |
| `#` | alternate form: `0x`, `0b`, `0o` prefixes | `f"{255:#x}"` → `0xff` |
| width | minimum field width | `f"{7:5d}"` → `    7` |
| grouping | `,` or `_` thousands separator | `f"{1234567:,}"` → `1,234,567` |
| precision | digits after the point (or max characters for strings) | `f"{3.14159:.2f}"` → `3.14` |
| type | `d` int, `f` fixed, `e` exponent, `%` percent, `x` hex, `b` binary, `o` octal | `f"{0.256:.1%}"` → `25.6%` |

Dates have their own spec, passed to `strftime`: `f"{today:%Y-%m-%d}"`.

> **Note:** grouping is always in thousands (`12,34,567` lakh style is not built in).

In [118]:
order_total = 1234567.891
print(f"[{'left':<10}] [{'right':>10}] [{'centre':^10}] [{'fill':*^10}]")
print(f"{order_total:,.2f}  {order_total:_.1f}  {1_000_000:,d}  {1_000_000:_d}")
print(f"{order_total:.3e}  {0.1875:.1%}  [{order_total:15,.2f}]")
print(f"{255:x} {255:#x} {255:X} {5:b} {5:#010b} {8:o} {8:#o}")
print(f"[{42:+d}] [{-42:+d}] [{42: d}] [{42:d}]")
invoice_date = date(2026, 10, 30)
print(f"{invoice_date:%Y-%m-%d}  {invoice_date:%d %b %Y, %A}")
print(f"[{'truncated text':.9}]  (precision on a string cuts it)")

[left      ] [     right] [  centre  ] [***fill***]
1,234,567.89  1_234_567.9  1,000,000  1_000_000
1.235e+06  18.8%  [   1,234,567.89]
ff 0xff FF 101 0b00000101 10 0o10
[+42] [-42] [ 42] [42]
2026-10-30  30 Oct 2026, Friday
[truncated]  (precision on a string cuts it)


**Legacy styles** you will meet in older code and libraries:

| Style | Where you still see it |
| --- | --- |
| `"{} {name}".format(...)` | code from before 3.6; building a template once and filling it later |
| `"{name}".format_map(mapping)` | filling from a dict (or a dict subclass with `__missing__`) |
| `"%s %.2f" % (...)` | the `logging` module's lazy messages, old C-style code |
| `string.Template("$name")` | user-editable templates: no expressions, no attribute access |

The `%` line carries `# noqa: UP031` because ruff (rightly) suggests f-strings instead.

In [119]:
customer_record = {"name": "Asha", "balance": 2500.5}
print("{} owes ₹{:,.2f}".format("Asha", 2500.5))
print("{name} owes ₹{balance:,.2f}".format(**customer_record))
print("{name} owes ₹{balance:,.2f}".format_map(customer_record))
print("%s owes ₹%.2f (%d%% overdue)" % ("Asha", 2500.5, 12))  # noqa: UP031
greeting_template = string.Template("Hello $name, order $order_id is ready")
print(greeting_template.substitute(name="Asha", order_id="A-17"))
print(greeting_template.safe_substitute(name="Asha"))
try:
    greeting_template.substitute(name="Asha")
except KeyError as error:
    print(f"substitute: {type(error).__name__}: {error}")

Asha owes ₹2,500.50
Asha owes ₹2,500.50
Asha owes ₹2,500.50
Asha owes ₹2500.50 (12% overdue)
Hello Asha, order A-17 is ready
Hello Asha, order $order_id is ready
substitute: KeyError: 'order_id'


**Template strings, t-strings** (*new in 3.14*, PEP 750). A `t"..."` literal looks like
an f-string but produces a `string.templatelib.Template` object instead of a `str`:

- `.strings`: the literal text parts (always one more than the interpolations).
- `.interpolations`: `Interpolation` objects with `.value`, `.expression`,
  `.conversion` and `.format_spec`.
- Iterating yields the non-empty strings and interpolations in order.

Nothing is joined until **your** code renders it. That is the point: a renderer can
escape every value for HTML, turn values into SQL parameters, and so on.

In [120]:
customer_name = "Asha"
amount_due = 1234.5
reminder_template = t"Dear {customer_name}, you owe ₹{amount_due:,.2f}"
print(f"{type(reminder_template)=}")
print(f"{reminder_template.strings=}")
for interpolation in reminder_template.interpolations:
    print(f"{interpolation.expression=}, {interpolation.value=}, {interpolation.format_spec=}")
print(f"{list(reminder_template)=}")

type(reminder_template)=<class 'string.templatelib.Template'>
reminder_template.strings=('Dear ', ', you owe ₹', '')
interpolation.expression='customer_name', interpolation.value='Asha', interpolation.format_spec=''
interpolation.expression='amount_due', interpolation.value=1234.5, interpolation.format_spec=',.2f'
list(reminder_template)=['Dear ', Interpolation('Asha', 'customer_name', None, ''), ', you owe ₹', Interpolation(1234.5, 'amount_due', None, ',.2f')]


A safe HTML renderer: literal parts are trusted (the developer wrote them),
interpolated values are not, so each value is formatted with its spec and then passed
through `html.escape`. An f-string would have put the script tag straight into the
page.

In [121]:
def render_safe_html(template: Template) -> str:
    """Render a t-string, HTML-escaping every interpolated value."""
    rendered_parts: list[str] = []
    for template_part in template:
        if isinstance(template_part, Interpolation):
            formatted_value = format(template_part.value, template_part.format_spec)
            rendered_parts.append(html.escape(formatted_value))
        else:
            rendered_parts.append(template_part)
    return "".join(rendered_parts)


user_comment = "<script>alert('stolen cookie')</script>"
print("f-string:", f"<p>{user_comment}</p>")
print("t-string:", render_safe_html(t"<p>{user_comment}</p>"))

f-string: <p><script>alert('stolen cookie')</script></p>
t-string: <p>&lt;script&gt;alert(&#x27;stolen cookie&#x27;)&lt;/script&gt;</p>


### 1.4.4 Unicode correctness

The same visible text can be stored as different code-point sequences: `é` can be one
precomposed code point (U+00E9) or `e` followed by a **combining** acute accent
(U+0301). `==` compares code points, so the two are not equal until you
**normalize** them:

| Form | What it does |
| --- | --- |
| NFC | compose: `e` + `\u0301` → `é` (the usual storage form) |
| NFD | decompose: `é` → `e` + `\u0301` |
| NFKC / NFKD | also fold *compatibility* characters: `ﬁ` → `fi`, full-width `２` → `2`, `²` → `2` |

In [122]:
precomposed_name = "Jos\u00e9"
combining_name = "Jose\u0301"
print(f"{precomposed_name=}, {combining_name=}")
print(f"{precomposed_name == combining_name=}")
print(f"{len(precomposed_name)=}, {len(combining_name)=}")
nfc_combining_name = unicodedata.normalize("NFC", combining_name)
print(f"{nfc_combining_name == precomposed_name=}")
print(f"{len(unicodedata.normalize('NFD', precomposed_name))=}")
print(f"{unicodedata.is_normalized('NFC', combining_name)=}")
print(f"{unicodedata.normalize('NFKC', 'ﬁle ２０２６ x²')=}")

precomposed_name='José', combining_name='José'
precomposed_name == combining_name=False
len(precomposed_name)=4, len(combining_name)=5
nfc_combining_name == precomposed_name=True
len(unicodedata.normalize('NFD', precomposed_name))=5
unicodedata.is_normalized('NFC', combining_name)=False
unicodedata.normalize('NFKC', 'ﬁle ２０２６ x²')='file 2026 x2'


**Case-insensitive comparison** needs `casefold`, not `lower`: the German `ß` lowers
to itself but casefolds to `ss`. For a fully correct caseless match, normalize *and*
casefold both sides. `unicodedata.name` tells you exactly which code point you are
looking at, which helps with "identical-looking" bugs.

In [123]:
def texts_match(first_text: str, second_text: str) -> bool:
    """Compare text the Unicode way: NFC-normalize and casefold both sides."""
    first_key = unicodedata.normalize("NFC", first_text).casefold()
    second_key = unicodedata.normalize("NFC", second_text).casefold()
    return first_key == second_key


print(f"{'Straße'.lower() == 'STRASSE'.lower()=}")
print(f"{'Straße'.casefold() == 'STRASSE'.casefold()=}")
print(f"{texts_match('JOSE\u0301', 'josé')=}")
for character in "é₹ß😀\u0301\u00a0":
    print(f"U+{ord(character):04X} {unicodedata.name(character)}")

'Straße'.lower() == 'STRASSE'.lower()=False
'Straße'.casefold() == 'STRASSE'.casefold()=True
texts_match('JOSE\u0301', 'josé')=True
U+00E9 LATIN SMALL LETTER E WITH ACUTE
U+20B9 INDIAN RUPEE SIGN
U+00DF LATIN SMALL LETTER SHARP S
U+1F600 GRINNING FACE
U+0301 COMBINING ACUTE ACCENT
U+00A0 NO-BREAK SPACE


### 1.4.5 Bytes (`bytes`) and byte arrays (`bytearray`)

`str` is text (code points); `bytes` is binary data (integers 0–255). You convert at
the boundary: **encode** text when it leaves your program (files, sockets, hashes) and
**decode** bytes when they come in.

```mermaid
flowchart LR
    text["str<br/>'₹500'"] -- "encode('utf-8')" --> raw["bytes<br/>b'\xe2\x82\xb9500'"]
    raw -- "decode('utf-8')" --> text
```

<details><summary>ASCII fallback</summary>

```text
str '₹500'  --encode('utf-8')-->  bytes b'\xe2\x82\xb9500'
            <--decode('utf-8')--
```

</details>

A `b"..."` literal may contain only ASCII characters and escapes. **Indexing a
`bytes` object returns an `int`**; slicing returns `bytes`. `bytes` is immutable,
`bytearray` is its mutable twin.

In [124]:
png_header = b"\x89PNG\r\n\x1a\n"
http_request_line = b"GET / HTTP/1.1\r\n"
print(f"{png_header=}, {len(png_header)=}")
print(f"{http_request_line[0]=}  (indexing gives an int)")
print(f"{http_request_line[:3]=}  (slicing gives bytes)")
print(f"{list(b'AZ az')=}")
try:
    eval('b"café"')
except SyntaxError as error:
    print(f"{type(error).__name__}: {error.msg}")

png_header=b'\x89PNG\r\n\x1a\n', len(png_header)=8
http_request_line[0]=71  (indexing gives an int)
http_request_line[:3]=b'GET'  (slicing gives bytes)
list(b'AZ az')=[65, 90, 32, 97, 122]
SyntaxError: bytes can only contain ASCII literal characters


`bytes` cannot change after it is created. `bytearray` supports item assignment,
`extend`, `append` and `del`, and `bytes(…)` freezes a copy of it.

In [125]:
frozen_payload = b"hello"
try:
    frozen_payload[0] = 72
except TypeError as error:
    print(f"bytes: {type(error).__name__}: {error}")
mutable_buffer = bytearray(b"hello")
mutable_buffer[0] = ord("H")
mutable_buffer.extend(b" world")
del mutable_buffer[-1]
print(f"{mutable_buffer=}, {bytes(mutable_buffer)=}")
print(f"{bytearray(4)=}  (four zero bytes)")

bytes: TypeError: 'bytes' object does not support item assignment
mutable_buffer=bytearray(b'Hello worl'), bytes(mutable_buffer)=b'Hello worl'
bytearray(4)=bytearray(b'\x00\x00\x00\x00')  (four zero bytes)


**`encode` / `decode` and the `errors` argument.** The default `errors="strict"`
raises on bad data. The alternatives:

| `errors=` | On a bad character or byte |
| --- | --- |
| `strict` | raise `UnicodeEncodeError` / `UnicodeDecodeError` (default) |
| `replace` | substitute `?` (encode) or `�` U+FFFD (decode) |
| `ignore` | drop it silently (data loss!) |
| `backslashreplace` | write a `\x..` / `\u....` escape so nothing is lost |

The bytes below are cut in the middle of the 3-byte `₹`, which is exactly what happens
when a network read stops halfway through a character.

In [126]:
customer_note = "Paid ₹500 — thanks"
note_bytes = customer_note.encode("utf-8")
print(f"{note_bytes=}")
print(f"{note_bytes.decode('utf-8')=}")
truncated_bytes = note_bytes[:7]
for error_policy in ["strict", "replace", "ignore", "backslashreplace"]:
    try:
        decoded_text = truncated_bytes.decode("utf-8", errors=error_policy)
    except UnicodeDecodeError as error:
        decoded_text = f"{type(error).__name__}: {error}"
    print(f"decode {error_policy:>16}: {decoded_text!r}")
print(f"{customer_note.encode('ascii', errors='backslashreplace')=}")
print(f"{customer_note.encode('ascii', errors='ignore')=}")

note_bytes=b'Paid \xe2\x82\xb9500 \xe2\x80\x94 thanks'
note_bytes.decode('utf-8')='Paid ₹500 — thanks'
decode           strict: "UnicodeDecodeError: 'utf-8' codec can't decode bytes in position 5-6: unexpected end of data"
decode          replace: 'Paid �'
decode           ignore: 'Paid '
decode backslashreplace: 'Paid \\xe2\\x82'
customer_note.encode('ascii', errors='backslashreplace')=b'Paid \\u20b9500 \\u2014 thanks'
customer_note.encode('ascii', errors='ignore')=b'Paid 500  thanks'


**UTF-8 by default** (*changed in 3.15*, PEP 686). Python 3.15 turns on **UTF-8 mode**
by default, so `open()` without `encoding=` reads and writes UTF-8 on every OS, no
matter the locale (before 3.15, Windows used the ANSI code page). `locale.getencoding()`
still reports the OS locale. Set `PYTHONUTF8=0` to opt out. Code that must also run on
older Pythons should keep passing `encoding="utf-8"` explicitly.

In [127]:
print(f"{sys.flags.utf8_mode=}")
print(f"{locale.getpreferredencoding(False)=}  (what open() uses without encoding=)")
print(f"{locale.getencoding()=}  (the OS locale encoding)")
if sys.version_info >= (3, 15):
    print("Python 3.15+: UTF-8 mode is on by default (PEP 686)")
else:
    print("Before 3.15: open() uses the locale encoding; pass encoding='utf-8'")

sys.flags.utf8_mode=1
locale.getpreferredencoding(False)='utf-8'  (what open() uses without encoding=)
locale.getencoding()='UTF-8'  (the OS locale encoding)
Python 3.15+: UTF-8 mode is on by default (PEP 686)


**Methods shared with `str`.** Most `str` methods exist on `bytes` and `bytearray`, but
their arguments must be bytes too. `hex()` and `bytes.fromhex()` convert to and from
hex text, which you see in checksums and protocol dumps.

In [128]:
raw_header = b"Content-Type: text/html; charset=utf-8"
print(f"{raw_header.split(b': ', maxsplit=1)=}")
print(f"{raw_header.replace(b'html', b'plain')=}")
print(f"{raw_header.startswith(b'Content')=}, {raw_header.upper().find(b'UTF')=}")
print(f"{raw_header.partition(b'; ')=}")
print(f"{b'  padded  '.strip()=}, {b'abc'.isalpha()=}")
checksum_bytes = bytes([0xDE, 0xAD, 0xBE, 0xEF])
print(f"{checksum_bytes.hex()=}, {checksum_bytes.hex(':')=}")
print(f"{bytes.fromhex('de ad be ef')=}")

raw_header.split(b': ', maxsplit=1)=[b'Content-Type', b'text/html; charset=utf-8']
raw_header.replace(b'html', b'plain')=b'Content-Type: text/plain; charset=utf-8'
raw_header.startswith(b'Content')=True, raw_header.upper().find(b'UTF')=33
raw_header.partition(b'; ')=(b'Content-Type: text/html', b'; ', b'charset=utf-8')
b'  padded  '.strip()=b'padded', b'abc'.isalpha()=True
checksum_bytes.hex()='deadbeef', checksum_bytes.hex(':')='de:ad:be:ef'
bytes.fromhex('de ad be ef')=b'\xde\xad\xbe\xef'


**Mixing `str` and `bytes` raises `TypeError`.** Python 2 silently mixed them and
corrupted text; Python 3 refuses, so the bug shows up where it starts. Equality is the
one exception: it simply returns `False`. The fix is always to decode (or encode) at
the boundary and then work in one type.

In [129]:
text_part = "order="
byte_part = b"42"
try:
    text_part + byte_part
except TypeError as error:
    print(f"str + bytes: {type(error).__name__}: {error}")
print(f"bytes in str: {raised_error_name(lambda: byte_part in text_part)}")
print(f"str.join([str, bytes]): {raised_error_name(lambda: ''.join([text_part, byte_part]))}")
print(f"{b'42' == '42'=}  (no error, just never equal)")
print(f"{text_part + byte_part.decode()=}  (decode first, then work in str)")

str + bytes: TypeError: can only concatenate str (not "bytes") to str
bytes in str: TypeError
str.join([str, bytes]): TypeError
b'42' == '42'=False  (no error, just never equal)
text_part + byte_part.decode()='order=42'  (decode first, then work in str)


### 1.4.6 Memory views (`memoryview`)

Slicing `bytes` **copies** the data. A `memoryview` exposes another object's buffer
without copying: slicing a view creates another view, and over a `bytearray` you can
write through it. This matters when you parse large binary payloads. Phase 5 returns
to buffers.

> **Note:** while a view exists, the `bytearray` cannot be resized.

In [130]:
frame_buffer = bytearray(b"HEADER:payload-data:CRC")
header_view = memoryview(frame_buffer)[:6]
payload_view = memoryview(frame_buffer)[7:19]
print(f"{bytes(header_view)=}, {bytes(payload_view)=}")
print(f"{payload_view.nbytes=}, {payload_view.obj is frame_buffer=}, {payload_view.readonly=}")
payload_view[:7] = b"PAYLOAD"
print(f"{frame_buffer=}  (written through the view)")
print(f"append while viewed: {raised_error_name(lambda: frame_buffer.extend(b'!'))}")
large_blob = bytes(10_000_000)
print(f"bytes slice:      {sys.getsizeof(large_blob[1:]):>12,} bytes (a copy)")
print(f"memoryview slice: {sys.getsizeof(memoryview(large_blob)[1:]):>12,} bytes")
header_view.release()
payload_view.release()

bytes(header_view)=b'HEADER', bytes(payload_view)=b'payload-data'
payload_view.nbytes=12, payload_view.obj is frame_buffer=True, payload_view.readonly=False
frame_buffer=bytearray(b'HEADER:PAYLOAD-data:CRC')  (written through the view)
append while viewed: BufferError
bytes slice:        10,000,032 bytes (a copy)
memoryview slice:          184 bytes


### 1.4.7 Real-world use cases

**Normalizing user input.** Store and compare a canonical form: strip the edges,
lowercase or casefold, and reject what is not valid. Here, emails: domains are
case-insensitive, and in practice the whole address is treated that way. Lab 1.4 does
the same for usernames and search queries.

In [131]:
def normalize_email(raw_email: str) -> str:
    """Return a trimmed, lowercased email address, or raise ValueError."""
    cleaned_email = raw_email.strip().lower()
    local_part, separator, domain_part = cleaned_email.rpartition("@")
    if not separator or not local_part or "." not in domain_part:
        raise ValueError(f"not an email address: {raw_email!r}")
    return cleaned_email


for raw_email in ["  Asha.K@Example.COM ", "ravi@mail.in", "no-at-sign.com"]:
    try:
        print(f"{raw_email!r:24} -> {normalize_email(raw_email)!r}")
    except ValueError as error:
        print(f"{raw_email!r:24} -> {type(error).__name__}: {error}")

'  Asha.K@Example.COM '  -> 'asha.k@example.com'
'ravi@mail.in'           -> 'ravi@mail.in'
'no-at-sign.com'         -> ValueError: not an email address: 'no-at-sign.com'


**Parsing structured text.** `split(maxsplit=…)` handles fixed-shape log lines, and
`partition` handles `key=value` lines (it never fails to unpack). `parse_log_line`
returns to the Phase 1 project, the log analyzer.

In [132]:
def parse_log_line(log_line: str) -> dict[str, str]:
    """Split 'DATE TIME LEVEL message' into its four named fields."""
    log_date, log_time, level, message = log_line.split(maxsplit=3)
    return {"date": log_date, "time": log_time, "level": level, "message": message}


def parse_key_value_line(config_line: str) -> tuple[str, str] | None:
    """Parse 'key = value'; return None for blank and comment-only lines."""
    content = config_line.partition("#")[0].strip()
    if not content:
        return None
    key, separator, value = content.partition("=")
    if not separator:
        raise ValueError(f"missing '=' in {config_line!r}")
    return key.strip(), value.strip()


print(parse_log_line("2026-10-30 12:00:01 ERROR payment failed: card declined"))

{'date': '2026-10-30', 'time': '12:00:01', 'level': 'ERROR', 'message': 'payment failed: card declined'}


Feeding a small config file through `parse_key_value_line`: blank lines and comments
are skipped, and the spaces around `=` are optional.

In [133]:
config_text = """
# database settings
host = db.internal
port=5432
timeout = 30  # seconds
"""
settings: dict[str, str] = {}
for config_line in config_text.splitlines():
    if (parsed_setting := parse_key_value_line(config_line)) is not None:
        setting_key, setting_value = parsed_setting
        settings[setting_key] = setting_value
print(f"{settings=}")

settings={'host': 'db.internal', 'port': '5432', 'timeout': '30'}


CSV fields: `split(",")` is fine for simple, unquoted data. As soon as a field can
contain a comma inside quotes, use the `csv` module.

In [134]:
simple_csv_row = "A-17,Asha,1200.50"
order_id, customer, amount_text = simple_csv_row.split(",")
print(f"{order_id=}, {customer=}, {Decimal(amount_text)=}")
quoted_csv_row = 'A-18,"Kumar, Asha",950.00'
print(f"{quoted_csv_row.split(',')=}  (wrong: 4 fields)")
print(f"{next(csv.reader([quoted_csv_row]))=}  (right: 3 fields)")

order_id='A-17', customer='Asha', Decimal(amount_text)=Decimal('1200.50')
quoted_csv_row.split(',')=['A-18', '"Kumar', ' Asha"', '950.00']  (wrong: 4 fields)
next(csv.reader([quoted_csv_row]))=['A-18', 'Kumar, Asha', '950.00']  (right: 3 fields)


**Reports with the format-spec mini-language.** Fixed-width columns, thousands
separators and percentages need no string arithmetic.

In [135]:
revenue_by_region = {
    "North": 1_250_000.0,
    "South": 2_480_500.5,
    "East": 975_250.25,
    "West": 1_810_000.0,
}
total_revenue = sum(revenue_by_region.values())
print(f"{'Region':<8}{'Revenue (₹)':>16}{'Share':>9}")
print("-" * 33)
for region_name, region_revenue in revenue_by_region.items():
    print(f"{region_name:<8}{region_revenue:>16,.2f}{region_revenue / total_revenue:>9.1%}")
print("-" * 33)
print(f"{'Total':<8}{total_revenue:>16,.2f}{1:>9.0%}")

Region       Revenue (₹)    Share
---------------------------------
North       1,250,000.00    19.2%
South       2,480,500.50    38.1%
East          975,250.25    15.0%
West        1,810,000.00    27.8%
---------------------------------
Total       6,515,750.75     100%


**URL slugs.** NFKD splits accented letters into base letter plus mark; encoding to
ASCII with `errors="ignore"` drops the marks; every run of other characters becomes
one hyphen.

In [136]:
def slugify(title: str) -> str:
    """Turn a title into a URL slug: ASCII, lowercase, words joined by hyphens."""
    decomposed_title = unicodedata.normalize("NFKD", title)
    ascii_title = decomposed_title.encode("ascii", errors="ignore").decode("ascii")
    words = "".join(
        character if character.isalnum() else " " for character in ascii_title.lower()
    ).split()
    return "-".join(words)


for blog_title in ["Crème Brûlée: 10 Best Recipes (2026)!", "  Hello,   World  "]:
    print(f"{blog_title!r} -> {slugify(blog_title)!r}")

'Crème Brûlée: 10 Best Recipes (2026)!' -> 'creme-brulee-10-best-recipes-2026'
'  Hello,   World  ' -> 'hello-world'


**Safe SQL with t-strings.** `render_sql` turns a t-string into a parameterized query:
each interpolation becomes a `?` placeholder and its value goes into a separate tuple,
so the database never treats data as SQL. Compare it with an f-string built query on
an in-memory SQLite database. (`render_safe_html` from 1.4.3 covers the HTML side.)

In [137]:
def render_sql(template: Template) -> tuple[str, tuple[object, ...]]:
    """Turn a t-string into '?' placeholders plus a tuple of parameters."""
    query_parts: list[str] = []
    query_parameters: list[object] = []
    for template_part in template:
        if isinstance(template_part, Interpolation):
            query_parts.append("?")
            query_parameters.append(template_part.value)
        else:
            query_parts.append(template_part)
    return "".join(query_parts), tuple(query_parameters)


print(render_sql(t"SELECT id FROM orders WHERE customer = {'Asha'} AND total > {500}"))

('SELECT id FROM orders WHERE customer = ? AND total > ?', ('Asha', 500))


Now against a real (in-memory) database. The hostile name closes the quote in the
f-string version and matches every row; as a parameter it is just an odd name that
matches nothing.

In [138]:
connection = sqlite3.connect(":memory:")
connection.execute("CREATE TABLE orders (id INTEGER, customer TEXT, total INTEGER)")
connection.executemany("INSERT INTO orders VALUES (?, ?, ?)", [(1, "Asha", 900), (2, "Ravi", 300)])
hostile_name = "nobody' OR '1'='1"
unsafe_query = f"SELECT id FROM orders WHERE customer = '{hostile_name}'"
print(f"f-string query rows: {connection.execute(unsafe_query).fetchall()}  (injected!)")
safe_query, safe_parameters = render_sql(t"SELECT id FROM orders WHERE customer = {hostile_name}")
print(f"t-string query rows: {connection.execute(safe_query, safe_parameters).fetchall()}")
connection.close()

f-string query rows: [(1,), (2,)]  (injected!)
t-string query rows: []


**`bytes` in the real world.** Network protocols and file formats are bytes. Files
announce their type in the first few bytes (the "magic number"), hashes work on bytes,
and base64 turns bytes into ASCII text that is safe for JSON, emails and URLs.

In [139]:
FILE_SIGNATURES: dict[bytes, str] = {
    b"\x89PNG\r\n\x1a\n": "PNG image",
    b"%PDF-": "PDF document",
    b"PK\x03\x04": "ZIP archive (also .docx, .xlsx)",
    b"\xff\xd8\xff": "JPEG image",
}


def detect_file_type(file_header: bytes) -> str:
    """Name the file type from its first bytes, or 'unknown'."""
    for signature, file_type in FILE_SIGNATURES.items():
        if file_header.startswith(signature):
            return file_type
    return "unknown"


for sample_header in [b"%PDF-1.7\n%\xe2\xe3", b"\x89PNG\r\n\x1a\n\x00\x00", b"hello"]:
    print(f"{sample_header[:8]!r:36} -> {detect_file_type(sample_header)}")

b'%PDF-1.7'                          -> PDF document
b'\x89PNG\r\n\x1a\n'                 -> PNG image
b'hello'                             -> unknown


A JSON request body: its UTF-8 byte length (not its character count) is the HTTP
`Content-Length`; SHA-256 fingerprints it; base64 makes it ASCII-safe. The URL-safe
alphabet swaps `+` and `/` for `-` and `_`.

In [140]:
request_body = '{"item": "dosa", "price": "₹120"}'
payload_bytes = request_body.encode("utf-8")
print(f"{len(request_body)=} characters, {len(payload_bytes)=} bytes (Content-Length)")
payload_digest = hashlib.sha256(payload_bytes).hexdigest()
print(f"{payload_digest=}")
encoded_payload = base64.b64encode(payload_bytes)
print(f"{encoded_payload=}")
print(f"{base64.b64decode(encoded_payload).decode('utf-8')=}")
print(f"{base64.b64encode(b'\xfb\xff')=}, {base64.urlsafe_b64encode(b'\xfb\xff')=}")

len(request_body)=33 characters, len(payload_bytes)=35 bytes (Content-Length)
payload_digest='9fb30cb1ec0a47b5b0e8d4a3698a760e662aa6cfdc88d4a0cd2f97b49e87025e'
encoded_payload=b'eyJpdGVtIjogImRvc2EiLCAicHJpY2UiOiAi4oK5MTIwIn0='
base64.b64decode(encoded_payload).decode('utf-8')='{"item": "dosa", "price": "₹120"}'
base64.b64encode(b'\xfb\xff')=b'+/8=', base64.urlsafe_b64encode(b'\xfb\xff')=b'-_8='


### Lab 1.4

Three text-cleaning functions of the kind every web application needs. For each one,
replace the `raise NotImplementedError` in the stub, then run the check cell. Until
then the checks print FAIL with `NotImplementedError`; that is expected.

#### Exercise 1 — `normalize_username`

Write `normalize_username(raw_username: str) -> str`:

1. Normalize with **NFKC** (full-width `ＡＳＨＡ` becomes `ASHA`; `e` + combining accent
   becomes `é`).
2. Strip the edges and **casefold**.
3. Replace each run of inner whitespace with a single `_`.
4. Raise `ValueError` if the result is empty, or if it contains any character that is
   not alphanumeric (`str.isalnum`), `_` or `.`.

`normalize_username("  Ravi   Shankar ") == "ravi_shankar"`.

In [141]:
def normalize_username(raw_username: str) -> str:
    """Return the canonical form of a username, or raise ValueError."""
    raise NotImplementedError("Lab 1.4: implement normalize_username()")

Run the checks. They cover full-width letters, `ß`, a combining accent and two
invalid inputs.

In [142]:
run_checks(
    [
        ("strip and casefold", lambda: normalize_username("  Asha.Kumar "), "asha.kumar"),
        ("inner spaces become _", lambda: normalize_username("Ravi   Shankar"), "ravi_shankar"),
        ("full-width letters", lambda: normalize_username("ＡＳＨＡ２６"), "asha26"),
        ("ß casefolds to ss", lambda: normalize_username("STRAßE"), "strasse"),
        (
            "combining accent equals precomposed",
            lambda: normalize_username("Jose\u0301") == normalize_username("JOSÉ"),
            True,
        ),
        (
            "blank raises",
            lambda: raised_error_name(lambda: normalize_username("   ")),
            "ValueError",
        ),
        (
            "@ raises",
            lambda: raised_error_name(lambda: normalize_username("asha@home")),
            "ValueError",
        ),
    ]
)

FAIL  strip and casefold: raised NotImplementedError: Lab 1.4: implement normalize_username()
FAIL  inner spaces become _: raised NotImplementedError: Lab 1.4: implement normalize_username()
FAIL  full-width letters: raised NotImplementedError: Lab 1.4: implement normalize_username()
FAIL  ß casefolds to ss: raised NotImplementedError: Lab 1.4: implement normalize_username()
FAIL  combining accent equals precomposed: raised NotImplementedError: Lab 1.4: implement normalize_username()
FAIL  blank raises: expected 'ValueError', got 'NotImplementedError'
FAIL  @ raises: expected 'ValueError', got 'NotImplementedError'
0/7 checks passed


#### Exercise 2 — `format_receipt_line`

Write `format_receipt_line(item_name: str, quantity: int, unit_price: Decimal) -> str`
that returns one fixed-width receipt line of exactly **49 characters**:

| Column | Width | Rule |
| --- | --- | --- |
| item name | 20 | left-aligned; a longer name is cut to 19 characters plus `…` |
| quantity | 4 | right-aligned |
| separator | 3 | the text `" x "` |
| unit price | 10 | right-aligned, `,` separators, 2 decimal places |
| line total | 12 | `quantity * unit_price`, right-aligned, `,` separators, 2 decimal places |

```text
Masala Dosa            2 x     120.00      240.00
```

You may assume the quantity, price and total fit their columns.

In [143]:
def format_receipt_line(item_name: str, quantity: int, unit_price: Decimal) -> str:
    """Return a 49-character receipt line: name, quantity, unit price and line total."""
    raise NotImplementedError("Lab 1.4: implement format_receipt_line()")

The checks compare exact strings, so every space counts.

In [144]:
run_checks(
    [
        (
            "basic line",
            lambda: format_receipt_line("Masala Dosa", 2, Decimal("120")),
            "Masala Dosa            2 x     120.00      240.00",
        ),
        (
            "long name is truncated with …",
            lambda: format_receipt_line("Paneer Butter Masala Special", 1, Decimal("320.5")),
            "Paneer Butter Masal…   1 x     320.50      320.50",
        ),
        (
            "thousands separators",
            lambda: format_receipt_line("Wedding Catering", 3, Decimal("125000")),
            "Wedding Catering       3 x 125,000.00  375,000.00",
        ),
        (
            "always 49 characters",
            lambda: {
                len(format_receipt_line(name, count, Decimal(price)))
                for name, count, price in [("Tea", 1, "10"), ("A" * 40, 99, "9999.99")]
            },
            {49},
        ),
    ]
)

FAIL  basic line: raised NotImplementedError: Lab 1.4: implement format_receipt_line()
FAIL  long name is truncated with …: raised NotImplementedError: Lab 1.4: implement format_receipt_line()
FAIL  thousands separators: raised NotImplementedError: Lab 1.4: implement format_receipt_line()
FAIL  always 49 characters: raised NotImplementedError: Lab 1.4: implement format_receipt_line()
0/4 checks passed


#### Exercise 3 — search-query normalization

Write `normalize_search_query(raw_query: str) -> str` so that differently typed
queries match the same search index entry:

1. Normalize with **NFKD**, then drop every combining mark (`unicodedata.combining`
   returns non-zero for them), so `é` becomes `e`.
2. Casefold.
3. Turn every character that is not alphanumeric into a space.
4. Collapse whitespace runs into single spaces and strip the edges.

`normalize_search_query("  Café   CRÈME! ") == "cafe creme"`.

> **Note:** dropping marks is right for accented Latin text but wrong for scripts such
> as Devanagari, where marks are part of the letters. Real search engines use
> per-language analyzers.

In [145]:
def normalize_search_query(raw_query: str) -> str:
    """Return a lowercase, accent-free, single-spaced version of a search query."""
    raise NotImplementedError("Lab 1.4: implement normalize_search_query()")

Run the checks.

In [146]:
run_checks(
    [
        ("accents and case", lambda: normalize_search_query("  Café   CRÈME! "), "cafe creme"),
        (
            "punctuation becomes spaces",
            lambda: normalize_search_query("Masala-Dosa,  Bengaluru"),
            "masala dosa bengaluru",
        ),
        ("full-width text", lambda: normalize_search_query("ＤＯＳＡ ２０２６"), "dosa 2026"),
        (
            "ß and SS match",
            lambda: normalize_search_query("Straße") == normalize_search_query("STRASSE"),
            True,
        ),
        ("empty query", lambda: normalize_search_query(""), ""),
        ("only punctuation", lambda: normalize_search_query("!!! ???"), ""),
    ]
)

FAIL  accents and case: raised NotImplementedError: Lab 1.4: implement normalize_search_query()
FAIL  punctuation becomes spaces: raised NotImplementedError: Lab 1.4: implement normalize_search_query()
FAIL  full-width text: raised NotImplementedError: Lab 1.4: implement normalize_search_query()
FAIL  ß and SS match: raised NotImplementedError: Lab 1.4: implement normalize_search_query()
FAIL  empty query: raised NotImplementedError: Lab 1.4: implement normalize_search_query()
FAIL  only punctuation: raised NotImplementedError: Lab 1.4: implement normalize_search_query()
0/6 checks passed


### Recap — Unit 1.4

| Concept | One-line takeaway | JS equivalent |
| --- | --- | --- |
| `str` | Immutable sequence of code points; every method returns a new string | `string` |
| `len` | Counts code points, not UTF-16 units or visible characters | `.length` (UTF-16 units) |
| PEP 393 | CPython stores 1, 2 or 4 bytes per character, chosen per string | — |
| Slicing | `[start:stop:step]`, `[::-1]`; slices never raise | `slice()`, `at()` |
| `strip(chars)` | Removes a *set* of characters; use `removeprefix`/`removesuffix` for exact text | `trim()` |
| `split` / `partition` | `split()` for whitespace, `split(sep)` keeps empty fields, `partition` always gives 3 parts | `split()` |
| `isdecimal` | The numeric test that matches what `int()` accepts | `/^\d+$/` |
| f-strings | Expressions, `=` debugging, `!r`, nested specs, multi-line (3.12+) | template literals |
| Format spec | `fill align sign # width , .precision type`, plus `strftime` codes for dates | `toLocaleString`, `toFixed` |
| t-strings (3.14) | `Template` objects that a renderer escapes safely | tagged templates |
| Normalization | NFC before comparing; NFKC/NFKD also fold look-alike characters | `normalize("NFC")` |
| `casefold` | Use for caseless comparison (`ß` → `ss`) | `toLowerCase()` (not enough) |
| `bytes` / `bytearray` | Binary data; indexing gives `int`; encode/decode at the boundary | `Uint8Array`, `Buffer` |
| `errors=` | `strict`, `replace`, `ignore`, `backslashreplace` | `TextDecoder({fatal})` |
| UTF-8 mode (3.15) | `open()` defaults to UTF-8 everywhere (PEP 686) | — |
| `memoryview` | Zero-copy slices of a buffer | `subarray()` |

**Further reading**

- [Text sequence type — `str`](https://docs.python.org/3/library/stdtypes.html#text-sequence-type-str),
  [string methods](https://docs.python.org/3/library/stdtypes.html#string-methods)
- [Format specification mini-language](https://docs.python.org/3/library/string.html#format-specification-mini-language),
  [f-strings](https://docs.python.org/3/reference/lexical_analysis.html#f-strings),
  [`string.templatelib`](https://docs.python.org/3/library/string.templatelib.html)
- [Unicode HOWTO](https://docs.python.org/3/howto/unicode.html),
  [`unicodedata`](https://docs.python.org/3/library/unicodedata.html)
- [Binary sequence types — `bytes`, `bytearray`, `memoryview`](https://docs.python.org/3/library/stdtypes.html#binary-sequence-types-bytes-bytearray-memoryview),
  [codec error handlers](https://docs.python.org/3/library/codecs.html#error-handlers)
- [PEP 393 — Flexible string representation](https://peps.python.org/pep-0393/),
  [PEP 498 — Literal string interpolation](https://peps.python.org/pep-0498/),
  [PEP 701 — Syntactic formalization of f-strings](https://peps.python.org/pep-0701/),
  [PEP 750 — Template strings](https://peps.python.org/pep-0750/),
  [PEP 686 — Make UTF-8 mode default](https://peps.python.org/pep-0686/)

## Unit 1.5 — Collection Data Types I: Lists, Tuples and Ranges

**Objectives**

- Use the operations every sequence shares, and know which types support which.
- Explain how a `list` is stored, and the cost of each list operation.
- Use the complete `list` API, slice assignment and starred unpacking.
- Sort by several keys, stably, in mixed directions.
- Choose between `list`, `tuple`, named tuples, `range` and `array.array`.

**JS bridge**

| JavaScript concept | Python equivalent | The difference that bites |
| --- | --- | --- |
| `arr.push(x)` / `arr.pop()` | `items.append(x)` / `items.pop()` | `append` returns `None`, not the new length |
| `arr.shift()` / `arr.unshift(x)` | `items.pop(0)` / `items.insert(0, x)` | O(n): use `collections.deque` (1.6.7) for queues |
| `[10, 9, 1].sort()` → `[1, 10, 9]` | `sorted([10, 9, 1])` → `[1, 9, 10]` | Python compares values; mixing `int` and `str` raises `TypeError` |
| `arr.sort()` returns the array | `items.sort()` returns `None` | Use `sorted(items)` when you want a new list |
| `arr.slice(1, 3)`, `arr.at(-1)` | `items[1:3]`, `items[-1]` | Slices also take a step, and assigning to a slice replaces it |
| `Object.freeze([lat, lng])` | `(lat, lng)` tuple | A tuple is hashable, so it can be a dict key |
| `const [first, ...rest] = arr` | `first, *rest = items` | Works on any iterable, and the star can sit in the middle |
| `Array.from({length: 5}, (_, i) => i)` | `range(5)` | `range` is lazy: no list is built |

The next cell is this unit's setup.

In [147]:
import array
import csv
import dis
import io
import operator
import sqlite3
import sys
import timeit
from collections import namedtuple
from collections.abc import Callable, Hashable, Iterable
from typing import Any, NamedTuple


def show_size(label: str, obj: object) -> None:
    """Print the shallow size in bytes, the type and the id of an object."""
    print(f"{label}: {sys.getsizeof(obj)} bytes, type={type(obj).__name__}, id={id(obj)}")


def raised_error_name(thunk: Callable[[], object]) -> str:
    """Call thunk and return the name of the exception it raised, or "no error"."""
    try:
        thunk()
    except Exception as error:
        return type(error).__name__
    return "no error"


print(f"{sys.version_info[:2]=}")

sys.version_info[:2]=(3, 15)


### 1.5.1 Common sequence operations

Every sequence type supports indexing, slicing, `in`, `len`, `min`, `max`, `count` and
`index`; most also support `+` (concatenate) and `*` (repeat). `index(x, start)`
starts searching at `start`.

In [148]:
weekly_orders = [12, 7, 19, 7, 25]
print(f"{weekly_orders[0]=}, {weekly_orders[-1]=}, {weekly_orders[1:4]=}")
print(f"{weekly_orders[::-1]=}, {19 in weekly_orders=}, {len(weekly_orders)=}")
print(f"{weekly_orders + [30]=}, {[0] * 3=}")
print(f"{min(weekly_orders)=}, {max(weekly_orders)=}")
print(f"{weekly_orders.count(7)=}, {weekly_orders.index(7)=}, {weekly_orders.index(7, 2)=}")

weekly_orders[0]=12, weekly_orders[-1]=25, weekly_orders[1:4]=[7, 19, 7]
weekly_orders[::-1]=[25, 7, 19, 7, 12], 19 in weekly_orders=True, len(weekly_orders)=5
weekly_orders + [30]=[12, 7, 19, 7, 25, 30], [0] * 3=[0, 0, 0]
min(weekly_orders)=7, max(weekly_orders)=25
weekly_orders.count(7)=2, weekly_orders.index(7)=1, weekly_orders.index(7, 2)=3


**Support table.** The same operations across five sequence types (✓ supported,
✗ raises `TypeError`):

| Operation | `list` | `tuple` | `str` | `range` | `bytes` |
| --- | --- | --- | --- | --- | --- |
| `s[i]` | ✓ | ✓ | ✓ | ✓ | ✓ (gives `int`) |
| `s[a:b]` | ✓ | ✓ | ✓ | ✓ (gives `range`) | ✓ |
| `x in s` | ✓ | ✓ | ✓ (substring) | ✓ (O(1) for ints) | ✓ |
| `len(s)` | ✓ | ✓ | ✓ | ✓ | ✓ |
| `s + t` | ✓ | ✓ | ✓ | ✗ | ✓ |
| `s * n` | ✓ | ✓ | ✓ | ✗ | ✓ |
| `min(s)`, `max(s)` | ✓ | ✓ | ✓ | ✓ | ✓ |
| `s.count(x)` | ✓ | ✓ | ✓ | ✓ | ✓ |
| `s.index(x)` | ✓ | ✓ | ✓ | ✓ | ✓ |
| `s[i] = x` | ✓ | ✗ | ✗ | ✗ | ✗ (use `bytearray`) |

The next cell checks every row of the table by running the operation on each type.

In [149]:
def supports(operation: Callable[[Any], object], sequence: Any) -> bool:
    """Return True if operation(sequence) runs without TypeError."""
    try:
        operation(sequence)
    except TypeError:
        return False
    return True


def assign_first(sequence: Any) -> None:
    """Try to replace the first item in place."""
    sequence[0] = sequence[0]


sequence_samples = {
    "list": [3, 1, 2],
    "tuple": (3, 1, 2),
    "str": "312",
    "range": range(3),
    "bytes": b"312",
}
sequence_operations: dict[str, Callable[[Any], object]] = {
    "s[i]": lambda sequence: sequence[0],
    "s[a:b]": lambda sequence: sequence[1:],
    "x in s": lambda sequence: sequence[0] in sequence,
    "len(s)": len,
    "s + t": lambda sequence: sequence + sequence,
    "s * n": lambda sequence: sequence * 2,
    "min, max": lambda sequence: (min(sequence), max(sequence)),
    "s.count(x)": lambda sequence: sequence.count(sequence[0]),
    "s.index(x)": lambda sequence: sequence.index(sequence[0]),
    "s[i] = x": assign_first,
}
print(f"{'operation':<12}" + "".join(f"{type_name:>7}" for type_name in sequence_samples))
for operation_label, operation in sequence_operations.items():
    support_marks = [
        "✓" if supports(operation, sample) else "✗" for sample in sequence_samples.values()
    ]
    print(f"{operation_label:<12}" + "".join(f"{mark:>7}" for mark in support_marks))

operation      list  tuple    str  range  bytes
s[i]              ✓      ✓      ✓      ✓      ✓
s[a:b]            ✓      ✓      ✓      ✓      ✓
x in s            ✓      ✓      ✓      ✓      ✓
len(s)            ✓      ✓      ✓      ✓      ✓
s + t             ✓      ✓      ✓      ✗      ✓
s * n             ✓      ✓      ✓      ✗      ✓
min, max          ✓      ✓      ✓      ✓      ✓
s.count(x)        ✓      ✓      ✓      ✓      ✓
s.index(x)        ✓      ✓      ✓      ✓      ✓
s[i] = x          ✓      ✗      ✗      ✗      ✗


### 1.5.2 Lists (`list`) — internals

A CPython `list` is a **dynamic array of pointers**. The list object holds its length,
its capacity (`allocated`) and a pointer to a separate array of `PyObject*` slots;
each slot points to an item object stored elsewhere. Items are never stored inline,
which is why one list can hold any mix of types.

```mermaid
flowchart LR
    header["list object<br/>ob_size = 3<br/>allocated = 4"] --> slots["pointer array<br/>slot 0 | slot 1 | slot 2 | (spare)"]
    slots --> item_int["int 10"]
    slots --> item_str["str 'dosa'"]
    slots --> item_float["float 2.5"]
```

<details><summary>ASCII fallback</summary>

```text
list object                pointer array (capacity 4)
+----------------+        +--------+--------+--------+--------+
| ob_size = 3    |  --->  | slot 0 | slot 1 | slot 2 | spare  |
| allocated = 4  |        +---|----+---|----+---|----+--------+
+----------------+            v        v        v
                           int 10  str 'dosa' float 2.5
```

</details>

When the array is full, `append` **over-allocates**: it grows by roughly 1/8 plus a
few slots, so most appends just fill a spare slot. That is why `append` is O(1)
*amortized*. `sys.getsizeof` measures the list and its slot array, not the items.

In [150]:
growing_list: list[int] = []
previous_size = sys.getsizeof(growing_list)
print(f"len=0    size={previous_size} bytes")
for item_count in range(1, 41):
    growing_list.append(item_count)
    current_size = sys.getsizeof(growing_list)
    if current_size != previous_size:
        added_slots = (current_size - previous_size) // 8
        print(f"len={item_count:<4} size={current_size} bytes (+{added_slots} slots)")
        previous_size = current_size
print(f"{sys.getsizeof([1]) == sys.getsizeof([10**1000])=}  (items are not counted)")

len=0    size=56 bytes
len=1    size=88 bytes (+4 slots)
len=5    size=120 bytes (+4 slots)
len=9    size=184 bytes (+8 slots)
len=17   size=248 bytes (+8 slots)
len=25   size=312 bytes (+8 slots)
len=33   size=376 bytes (+8 slots)
sys.getsizeof([1]) == sys.getsizeof([10**1000])=True  (items are not counted)


**Cost of list operations** (*n* = length of the list, *k* = length of the other
iterable or slice):

| Operation | Cost | Why |
| --- | --- | --- |
| `items[i]`, `items[i] = x` | O(1) | pointer arithmetic |
| `append(x)`, `pop()` | O(1) amortized | work at the end only |
| `pop(0)`, `insert(0, x)` | O(n) | every pointer shifts by one slot |
| `x in items`, `index(x)`, `remove(x)` | O(n) | linear scan comparing with `==` |
| `extend(other)`, `items[a:b]` | O(k) | copy *k* pointers |
| `sort()`, `sorted()` | O(n log n) | Timsort (O(n) when already sorted) |

The timings make the table concrete. Absolute numbers depend on the machine; the gaps
between rows are what matter.

In [151]:
timing_cases = [
    ("items[50_000]", "O(1)"),
    ("items.append(0); items.pop()", "O(1)"),
    ("items.insert(0, 0); items.pop(0)", "O(n)"),
    ("99_999 in items", "O(n)"),
]
run_count = 1_000
for statement, complexity in timing_cases:
    elapsed_seconds = timeit.timeit(
        statement, setup="items = list(range(100_000))", number=run_count
    )
    nanoseconds_per_run = elapsed_seconds / run_count * 1e9
    print(f"{statement:<34} {complexity:<5} {nanoseconds_per_run:>12,.0f} ns per run")

items[50_000]                      O(1)            27 ns per run
items.append(0); items.pop()       O(1)            71 ns per run


items.insert(0, 0); items.pop(0)   O(n)        59,309 ns per run


99_999 in items                    O(n)     1,018,555 ns per run


### 1.5.3 List methods (complete API)

**Adding.** `append` adds one object (even a list, which becomes a nested item);
`extend` adds every item of any iterable; `insert(i, x)` puts `x` before index `i`.
`items += iterable` is `extend`: it changes the list **in place**. `items + other`
builds a **new** list and requires `other` to be a list. Mutating methods return
`None`.

In [152]:
cart_items = ["idli"]
cart_items.append("dosa")
cart_items.extend(["vada", "chutney"])
cart_items.insert(1, "coffee")
print(f"{cart_items=}")
print(f"{cart_items.append('tea')=}  (mutating methods return None)")
nested_cart = ["idli"]
nested_cart.append(["vada", "chutney"])
print(f"{nested_cart=}  (append adds the list as one item)")
original_cart = cart_items
cart_items += ("sambar",)
print(f"{cart_items is original_cart=}  (+= extends in place, accepts any iterable)")
cart_items = cart_items + ["payasam"]
print(f"{cart_items is original_cart=}  (+ builds a new list)")
print(f"list + tuple: {raised_error_name(lambda: ['idli'] + ('vada',))}")

cart_items=['idli', 'coffee', 'dosa', 'vada', 'chutney']
cart_items.append('tea')=None  (mutating methods return None)
nested_cart=['idli', ['vada', 'chutney']]  (append adds the list as one item)
cart_items is original_cart=True  (+= extends in place, accepts any iterable)
cart_items is original_cart=False  (+ builds a new list)
list + tuple: TypeError


**Removing.** `remove(x)` deletes the **first** item equal to `x` and raises
`ValueError` if there is none. `pop(i)` removes and returns the item at `i` (default:
the last one). `clear()` empties the list. `del items[i]` and `del items[a:b]` delete
by position without returning anything.

In [153]:
order_queue = ["idli", "dosa", "vada", "dosa", "upma", "poha", "khichdi"]
order_queue.remove("dosa")
print(f"{order_queue=}  (only the first 'dosa' is gone)")
try:
    order_queue.remove("pizza")
except ValueError as error:
    print(f"remove: {type(error).__name__}: {error}")
last_order = order_queue.pop()
second_order = order_queue.pop(1)
print(f"{last_order=}, {second_order=}, {order_queue=}")
del order_queue[0]
print(f"after del [0]: {order_queue}")
del order_queue[1:3]
print(f"after del [1:3]: {order_queue}")
order_queue.clear()
print(f"after clear(): {order_queue}, pop on empty: {raised_error_name(order_queue.pop)}")

order_queue=['idli', 'vada', 'dosa', 'upma', 'poha', 'khichdi']  (only the first 'dosa' is gone)
remove: ValueError: list.remove(x): x not in list
last_order='khichdi', second_order='vada', order_queue=['idli', 'dosa', 'upma', 'poha']
after del [0]: ['dosa', 'upma', 'poha']
after del [1:3]: ['dosa']
after clear(): [], pop on empty: IndexError


**Searching.** `index(x, start, end)` returns the first position of `x` within
`items[start:end]` (positions are still counted from the start of the list). `count`
counts equal items. Both compare with `==`, as does `in`.

In [154]:
daily_statuses = ["ok", "late", "ok", "failed", "late", "ok"]
print(f"{daily_statuses.index('late')=}")
print(f"{daily_statuses.index('late', 2)=}  (search from position 2)")
print(f"{daily_statuses.index('ok', 3, 6)=}")
print(f"index outside the window: {raised_error_name(lambda: daily_statuses.index('failed', 4))}")
print(f"{daily_statuses.count('ok')=}, {'failed' in daily_statuses=}")
print(f"{[1, 1.0, True].count(1)=}  (1 == 1.0 == True)")

daily_statuses.index('late')=1
daily_statuses.index('late', 2)=4  (search from position 2)
daily_statuses.index('ok', 3, 6)=5
index outside the window: ValueError
daily_statuses.count('ok')=3, 'failed' in daily_statuses=True
[1, 1.0, True].count(1)=3  (1 == 1.0 == True)


**Ordering.** `items.sort()` sorts **in place** and returns `None`; `sorted(iterable)`
returns a new list and works on any iterable. Both take `key=` and `reverse=`.
Likewise `items.reverse()` reverses in place, while `reversed(items)` returns a lazy
iterator.

In [155]:
dish_names = ["vada", "Idli", "dosa", "Upma"]
sorted_copy = sorted(dish_names)
print(f"{sorted_copy=}, {dish_names=}  (original untouched)")
print(f"{dish_names.sort(key=str.lower)=}  (sort() returns None)")
print(f"{dish_names=}  (sorted in place, case-insensitive)")
print(f"{sorted(dish_names, key=len, reverse=True)=}")
dish_names.reverse()
print(f"{dish_names=}  (reversed in place)")
backwards_iterator = reversed(dish_names)
print(f"{type(backwards_iterator).__name__=}, {list(backwards_iterator)=}")
print(f"{sorted('dosa')=}, {sorted({3, 1, 2})=}  (any iterable)")

sorted_copy=['Idli', 'Upma', 'dosa', 'vada'], dish_names=['vada', 'Idli', 'dosa', 'Upma']  (original untouched)
dish_names.sort(key=str.lower)=None  (sort() returns None)
dish_names=['dosa', 'Idli', 'Upma', 'vada']  (sorted in place, case-insensitive)
sorted(dish_names, key=len, reverse=True)=['dosa', 'Idli', 'Upma', 'vada']
dish_names=['vada', 'Upma', 'Idli', 'dosa']  (reversed in place)
type(backwards_iterator).__name__='list_reverseiterator', list(backwards_iterator)=['dosa', 'Idli', 'Upma', 'vada']
sorted('dosa')=['a', 'd', 'o', 's'], sorted({3, 1, 2})=[1, 2, 3]  (any iterable)


**Copying.** `items.copy()`, `items[:]` and `list(items)` all make a **shallow** copy:
a new list holding the *same* item objects. Changing the new list does not affect the
old one, but mutating a shared inner object shows up in both. Unit 1.7 covers deep
copies.

In [156]:
menu_rows = [["idli", 40], ["dosa", 60]]
copy_by_method = menu_rows.copy()
copy_by_slice = menu_rows[:]
copy_by_constructor = list(menu_rows)
print(f"{copy_by_method is menu_rows=}, {copy_by_slice == menu_rows=}")
copy_by_method.append(["vada", 30])
print(f"{len(menu_rows)=}, {len(copy_by_method)=}  (outer lists are independent)")
copy_by_constructor[0][1] = 45
print(f"{menu_rows[0]=}  (inner lists are shared: the original changed too)")
print(f"{copy_by_slice[0] is menu_rows[0]=}")

copy_by_method is menu_rows=False, copy_by_slice == menu_rows=True
len(menu_rows)=2, len(copy_by_method)=3  (outer lists are independent)
menu_rows[0]=['idli', 45]  (inner lists are shared: the original changed too)
copy_by_slice[0] is menu_rows[0]=True


**Slice assignment** replaces a slice with the items of any iterable, and the lengths
may differ, so the list can grow or shrink. An empty slice `items[i:i]` inserts.
**Extended slices** (with a step) need exactly as many items as they select.
**Starred unpacking** collects "the rest" into a list.

In [157]:
seat_row = ["A1", "A2", "A3", "A4", "A5"]
seat_row[1:3] = ["B2"]
print(f"{seat_row=}  (two items replaced by one)")
seat_row[1:1] = ["NEW1", "NEW2"]
print(f"{seat_row=}  (empty slice: pure insert)")
seat_row[::2] = ["x", "y", "z"]
print(f"{seat_row=}  (extended slice, same length)")
try:
    seat_row[::2] = ["only one"]
except ValueError as error:
    print(f"{type(error).__name__}: {error}")
first_seat, *other_seats = seat_row
*_, last_seat = seat_row
front_seat, *middle_seats, back_seat = seat_row
print(f"{first_seat=}, {other_seats=}, {last_seat=}")
print(f"{front_seat=}, {middle_seats=}, {back_seat=}")

seat_row=['A1', 'B2', 'A4', 'A5']  (two items replaced by one)
seat_row=['A1', 'NEW1', 'NEW2', 'B2', 'A4', 'A5']  (empty slice: pure insert)
seat_row=['x', 'NEW1', 'y', 'B2', 'z', 'A5']  (extended slice, same length)
ValueError: attempt to assign sequence of size 1 to extended slice of size 3
first_seat='x', other_seats=['NEW1', 'y', 'B2', 'z', 'A5'], last_seat='A5'
front_seat='x', middle_seats=['NEW1', 'y', 'B2', 'z'], back_seat='A5'


### 1.5.4 Sorting in depth

Python sorts with **Timsort**, a merge sort that finds existing ordered runs, so
nearly sorted data costs close to O(n). (Since 3.11 CPython merges runs with the
*powersort* strategy.) The sort is **stable**: items with equal keys keep their
original relative order.

Stability enables **multi-pass sorting**: sort by the least important key first, then
by more important keys. Each pass keeps the order the earlier passes produced among
ties.

In [158]:
customer_orders = [("Ravi", "Pune"), ("Asha", "Delhi"), ("Meena", "Pune"), ("John", "Delhi")]
print(f"{sorted(customer_orders, key=lambda order: order[1])=}")
print("  ^ within each city the original order (Ravi before Meena) is kept")
by_name = sorted(customer_orders, key=lambda order: order[0])
by_city_then_name = sorted(by_name, key=lambda order: order[1])
print(f"{by_city_then_name=}")

sorted(customer_orders, key=lambda order: order[1])=[('Asha', 'Delhi'), ('John', 'Delhi'), ('Ravi', 'Pune'), ('Meena', 'Pune')]
  ^ within each city the original order (Ravi before Meena) is kept
by_city_then_name=[('Asha', 'Delhi'), ('John', 'Delhi'), ('Meena', 'Pune'), ('Ravi', 'Pune')]


**Key functions.** `key` is called once per item, and items are compared by the key's
result:

- `key=str.lower` for case-insensitive order.
- `operator.itemgetter(1)` reads `item[1]`; `itemgetter(2, 0)` returns a tuple of both.
- `operator.attrgetter("price")` reads `item.price` (works on any object with
  attributes).
- **Tuple keys** compare element by element. For mixed directions, negate numeric
  parts: `key=lambda p: (-p.rating, p.price)` means rating descending, price ascending.
  Strings cannot be negated, so use multi-pass sorting with `reverse=True` for them.

The products below use `NamedTuple`, covered in 1.5.6; here it is just a record with
attributes.

In [159]:
class Product(NamedTuple):
    """A product in a catalogue."""

    name: str
    price: int
    rating: float


catalogue = [
    Product("pen", 20, 4.2),
    Product("Notebook", 60, 4.8),
    Product("eraser", 5, 4.2),
    Product("Bag", 900, 4.8),
]
print(f"{sorted(['pen', 'Notebook', 'eraser'])=}")
print(f"{sorted(['pen', 'Notebook', 'eraser'], key=str.lower)=}")
print([product.name for product in sorted(catalogue, key=operator.itemgetter(1))])
print(sorted(catalogue, key=operator.itemgetter(2, 0))[:2])
print([product.name for product in sorted(catalogue, key=operator.attrgetter("rating"))])
best_then_cheapest = sorted(catalogue, key=lambda product: (-product.rating, product.price))
print([(product.name, product.rating, product.price) for product in best_then_cheapest])

sorted(['pen', 'Notebook', 'eraser'])=['Notebook', 'eraser', 'pen']
sorted(['pen', 'Notebook', 'eraser'], key=str.lower)=['eraser', 'Notebook', 'pen']
['eraser', 'pen', 'Notebook', 'Bag']
[Product(name='eraser', price=5, rating=4.2), Product(name='pen', price=20, rating=4.2)]
['pen', 'eraser', 'Notebook', 'Bag']
[('Notebook', 4.8, 60), ('Bag', 4.8, 900), ('eraser', 4.2, 5), ('pen', 4.2, 20)]


**JS bridge.** JavaScript's default `sort()` converts items to strings, so
`[10, 9, 1, 100].sort()` gives `[1, 10, 100, 9]`. Python compares the values
themselves, and refuses to compare unrelated types instead of guessing.

In [160]:
print(f"{sorted([10, 9, 1, 100])=}")
print(f"{sorted(['10', '9', '1', '100'])=}  (what JS does to numbers)")
try:
    sorted([3, "2", 1])
except TypeError as error:
    print(f"{type(error).__name__}: {error}")
print(f"{sorted([3, '2', 1], key=str)=}  (an explicit key makes it work)")

sorted([10, 9, 1, 100])=[1, 9, 10, 100]
sorted(['10', '9', '1', '100'])=['1', '10', '100', '9']  (what JS does to numbers)
TypeError: '<' not supported between instances of 'str' and 'int'
sorted([3, '2', 1], key=str)=[1, '2', 3]  (an explicit key makes it work)


### 1.5.5 Tuples (`tuple`)

**The comma makes the tuple**, not the parentheses: `(42)` is just the int 42, while
`42,` and `(42,)` are one-item tuples. `()` is the empty tuple. Writing several values
separated by commas **packs** them; assigning a tuple to several names **unpacks** it.
Unpacking can be nested and starred, and `a, b = b, a` swaps without a temporary.

> **Note:** the two `# fmt: skip` comments stop the formatter from rewriting `(42)`
> as `42` and `42,` as `(42,)`, which would hide the point.

In [161]:
not_a_tuple = (42)  # fmt: skip
one_item_tuple = 42,  # fmt: skip
also_one_item = (42,)
empty_tuple = ()
print(f"{type(not_a_tuple).__name__=}, {one_item_tuple=}, {also_one_item=}, {empty_tuple=}")
office_location = 12.97, 77.59
latitude, longitude = office_location
print(f"{office_location=}, {latitude=}, {longitude=}")
city_name, (city_latitude, city_longitude) = ("Bengaluru", (12.97, 77.59))
print(f"{city_name=}, {city_latitude=}, {city_longitude=}")
top_scorer, *other_scorers = ("Asha", "Ravi", "Meena")
print(f"{top_scorer=}, {other_scorers=}  (the starred name is always a list)")
home_team, away_team = "India", "Australia"
home_team, away_team = away_team, home_team
print(f"{home_team=}, {away_team=}")

type(not_a_tuple).__name__='int', one_item_tuple=(42,), also_one_item=(42,), empty_tuple=()
office_location=(12.97, 77.59), latitude=12.97, longitude=77.59
city_name='Bengaluru', city_latitude=12.97, city_longitude=77.59
top_scorer='Asha', other_scorers=['Ravi', 'Meena']  (the starred name is always a list)
home_team='Australia', away_team='India'


**Methods and hashing.** Tuples have just two methods, `count` and `index`. A tuple is
**hashable only if all its items are**, so a tuple of strings and numbers can be a
dict key or a set member, but a tuple holding a list cannot.

In [162]:
match_ratings = (5, 4, 5, 3, 5)
print(f"{match_ratings.count(5)=}, {match_ratings.index(3)=}")
print(f"{hash((1, 'a', (2, 3))) == hash((1, 'a', (2, 3)))=}")
try:
    hash((1, [2, 3]))
except TypeError as error:
    print(f"{type(error).__name__}: {error}")
sales_by_country_and_product = {("IN", "dosa"): 1200, ("US", "dosa"): 150}
print(f"{sales_by_country_and_product[('IN', 'dosa')]=}")
print(f"{isinstance((1, 'a'), Hashable)=}, {isinstance((1, []), Hashable)=}")
print("  ^ the Hashable ABC checks the type only; hash() checks the items")

match_ratings.count(5)=3, match_ratings.index(3)=3
hash((1, 'a', (2, 3))) == hash((1, 'a', (2, 3)))=True
TypeError: unhashable type: 'list'
sales_by_country_and_product[('IN', 'dosa')]=1200
isinstance((1, 'a'), Hashable)=True, isinstance((1, []), Hashable)=True
  ^ the Hashable ABC checks the type only; hash() checks the items


#### Under the hood: tuples are smaller and built once

A tuple stores its item pointers inline, with no spare capacity and no separate array,
so it is smaller than the equivalent list. A tuple of constants is built once at
compile time: `dis` shows a single `LOAD_CONST`, while the list is rebuilt on every
run. (Bytecode names vary by Python version.)

In [163]:
show_size("tuple (1, 2, 3)", (1, 2, 3))
show_size("list  [1, 2, 3]", [1, 2, 3])
appended_list = []
for number in (1, 2, 3):
    appended_list.append(number)
show_size("list built by append", appended_list)
dis.dis(compile("rgb_tuple = (255, 153, 51)", "<tuple>", "exec"))
dis.dis(compile("rgb_list = [255, 153, 51]", "<list>", "exec"))

tuple (1, 2, 3): 72 bytes, type=tuple, id=139669135427120
list  [1, 2, 3]: 88 bytes, type=list, id=139669135542336
list built by append: 88 bytes, type=list, id=139669155131456
  0           RESUME                   0

  1           LOAD_CONST               1 ((255, 153, 51))
              STORE_NAME               0 (rgb_tuple)
              LOAD_COMMON_CONSTANT     7 (None)
              RETURN_VALUE
  0           RESUME                   0

  1           BUILD_LIST               0
              LOAD_CONST               1 ((255, 153, 51))
              LIST_EXTEND              1
              STORE_NAME               0 (rgb_list)
              LOAD_COMMON_CONSTANT     7 (None)
              RETURN_VALUE


#### Predict the output: the `t[0] += [1]` puzzle

**Predict the output.**

```python
pair_of_lists = ([1, 2], "x")
try:
    pair_of_lists[0] += [3]
except TypeError as error:
    print(type(error).__name__)
print(pair_of_lists)
```

<details><summary>Answer</summary>

`TypeError`, then `([1, 2, 3], 'x')`. The statement **both** raises and mutates.
`pair_of_lists[0] += [3]` first runs `list.__iadd__`, which extends the inner list in
place and succeeds; then it tries to store the result back into slot 0 of the tuple,
which fails because tuples do not support item assignment.

</details>

Run the next cell to check your prediction.

In [164]:
pair_of_lists = ([1, 2], "x")
try:
    pair_of_lists[0] += [3]
except TypeError as error:
    print(type(error).__name__)
print(pair_of_lists)

TypeError
([1, 2, 3], 'x')


The bytecode shows the two steps: an in-place add (`BINARY_OP` with `+=`) followed by
`STORE_SUBSCR`, and the error comes from the store.

In [165]:
dis.dis(compile("pair_of_lists[0] += [3]", "<puzzle>", "exec"))

  0           RESUME                   0

  1           LOAD_NAME                0 (pair_of_lists)
              LOAD_SMALL_INT           0
              COPY                     2
              COPY                     2
              BINARY_OP               26 ([])
              LOAD_SMALL_INT           3
              BUILD_LIST               1
              BINARY_OP               13 (+=)
              SWAP                     3
              SWAP                     2
              STORE_SUBSCR
              LOAD_COMMON_CONSTANT     7 (None)
              RETURN_VALUE


### 1.5.6 Named tuples

A **named tuple** is a tuple subclass whose items also have names. It is still
immutable, still unpacks, and costs no more memory than a plain tuple.
`collections.namedtuple` builds one from a field list:

| API | What it does |
| --- | --- |
| `_fields` | tuple of field names |
| `_make(iterable)` | build an instance from any iterable (a CSV row, a DB row) |
| `_asdict()` | a `dict` of field name → value |
| `_replace(**changes)` | a modified **copy** (the original is unchanged) |

The leading underscore avoids clashes with field names; these methods are public.

In [166]:
GeoPoint = namedtuple("GeoPoint", ["latitude", "longitude"])
office_point = GeoPoint(12.97, 77.59)
print(f"{office_point=}, {office_point.latitude=}, {office_point[1]=}")
point_latitude, point_longitude = office_point
print(f"{GeoPoint._fields=}")
print(f"{GeoPoint._make(['28.61', '77.21'])=}  (values are not converted)")
print(f"{office_point._asdict()=}")
moved_point = office_point._replace(longitude=77.60)
print(f"{moved_point=}, {office_point=}  (original unchanged)")
try:
    office_point.latitude = 0.0
except AttributeError as error:
    print(f"{type(error).__name__}: {error}")

office_point=GeoPoint(latitude=12.97, longitude=77.59), office_point.latitude=12.97, office_point[1]=77.59
GeoPoint._fields=('latitude', 'longitude')
GeoPoint._make(['28.61', '77.21'])=GeoPoint(latitude='28.61', longitude='77.21')  (values are not converted)
office_point._asdict()={'latitude': 12.97, 'longitude': 77.59}
moved_point=GeoPoint(latitude=12.97, longitude=77.6), office_point=GeoPoint(latitude=12.97, longitude=77.59)  (original unchanged)
AttributeError: can't set attribute


**`typing.NamedTuple`** is the class syntax for the same thing: type-annotated fields,
**defaults** (fields with defaults must come last), a docstring and **methods**.
Prefer it in new code. `LogEntry` comes back in 1.5.11 and in the Phase 1 project.

In [167]:
class LogEntry(NamedTuple):
    """One parsed line of an access log."""

    level: str
    message: str
    status_code: int = 200

    def is_error(self) -> bool:
        """Return True for ERROR entries and 5xx status codes."""
        return self.level == "ERROR" or self.status_code >= 500


placed_entry = LogEntry("INFO", "order placed")
failed_entry = LogEntry("WARNING", "gateway timeout", 504)
print(f"{placed_entry=}, {placed_entry.is_error()=}")
print(f"{failed_entry.is_error()=}, {LogEntry._field_defaults=}")
print(f"{isinstance(placed_entry, tuple)=}, {placed_entry == ('INFO', 'order placed', 200)=}")

placed_entry=LogEntry(level='INFO', message='order placed', status_code=200), placed_entry.is_error()=False
failed_entry.is_error()=True, LogEntry._field_defaults={'status_code': 200}
isinstance(placed_entry, tuple)=True, placed_entry == ('INFO', 'order placed', 200)=True


### 1.5.7 Ranges (`range`)

A `range` is a **lazy, immutable sequence** of integers. It stores only `start`,
`stop` and `step`, so its size never depends on its length, and `len`, indexing,
slicing and `in` (for ints) are computed arithmetically in **O(1)**. Negative steps
count down; `stop` is always excluded.

In [168]:
page_offsets = range(0, 95, 20)
print(f"{page_offsets=}, {list(page_offsets)=}")
print(f"{list(range(10, 0, -3))=}, {list(range(5, 0, -1))=}")
huge_range = range(0, 10**18, 7)
print(f"{len(huge_range)=:,}")
print(f"{7 * 10**15 in huge_range=}, {7 * 10**15 + 1 in huge_range=}")
print(f"{huge_range[10**12]=:,}, {huge_range[-1]=:,}")
print(f"{huge_range[::2]=}  (slicing returns another range)")
print(f"{sys.getsizeof(huge_range)=} bytes, {sys.getsizeof(range(3))=} bytes")
try:
    page_offsets[0] = 5
except TypeError as error:
    print(f"{type(error).__name__}: {error}")

page_offsets=range(0, 95, 20), list(page_offsets)=[0, 20, 40, 60, 80]
list(range(10, 0, -3))=[10, 7, 4, 1], list(range(5, 0, -1))=[5, 4, 3, 2, 1]
len(huge_range)=142,857,142,857,142,858
7 * 10**15 in huge_range=True, 7 * 10**15 + 1 in huge_range=False
huge_range[10**12]=7,000,000,000,000, huge_range[-1]=999,999,999,999,999,999
huge_range[::2]=range(0, 1000000000000000006, 14)  (slicing returns another range)
sys.getsizeof(huge_range)=48 bytes, sys.getsizeof(range(3))=48 bytes
TypeError: 'range' object does not support item assignment


### 1.5.8 Arrays (`array.array`)

A list of floats stores a pointer per item **plus** a separate 24-byte `float` object
per item. `array.array` stores raw machine values of one type (type code `"d"` =
C double, 8 bytes) in one block. Use it for long numeric series when memory matters
(NumPy, covered later, does the same and adds fast math).

In [169]:
reading_count = 100_000
readings_list = [reading_index * 0.5 for reading_index in range(reading_count)]
readings_array = array.array("d", readings_list)
list_total_bytes = sys.getsizeof(readings_list) + sum(
    sys.getsizeof(reading) for reading in readings_list
)
array_total_bytes = sys.getsizeof(readings_array)
print(f"list of floats: {list_total_bytes:>10,} bytes (list + float objects)")
print(f"array('d'):     {array_total_bytes:>10,} bytes")
print(f"ratio: {list_total_bytes / array_total_bytes:.1f}x")
print(f"{readings_array.typecode=}, {readings_array.itemsize=}, {readings_array[:3]=}")
print(f"append a str: {raised_error_name(lambda: readings_array.append('x'))}")

list of floats:  3,200,984 bytes (list + float objects)
array('d'):        800,080 bytes
ratio: 4.0x
readings_array.typecode='d', readings_array.itemsize=8, readings_array[:3]=array('d', [0.0, 0.5, 1.0])
append a str: TypeError


### 1.5.9 Method reference — `list`

Every row gets a demo cell below.

| Method / operation | What it does | Returns | Cost |
| --- | --- | --- | --- |
| `append(x)` | add one item at the end | `None` | O(1) amortized |
| `extend(iterable)` / `+= iterable` | add every item of an iterable | `None` / same list | O(k) |
| `insert(i, x)` | insert before index `i` | `None` | O(n) |
| `remove(x)` | delete the first item equal to `x` (`ValueError` if absent) | `None` | O(n) |
| `pop()` / `pop(i)` | remove and return the last / i-th item (`IndexError` if empty) | item | O(1) / O(n) |
| `clear()` | remove every item | `None` | O(n) |
| `index(x[, start[, end]])` | position of the first `x` (`ValueError` if absent) | `int` | O(n) |
| `count(x)` | number of items equal to `x` | `int` | O(n) |
| `sort(*, key=None, reverse=False)` | stable in-place sort | `None` | O(n log n) |
| `reverse()` | reverse in place | `None` | O(n) |
| `copy()` | shallow copy (same as `[:]`, `list(x)`) | new `list` | O(n) |
| `items[i]`, `items[i] = x` | read / replace by index (negative indexes allowed) | item / — | O(1) |
| `items[a:b:c]`, `items[a:b] = iterable` | slice copy / slice replacement | new `list` / — | O(k) |
| `del items[i]`, `del items[a:b]` | delete by index or slice | — | O(n) |
| `x in items` | membership by `==` | `bool` | O(n) |
| `items + other`, `items * n` | concatenate / repeat into a new list | new `list` | O(n + m) |
| `len`, `min`, `max`, `sum`, `sorted`, `reversed`, `enumerate`, `any`, `all` | built-ins that work on lists | varies | O(n) |
| `==`, `<` | element-wise, lexicographic comparison | `bool` | O(n) |

#### `list`

**`append(x)`**: one item at the end; the call returns `None`.

In [170]:
shopping_list = ["rice"]
append_result = shopping_list.append("dal")
print(f"{shopping_list=}, {append_result=}")

shopping_list=['rice', 'dal'], append_result=None


**`extend(iterable)` / `+= iterable`**: every item of any iterable, in place.

In [171]:
shopping_list = ["rice"]
shopping_list.extend(("dal", "ghee"))
list_before_augmented_add = shopping_list
shopping_list += "ab"
print(f"{shopping_list=}  (a str is an iterable of characters)")
print(f"{shopping_list is list_before_augmented_add=}")

shopping_list=['rice', 'dal', 'ghee', 'a', 'b']  (a str is an iterable of characters)
shopping_list is list_before_augmented_add=True


**`insert(i, x)`**: before index `i`; an index past the end appends.

In [172]:
shopping_list = ["rice", "ghee"]
shopping_list.insert(1, "dal")
shopping_list.insert(0, "salt")
shopping_list.insert(100, "sugar")
shopping_list.insert(-1, "tea")
print(f"{shopping_list=}")

shopping_list=['salt', 'rice', 'dal', 'ghee', 'tea', 'sugar']


**`remove(x)`**: the first equal item; `ValueError` if absent.

In [173]:
shopping_list = ["rice", "dal", "rice"]
shopping_list.remove("rice")
print(f"{shopping_list=}")
print(f"remove missing: {raised_error_name(lambda: shopping_list.remove('oil'))}")

shopping_list=['dal', 'rice']
remove missing: ValueError


**`pop()` / `pop(i)`**: remove and return the last or the i-th item.

In [174]:
shopping_list = ["rice", "dal", "ghee", "salt"]
print(f"{shopping_list.pop()=}, {shopping_list.pop(0)=}, {shopping_list=}")
print(f"pop from empty: {raised_error_name(lambda: [].pop())}")
print(f"pop bad index: {raised_error_name(lambda: ['rice'].pop(5))}")

shopping_list.pop()='salt', shopping_list.pop(0)='rice', shopping_list=['dal', 'ghee']
pop from empty: IndexError
pop bad index: IndexError


**`clear()`**: empties the list in place, so every alias sees the change.

In [175]:
shopping_list = ["rice", "dal"]
same_list_alias = shopping_list
shopping_list.clear()
print(f"{shopping_list=}, {same_list_alias=}")

shopping_list=[], same_list_alias=[]


**`index(x[, start[, end]])`**: position of the first match in the window.

In [176]:
shopping_list = ["rice", "dal", "ghee", "dal"]
print(f"{shopping_list.index('dal')=}, {shopping_list.index('dal', 2)=}")
print(f"not in window: {raised_error_name(lambda: shopping_list.index('rice', 1, 3))}")

shopping_list.index('dal')=1, shopping_list.index('dal', 2)=3
not in window: ValueError


**`count(x)`**: how many items are `==` to `x`.

In [177]:
shopping_list = ["rice", "dal", "rice", "Rice"]
print(f"{shopping_list.count('rice')=}, {shopping_list.count('oil')=}")

shopping_list.count('rice')=2, shopping_list.count('oil')=0


**`sort(*, key=None, reverse=False)`**: stable, in place, keyword-only arguments.

In [178]:
shopping_list = ["ghee", "Rice", "dal", "salt"]
shopping_list.sort()
print(f"{shopping_list=}  (uppercase first)")
shopping_list.sort(key=str.lower, reverse=True)
print(f"{shopping_list=}")
print(f"positional key: {raised_error_name(lambda: shopping_list.sort(str.lower))}")

shopping_list=['Rice', 'dal', 'ghee', 'salt']  (uppercase first)
shopping_list=['salt', 'Rice', 'ghee', 'dal']
positional key: TypeError


**`reverse()`**: in place, returns `None`.

In [179]:
shopping_list = ["rice", "dal", "ghee"]
print(f"{shopping_list.reverse()=}, {shopping_list=}")

shopping_list.reverse()=None, shopping_list=['ghee', 'dal', 'rice']


**`copy()`**: a new list holding the same item objects.

In [180]:
shopping_list = ["rice", ["dal", "ghee"]]
copied_list = shopping_list.copy()
print(f"{copied_list == shopping_list=}, {copied_list is shopping_list=}")
print(f"{copied_list[1] is shopping_list[1]=}  (shallow)")

copied_list == shopping_list=True, copied_list is shopping_list=False
copied_list[1] is shopping_list[1]=True  (shallow)


**`items[i]`, `items[i] = x`**: read or replace one slot; negative indexes count from the end.

In [181]:
shopping_list = ["rice", "dal", "ghee"]
print(f"{shopping_list[0]=}, {shopping_list[-1]=}")
shopping_list[-1] = "butter"
print(f"{shopping_list=}")
print(f"assign past the end: {raised_error_name(lambda: shopping_list.__setitem__(5, 'x'))}")

shopping_list[0]='rice', shopping_list[-1]='ghee'
shopping_list=['rice', 'dal', 'butter']
assign past the end: IndexError


**`items[a:b:c]`, `items[a:b] = iterable`**: copy a slice, or replace one.

In [182]:
shopping_list = ["rice", "dal", "ghee", "salt", "tea"]
print(f"{shopping_list[1:4]=}, {shopping_list[::2]=}, {shopping_list[::-1]=}")
shopping_list[1:3] = ("moong dal", "toor dal", "urad dal")
print(f"{shopping_list=}")

shopping_list[1:4]=['dal', 'ghee', 'salt'], shopping_list[::2]=['rice', 'ghee', 'tea'], shopping_list[::-1]=['tea', 'salt', 'ghee', 'dal', 'rice']
shopping_list=['rice', 'moong dal', 'toor dal', 'urad dal', 'salt', 'tea']


**`del items[i]`, `del items[a:b]`**: delete by position or slice (an extended slice works too).

In [183]:
shopping_list = ["rice", "dal", "ghee", "salt", "tea", "jam"]
del shopping_list[0]
del shopping_list[-2:]
print(f"{shopping_list=}")
del shopping_list[::2]
print(f"{shopping_list=}")

shopping_list=['dal', 'ghee', 'salt']
shopping_list=['ghee']


**`x in items`**: a linear scan with `==`. CPython checks identity first, so an object
is always found in a list that holds it, even `nan`, which is not equal to itself.

In [184]:
shopping_list = ["rice", "dal"]
print(f"{'dal' in shopping_list=}, {'oil' not in shopping_list=}")
not_a_number = float("nan")
print(f"{not_a_number == not_a_number=}, {not_a_number in [not_a_number]=}")
print(f"{float('nan') in [float('nan')]=}  (two different nan objects)")

'dal' in shopping_list=True, 'oil' not in shopping_list=True
not_a_number == not_a_number=False, not_a_number in [not_a_number]=True
float('nan') in [float('nan')]=False  (two different nan objects)


**`items + other`, `items * n`**: new lists. Repetition copies **references**, which
matters when the item is mutable (see Lab 1.5, exercise 4).

In [185]:
breakfast_list = ["idli", "dosa"]
print(f"{breakfast_list + ['vada']=}, {breakfast_list * 2=}, {breakfast_list=}")
repeated_rows = [[]] * 3
repeated_rows[0].append("x")
print(f"{repeated_rows=}  (three references to one inner list)")

breakfast_list + ['vada']=['idli', 'dosa', 'vada'], breakfast_list * 2=['idli', 'dosa', 'idli', 'dosa'], breakfast_list=['idli', 'dosa']
repeated_rows=[['x'], ['x'], ['x']]  (three references to one inner list)


**Built-ins that work on lists**: `len`, `min`, `max`, `sum`, `sorted`, `reversed`, `enumerate`, `any`, `all`.

In [186]:
daily_sales = [1200, 950, 0, 1800, 1430]
print(f"{len(daily_sales)=}, {min(daily_sales)=}, {max(daily_sales)=}, {sum(daily_sales)=}")
print(f"{sorted(daily_sales)=}, {list(reversed(daily_sales))=}")
for day_number, sales_amount in enumerate(daily_sales, start=1):
    print(f"day {day_number}: ₹{sales_amount:,}")
print(f"{any(sale == 0 for sale in daily_sales)=}, {all(daily_sales)=}")
print(f"{max([], default=0)=}  (default avoids ValueError on empty input)")

len(daily_sales)=5, min(daily_sales)=0, max(daily_sales)=1800, sum(daily_sales)=5380
sorted(daily_sales)=[0, 950, 1200, 1430, 1800], list(reversed(daily_sales))=[1430, 1800, 0, 950, 1200]
day 1: ₹1,200
day 2: ₹950
day 3: ₹0
day 4: ₹1,800
day 5: ₹1,430
any(sale == 0 for sale in daily_sales)=True, all(daily_sales)=False
max([], default=0)=0  (default avoids ValueError on empty input)


**`==`, `<`**: compare item by item; the first difference decides, and a shorter list
that is a prefix of a longer one is smaller.

In [187]:
print(f"{[1, 2, 3] == [1, 2, 3]=}, {[1, 2] == [1.0, 2.0]=}")
print(f"{[1, 2, 3] < [1, 3]=}  (decided at index 1)")
print(f"{[1, 2] < [1, 2, 0]=}  (prefix is smaller)")
print(f"{['b'] > ['a', 'z']=}")
print(f"mixed types: {raised_error_name(lambda: [1, 'a'] < [1, 2])}")

[1, 2, 3] == [1, 2, 3]=True, [1, 2] == [1.0, 2.0]=True
[1, 2, 3] < [1, 3]=True  (decided at index 1)
[1, 2] < [1, 2, 0]=True  (prefix is smaller)
['b'] > ['a', 'z']=True
mixed types: TypeError


### 1.5.10 Method reference — `tuple`, `namedtuple` and `range`

| Type | Method / operation | What it does |
| --- | --- | --- |
| `tuple` | `count(x)`, `index(x[, start[, end]])` | the only two methods — tuples are immutable |
| `tuple` | indexing, slicing, `in`, `+`, `*`, `len`, comparison | same semantics as lists, but always returns new tuples |
| `tuple` | `hash(t)` | works when every item is hashable — enables dict keys and set members |
| `tuple` | unpacking `a, b = t`, `first, *rest = t` | the most common way to read tuples |
| `namedtuple` | `_make(iterable)`, `_asdict()`, `_replace(**changes)`, `_fields`, `_field_defaults` | construction, conversion, "modified copy", introspection |
| `range` | `start`, `stop`, `step`, `count(x)`, `index(x)` | attributes and O(1) arithmetic lookups |

#### `tuple`

**`count(x)`, `index(x[, start[, end]])`**: the only two methods.

In [188]:
weekday_codes = ("Mon", "Tue", "Wed", "Tue", "Fri")
tuple_method_names = [name for name in dir(tuple) if not name.startswith("_")]
print(f"{tuple_method_names=}")
print(f"{weekday_codes.count('Tue')=}, {weekday_codes.index('Tue', 2)=}")
print(f"missing: {raised_error_name(lambda: weekday_codes.index('Sun'))}")

tuple_method_names=['count', 'index']
weekday_codes.count('Tue')=2, weekday_codes.index('Tue', 2)=3
missing: ValueError


**Indexing, slicing, `in`, `+`, `*`, `len`, comparison**: like lists, but every result is a new tuple.

In [189]:
rgb_colour = (255, 153, 51)
print(f"{rgb_colour[0]=}, {rgb_colour[-2:]=}, {153 in rgb_colour=}, {len(rgb_colour)=}")
print(f"{rgb_colour + (128,)=}, {(0,) * 3=}")
print(f"{(1, 2, 3) < (1, 3)=}, {(1, 2) == (1, 2)=}, {(1, 2) == [1, 2]=}")

rgb_colour[0]=255, rgb_colour[-2:]=(153, 51), 153 in rgb_colour=True, len(rgb_colour)=3
rgb_colour + (128,)=(255, 153, 51, 128), (0,) * 3=(0, 0, 0)
(1, 2, 3) < (1, 3)=True, (1, 2) == (1, 2)=True, (1, 2) == [1, 2]=False


**`hash(t)`**: works only when every item is hashable.

In [190]:
route_distances = {("Delhi", "Agra"): 233, ("Mumbai", "Pune"): 148}
print(f"{route_distances[('Delhi', 'Agra')]=}")
visited_cells = {(0, 0), (0, 1), (0, 0)}
print(f"{visited_cells=}")
print(f"tuple holding a list: {raised_error_name(lambda: hash(('Delhi', ['Agra'])))}")

route_distances[('Delhi', 'Agra')]=233
visited_cells={(0, 1), (0, 0)}
tuple holding a list: TypeError


**Unpacking `a, b = t`, `first, *rest = t`**: the most common way to read a tuple; the counts must match.

In [191]:
delivery_record = ("A-17", "Asha", "Pune", 450)
order_id, customer_name, delivery_city, amount_due = delivery_record
first_field, *remaining_fields = delivery_record
print(f"{order_id=}, {amount_due=}, {first_field=}, {remaining_fields=}")
try:
    order_id, customer_name = delivery_record
except ValueError as error:
    print(f"{type(error).__name__}: {error}")

order_id='A-17', amount_due=450, first_field='A-17', remaining_fields=['Asha', 'Pune', 450]
ValueError: too many values to unpack (expected 2, got 4)


#### `namedtuple`

**`_make`, `_asdict`, `_replace`, `_fields`, `_field_defaults`**: construction,
conversion, modified copy and introspection. `defaults=` applies to the *rightmost*
fields.

In [192]:
OrderLine = namedtuple("OrderLine", ["item", "quantity", "unit_price"], defaults=[1, 0])
csv_line = OrderLine._make(["dosa", 2, 60])
print(f"{csv_line=}")
print(f"{csv_line._asdict()=}")
print(f"{csv_line._replace(quantity=3)=}")
print(f"{OrderLine._fields=}, {OrderLine._field_defaults=}")
print(f"{OrderLine('tea')=}  (defaults filled in)")

csv_line=OrderLine(item='dosa', quantity=2, unit_price=60)
csv_line._asdict()={'item': 'dosa', 'quantity': 2, 'unit_price': 60}
csv_line._replace(quantity=3)=OrderLine(item='dosa', quantity=3, unit_price=60)
OrderLine._fields=('item', 'quantity', 'unit_price'), OrderLine._field_defaults={'quantity': 1, 'unit_price': 0}
OrderLine('tea')=OrderLine(item='tea', quantity=1, unit_price=0)  (defaults filled in)


#### `range`

**`start`, `stop`, `step`, `count(x)`, `index(x)`**: the three attributes, plus
lookups computed arithmetically.

In [193]:
even_hours = range(0, 24, 2)
print(f"{even_hours.start=}, {even_hours.stop=}, {even_hours.step=}")
print(f"{even_hours.count(14)=}, {even_hours.count(15)=}, {even_hours.index(14)=}")
print(f"{range(10**18).index(10**17)=:,}  (instant: no scan)")
print(f"not in range: {raised_error_name(lambda: even_hours.index(15))}")

even_hours.start=0, even_hours.stop=24, even_hours.step=2
even_hours.count(14)=1, even_hours.count(15)=0, even_hours.index(14)=7
range(10**18).index(10**17)=100,000,000,000,000,000  (instant: no scan)
not in range: ValueError


### 1.5.11 Real-world use cases

**`list`**: an ordered, changing collection: shopping-cart items, a page of API
results, rows read from a CSV file, or a stack (`append` + `pop`). Ordered processing
queues of moderate size also fit, as long as you take items from the end.

In [194]:
undo_stack: list[str] = []
for editor_action in ["type 'a'", "type 'b'", "bold"]:
    undo_stack.append(editor_action)
print(f"undo {undo_stack.pop()!r}; remaining stack: {undo_stack}")
api_page = [{"id": 1, "item": "idli"}, {"id": 2, "item": "dosa"}]
print(f"{[result['item'] for result in api_page]=}")
csv_text = "id,item,price\n1,idli,40\n2,dosa,60\n"
header_row, *data_rows = csv.reader(io.StringIO(csv_text))
print(f"{header_row=}, {data_rows=}")

undo 'bold'; remaining stack: ["type 'a'", "type 'b'"]
[result['item'] for result in api_page]=['idli', 'dosa']
header_row=['id', 'item', 'price'], data_rows=[['1', 'idli', '40'], ['2', 'dosa', '60']]


**`tuple`**: a fixed record whose positions mean something: database rows (the
`sqlite3` module returns tuples), `(latitude, longitude)`, several return values from
one function, composite dict keys like `(country, product)`, and constants that must
not change.

In [195]:
WEEKDAY_NAMES = ("Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun")


def lowest_and_highest(prices: list[int]) -> tuple[int, int]:
    """Return the lowest and highest price as one tuple."""
    return min(prices), max(prices)


cheapest_price, dearest_price = lowest_and_highest([40, 60, 35, 120])
print(f"{cheapest_price=}, {dearest_price=}, {WEEKDAY_NAMES[4]=}")
connection = sqlite3.connect(":memory:")
connection.execute("CREATE TABLE stores (city TEXT, latitude REAL, longitude REAL)")
connection.execute("INSERT INTO stores VALUES ('Pune', 18.52, 73.86)")
store_row = connection.execute("SELECT * FROM stores").fetchone()
connection.close()
print(f"{store_row=}, {type(store_row).__name__=}")
units_sold = {("IN", "dosa"): 1200, ("IN", "idli"): 800, ("US", "dosa"): 150}
print(f"{sum(count for (country, _), count in units_sold.items() if country == 'IN')=}")

cheapest_price=35, dearest_price=120, WEEKDAY_NAMES[4]='Fri'


store_row=('Pune', 18.52, 73.86), type(store_row).__name__='tuple'
sum(count for (country, _), count in units_sold.items() if country == 'IN')=2000


**`namedtuple` / `NamedTuple`**: a lightweight read-only record with field names,
such as a parsed log line or a CSV row. This reuses `LogEntry` from 1.5.6.

In [196]:
raw_log_lines = [
    "INFO|order placed|201",
    "ERROR|payment failed|502",
    "WARNING|slow response|200",
]
parsed_entries = []
for raw_log_line in raw_log_lines:
    level, message, status_text = raw_log_line.split("|")
    parsed_entries.append(LogEntry(level, message, int(status_text)))
print([entry.message for entry in parsed_entries if entry.is_error()])
print(f"{parsed_entries[0]._asdict()=}")

['payment failed']
parsed_entries[0]._asdict()={'level': 'INFO', 'message': 'order placed', 'status_code': 201}


**`range`**: index loops, pagination offsets (`range(0, total, page_size)`) and
membership checks on numeric intervals.

In [197]:
total_results, page_size = 95, 20
for page_number, page_offset in enumerate(range(0, total_results, page_size), start=1):
    page_end = min(page_offset + page_size, total_results)
    print(f"page {page_number}: results {page_offset + 1}-{page_end}")
working_age = range(18, 61)
print(f"{25 in working_age=}, {61 in working_age=}, {17.5 in working_age=}")

page 1: results 1-20
page 2: results 21-40
page 3: results 41-60
page 4: results 61-80
page 5: results 81-95
25 in working_age=True, 61 in working_age=False, 17.5 in working_age=False


**`array.array`**: large numeric series where memory matters, such as sensor readings.
The type code `"f"` stores 4-byte C floats, and `tobytes()` gives the raw block,
ready for a file or a socket.

In [198]:
temperature_readings = array.array("f")
for minute in range(60 * 24):
    temperature_readings.append(20 + (minute % 120) / 10)
average_temperature = sum(temperature_readings) / len(temperature_readings)
print(f"{len(temperature_readings)=}, {average_temperature=:.2f}")
print(f"{len(temperature_readings.tobytes())=:,} bytes for a day of readings")

len(temperature_readings)=1440, average_temperature=25.95
len(temperature_readings.tobytes())=5,760 bytes for a day of readings


### Lab 1.5

Three coding exercises and a set of predict-the-output questions on aliasing. For each
coding exercise, replace the `raise NotImplementedError` in the stub, then run the
check cell. Until then the checks print FAIL; that is expected.

#### Exercise 1 — sort employee records by several keys

Order employees by **department ascending**, then **salary descending**, then
**name ascending**. Write it twice:

1. `sort_employees_multipass(employees)`: several stable `sorted()` passes, with no
   tuple keys.
2. `sort_employees_tuple_key(employees)`: one `sorted()` call with a tuple key.

Both return a **new** list and leave the input unchanged. Run the next cell first; it
defines `Employee` and the sample data.

In [199]:
class Employee(NamedTuple):
    """An employee record for the sorting exercise."""

    name: str
    department: str
    salary: int


SAMPLE_EMPLOYEES = [
    Employee("Asha", "Engineering", 120_000),
    Employee("Ravi", "Sales", 80_000),
    Employee("Meena", "Engineering", 150_000),
    Employee("John", "Sales", 80_000),
    Employee("Fatima", "Design", 95_000),
    Employee("Arjun", "Engineering", 120_000),
]

The two stubs:

In [200]:
def sort_employees_multipass(employees: list[Employee]) -> list[Employee]:
    """Sort by department, salary (high first), name using stable multi-pass sorting."""
    raise NotImplementedError("Lab 1.5: implement sort_employees_multipass()")


def sort_employees_tuple_key(employees: list[Employee]) -> list[Employee]:
    """Sort by department, salary (high first), name using one tuple key."""
    raise NotImplementedError("Lab 1.5: implement sort_employees_tuple_key()")

The check cell defines two helpers: `employee_names` (the names in order) and
`returns_new_list` (checks the input list is left alone).

In [201]:
def employee_names(employees: list[Employee]) -> list[str]:
    """Return just the names, in order."""
    return [employee.name for employee in employees]


def returns_new_list(sort_function: Callable[[list[Employee]], list[Employee]]) -> bool:
    """Return True if sort_function returns a new list and leaves its argument alone."""
    employees_argument = list(SAMPLE_EMPLOYEES)
    sorted_employees = sort_function(employees_argument)
    return sorted_employees is not employees_argument and employees_argument == SAMPLE_EMPLOYEES


expected_employee_order = ["Fatima", "Meena", "Arjun", "Asha", "John", "Ravi"]
run_checks(
    [
        (
            "multi-pass order",
            lambda: employee_names(sort_employees_multipass(SAMPLE_EMPLOYEES)),
            expected_employee_order,
        ),
        (
            "tuple-key order",
            lambda: employee_names(sort_employees_tuple_key(SAMPLE_EMPLOYEES)),
            expected_employee_order,
        ),
        (
            "both versions agree",
            lambda: (
                sort_employees_multipass(SAMPLE_EMPLOYEES)
                == sort_employees_tuple_key(SAMPLE_EMPLOYEES)
            ),
            True,
        ),
        ("multi-pass returns a new list", lambda: returns_new_list(sort_employees_multipass), True),
        ("tuple-key returns a new list", lambda: returns_new_list(sort_employees_tuple_key), True),
        ("empty list", lambda: sort_employees_tuple_key([]), []),
    ]
)

FAIL  multi-pass order: raised NotImplementedError: Lab 1.5: implement sort_employees_multipass()
FAIL  tuple-key order: raised NotImplementedError: Lab 1.5: implement sort_employees_tuple_key()
FAIL  both versions agree: raised NotImplementedError: Lab 1.5: implement sort_employees_multipass()
FAIL  multi-pass returns a new list: raised NotImplementedError: Lab 1.5: implement sort_employees_multipass()
FAIL  tuple-key returns a new list: raised NotImplementedError: Lab 1.5: implement sort_employees_tuple_key()
FAIL  empty list: raised NotImplementedError: Lab 1.5: implement sort_employees_tuple_key()
0/6 checks passed


#### Exercise 2 — `transpose(matrix)`

Write `transpose(matrix: list[list[T]]) -> list[list[T]]` using `zip(*matrix)`:
rows become columns. Return a list of **lists** (not tuples). `[]` transposes to
`[]`. If the rows have different lengths, raise `ValueError` (*hint:*
`zip(..., strict=True)`).

In [202]:
def transpose[T](matrix: list[list[T]]) -> list[list[T]]:
    """Return the transpose of a rectangular matrix given as a list of rows."""
    raise NotImplementedError("Lab 1.5: implement transpose()")

Run the checks.

In [203]:
run_checks(
    [
        ("2x3 becomes 3x2", lambda: transpose([[1, 2, 3], [4, 5, 6]]), [[1, 4], [2, 5], [3, 6]]),
        ("square", lambda: transpose([[1, 2], [3, 4]]), [[1, 3], [2, 4]]),
        ("single row", lambda: transpose([["a", "b"]]), [["a"], ["b"]]),
        ("empty matrix", lambda: transpose([]), []),
        ("rows are lists", lambda: type(transpose([[1, 2]])[0]).__name__, "list"),
        (
            "ragged rows raise",
            lambda: raised_error_name(lambda: transpose([[1, 2], [3]])),
            "ValueError",
        ),
    ]
)

FAIL  2x3 becomes 3x2: raised NotImplementedError: Lab 1.5: implement transpose()
FAIL  square: raised NotImplementedError: Lab 1.5: implement transpose()
FAIL  single row: raised NotImplementedError: Lab 1.5: implement transpose()
FAIL  empty matrix: raised NotImplementedError: Lab 1.5: implement transpose()
FAIL  rows are lists: raised NotImplementedError: Lab 1.5: implement transpose()
FAIL  ragged rows raise: expected 'ValueError', got 'NotImplementedError'
0/6 checks passed


#### Exercise 3 — `dedupe_preserving_order(items)`

Write `dedupe_preserving_order(items: Iterable[T]) -> list[T]` for hashable items:
keep the **first** occurrence of each item, in the original order. It must accept any
iterable, including a one-shot iterator. Note that `1`, `1.0` and `True` are equal
and hash the same, so they count as duplicates.

In [204]:
def dedupe_preserving_order[T](items: Iterable[T]) -> list[T]:
    """Return the items without duplicates, keeping first occurrences in order."""
    raise NotImplementedError("Lab 1.5: implement dedupe_preserving_order()")

Run the checks.

In [205]:
run_checks(
    [
        ("ints", lambda: dedupe_preserving_order([3, 1, 3, 2, 1]), [3, 1, 2]),
        ("strings", lambda: dedupe_preserving_order(["b", "a", "b", "c", "a"]), ["b", "a", "c"]),
        ("empty", lambda: dedupe_preserving_order([]), []),
        (
            "one-shot iterator",
            lambda: dedupe_preserving_order(iter("mississippi")),
            ["m", "i", "s", "p"],
        ),
        ("1, 1.0 and True are equal", lambda: dedupe_preserving_order([1, 1.0, True, 2]), [1, 2]),
        ("tuples", lambda: dedupe_preserving_order([(1, 2), (1, 2), (2, 1)]), [(1, 2), (2, 1)]),
    ]
)

FAIL  ints: raised NotImplementedError: Lab 1.5: implement dedupe_preserving_order()
FAIL  strings: raised NotImplementedError: Lab 1.5: implement dedupe_preserving_order()
FAIL  empty: raised NotImplementedError: Lab 1.5: implement dedupe_preserving_order()
FAIL  one-shot iterator: raised NotImplementedError: Lab 1.5: implement dedupe_preserving_order()
FAIL  1, 1.0 and True are equal: raised NotImplementedError: Lab 1.5: implement dedupe_preserving_order()
FAIL  tuples: raised NotImplementedError: Lab 1.5: implement dedupe_preserving_order()
0/6 checks passed


#### Exercise 4 — predict the output (aliasing)

**Predict the output.**

```python
seating_grid = [[0] * 3] * 3
seating_grid[0][0] = 1
print(seating_grid)
```

<details><summary>Answer</summary>

`[[1, 0, 0], [1, 0, 0], [1, 0, 0]]`. `[0] * 3` builds one inner list, and `* 3` on the
outer list copies the **reference** to it three times. All three rows are the same
object, so changing one changes them all. (`[0] * 3` itself is fine: ints are
immutable, so sharing them is harmless.)

</details>

Run the next cell to check your prediction.

In [206]:
seating_grid = [[0] * 3] * 3
seating_grid[0][0] = 1
print(seating_grid)
print(f"{seating_grid[0] is seating_grid[1]=}")

[[1, 0, 0], [1, 0, 0], [1, 0, 0]]
seating_grid[0] is seating_grid[1]=True


**Predict the output.**

```python
seating_grid = [[0] * 3 for _ in range(3)]
seating_grid[0][0] = 1
print(seating_grid)
```

<details><summary>Answer</summary>

`[[1, 0, 0], [0, 0, 0], [0, 0, 0]]`. The comprehension evaluates `[0] * 3` once per
row, so each row is a separate list. This is the correct way to build a grid.

</details>

Run the next cell to check your prediction.

In [207]:
seating_grid = [[0] * 3 for _ in range(3)]
seating_grid[0][0] = 1
print(seating_grid)
print(f"{seating_grid[0] is seating_grid[1]=}")

[[1, 0, 0], [0, 0, 0], [0, 0, 0]]
seating_grid[0] is seating_grid[1]=False


**Predict the output.**

```python
pending_orders = processed_orders = []
pending_orders.append("A-17")
print(processed_orders, pending_orders is processed_orders)
```

<details><summary>Answer</summary>

`['A-17'] True`. Chained assignment binds both names to **one** list object. Write
`pending_orders, processed_orders = [], []` for two lists.

</details>

Run the next cell to check your prediction.

In [208]:
pending_orders = processed_orders = []
pending_orders.append("A-17")
print(processed_orders, pending_orders is processed_orders)

['A-17'] True


**Predict the output.**

```python
shelf_rows = [[]] * 2
shelf_rows[0] += ["jam"]
shelf_rows[1] = shelf_rows[1] + ["tea"]
print(shelf_rows)
```

<details><summary>Answer</summary>

`[['jam'], ['jam', 'tea']]`. Both slots start as the same empty list. `+=` extends
that shared list in place, so both rows show `'jam'`. `shelf_rows[1] + ["tea"]` builds
a **new** list and rebinds only slot 1.

</details>

Run the next cell to check your prediction.

In [209]:
shelf_rows = [[]] * 2
shelf_rows[0] += ["jam"]
shelf_rows[1] = shelf_rows[1] + ["tea"]
print(shelf_rows)

[['jam'], ['jam', 'tea']]


### Recap — Unit 1.5

| Concept | One-line takeaway | JS equivalent |
| --- | --- | --- |
| Sequence operations | Indexing, slicing, `in`, `len`, `min`, `max`, `count`, `index` on every sequence; `range` has no `+`/`*` | Array methods |
| `list` internals | Dynamic array of pointers with spare capacity; `append` is O(1) amortized | `Array` |
| Front operations | `pop(0)` and `insert(0, x)` are O(n) | `shift`, `unshift` |
| Mutating methods | Return `None`; `sorted()` and `+` return new lists | `push` returns the length |
| `+=` vs `+` | `+=` extends in place (any iterable); `+` builds a new list | `push(...items)` vs `concat` |
| Slice assignment | Replaces a slice, may change the length; extended slices must match | `splice` |
| Copies | `copy()`, `[:]`, `list()` are shallow | `[...arr]`, `slice()` |
| Sorting | Stable Timsort; `key=`, `reverse=`, `itemgetter`, tuple keys with negation | `sort((a, b) => …)` |
| `tuple` | The comma makes it; immutable; hashable if all items are | `Object.freeze([])` |
| `t[0] += [x]` | Mutates the inner list *and* raises `TypeError` | — |
| Named tuples | `namedtuple` / `NamedTuple`: fields, defaults, `_replace`, `_asdict` | object literal |
| `range` | Lazy, O(1) `len`/`in`/index, constant memory | — |
| `array.array` | Typed, compact numeric storage | `Float64Array` |
| `[[0] * 3] * 3` | Repeats references; build grids with a comprehension | `Array(3).fill([])` has the same bug |

**Further reading**

- [Sequence types — `list`, `tuple`, `range`](https://docs.python.org/3/library/stdtypes.html#sequence-types-list-tuple-range),
  [more on lists](https://docs.python.org/3/tutorial/datastructures.html#more-on-lists)
- [Sorting techniques](https://docs.python.org/3/howto/sorting.html),
  [`operator.itemgetter` / `attrgetter`](https://docs.python.org/3/library/operator.html#operator.itemgetter)
- [`collections.namedtuple`](https://docs.python.org/3/library/collections.html#collections.namedtuple),
  [`typing.NamedTuple`](https://docs.python.org/3/library/typing.html#typing.NamedTuple)
- [`array`](https://docs.python.org/3/library/array.html),
  [Time complexity of CPython operations](https://wiki.python.org/moin/TimeComplexity)
- [PEP 3132 — Extended iterable unpacking](https://peps.python.org/pep-3132/),
  [PEP 448 — Additional unpacking generalizations](https://peps.python.org/pep-0448/),
  [PEP 695 — Type parameter syntax](https://peps.python.org/pep-0695/) (used by the
  `transpose[T]` stub)